In [2]:
# ============================================================
# EXP-005 — INTERNAL VALIDATION POPULATION CONSTRUCTION
# ============================================================
#
# Purpose:
#   Construct a reproducible population of NEOWISE source-like
#   groups for Phase III validator testing.
#
# IMPORTANT:
#   These are OPERATIONAL cohorts, not externally confirmed
#   scientific labels.
#
# Cohorts:
#   1. STABLE_CONTROL
#      Low measured variability under our operational criteria.
#
#   2. VARIABLE_CONTROL
#      Strong measured variability under our operational criteria.
#
#   3. QUALITY_FLAGGED_CONTROL
#      Source groups containing lower-quality measurements.
#
# Candidate #1 from EXP-001 is explicitly excluded.
#
# This experiment does NOT claim:
#   "stable = physically non-variable"
#   "variable = astrophysically variable"
#   "flagged = confirmed artifact"
#
# Those independent labels will be addressed in later
# Phase III experiments.
# ============================================================


# ------------------------------------------------------------
# 1. Environment
# ------------------------------------------------------------

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from pyarrow import dataset as ds
from pyarrow.fs import S3FileSystem

from sklearn.cluster import DBSCAN

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 2. Experiment configuration
# ------------------------------------------------------------

EXPERIMENT_ID = "EXP-005"
PROJECT_NAME = "Multi-view astronomical novelty detection and candidate validation"

PROJECT_DIR = Path("astronomy_exp005")
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# NEOWISE region used during EXP-001/003/004
RA_CENTER = 180.0
DEC_CENTER = 0.0
RADIUS_DEG = 0.10

RA_MIN = RA_CENTER - RADIUS_DEG
RA_MAX = RA_CENTER + RADIUS_DEG
DEC_MIN = DEC_CENTER - RADIUS_DEG
DEC_MAX = DEC_CENTER + RADIUS_DEG

YEAR = "year11"

BUCKET = "nasa-irsa-wise"
BASE_PREFIX = "wise/neowiser/catalogs/p1bs_psd/healpix_k5"

METADATA_PATH = (
    f"{BUCKET}/{BASE_PREFIX}/{YEAR}/"
    f"neowiser-healpix_k5-{YEAR}.parquet/_metadata"
)


# Candidate #1 — explicitly excluded
CANDIDATE_RA = 179.936465
CANDIDATE_DEC = -0.0897475

CANDIDATE_EXCLUSION_ARCSEC = 5.0


# Source-group construction parameters.
#
# This is NOT intended to replace VarWISE source association.
# It is only a deterministic grouping method for this experiment.
#
# 1.5 arcsec is deliberately smaller than the 2 arcsec scale
# used in the EXP-003 pair matching.
GROUP_RADIUS_ARCSEC = 1.5

# Minimum number of detections required for a source-like group.
MIN_OBSERVATIONS = 10


# Population sizes.
#
# We construct more than we necessarily use later so that
# downstream experiments can select reproducible subsets.
N_STABLE = 30
N_VARIABLE = 30
N_FLAGGED = 30


# Operational thresholds.
#
# These are engineering criteria for constructing the test
# population, NOT universal astronomy thresholds.
STABLE_REDUCED_CHI2_MAX = 1.2
VARIABLE_REDUCED_CHI2_MIN = 2.0


print("=" * 70)
print(f"{EXPERIMENT_ID} — INTERNAL VALIDATION POPULATION")
print("=" * 70)
print(f"Project directory : {PROJECT_DIR}")
print(f"NEOWISE release   : {YEAR}")
print(f"RA range          : {RA_MIN} → {RA_MAX}")
print(f"Dec range         : {DEC_MIN} → {DEC_MAX}")
print()


# ------------------------------------------------------------
# 3. Connect to NEOWISE Parquet
# ------------------------------------------------------------

print("[1/8] Connecting to NEOWISE...")

fs = S3FileSystem(
    region="us-west-2",
    anonymous=True
)

neowise_ds = ds.parquet_dataset(
    METADATA_PATH,
    filesystem=fs,
    partitioning="hive"
)

print("[OK] NEOWISE dataset opened.")


# ------------------------------------------------------------
# 4. Retrieve local region
# ------------------------------------------------------------

print()
print("[2/8] Querying local validation region...")

columns = [
    "ra",
    "dec",
    "mjd",
    "w1mpro",
    "w1sigmpro",
    "w2mpro",
    "w2sigmpro",
    "w1snr",
    "w2snr",
    "qual_frame",
    "ph_qual",
    "cc_flags",
    "scan_id",
    "frame_num",
]

filter_expr = (
    (ds.field("ra") >= RA_MIN)
    & (ds.field("ra") <= RA_MAX)
    & (ds.field("dec") >= DEC_MIN)
    & (ds.field("dec") <= DEC_MAX)
)

region_df = (
    neowise_ds
    .scanner(
        filter=filter_expr,
        columns=columns
    )
    .to_table()
    .to_pandas()
)

print(f"[OK] Retrieved detections: {len(region_df):,}")


# ------------------------------------------------------------
# 5. Basic numeric cleaning
# ------------------------------------------------------------

print()
print("[3/8] Cleaning measurements...")

numeric_columns = [
    "ra",
    "dec",
    "mjd",
    "w1mpro",
    "w1sigmpro",
    "w2mpro",
    "w2sigmpro",
    "w1snr",
    "w2snr",
    "qual_frame",
    "frame_num",
]

for col in numeric_columns:
    region_df[col] = pd.to_numeric(
        region_df[col],
        errors="coerce"
    )

region_df["ph_qual"] = (
    region_df["ph_qual"]
    .fillna("")
    .astype(str)
)

region_df["cc_flags"] = (
    region_df["cc_flags"]
    .fillna("")
    .astype(str)
)

# Valid W1 measurements
region_df = region_df[
    np.isfinite(region_df["ra"])
    & np.isfinite(region_df["dec"])
    & np.isfinite(region_df["mjd"])
    & np.isfinite(region_df["w1mpro"])
    & np.isfinite(region_df["w1sigmpro"])
    & (region_df["w1sigmpro"] > 0)
].copy()

region_df.reset_index(drop=True, inplace=True)

print(f"[OK] Usable W1 detections: {len(region_df):,}")


# ------------------------------------------------------------
# 6. Define quality status
# ------------------------------------------------------------

print()
print("[4/8] Assigning measurement-quality status...")

region_df["good_quality"] = (
    (region_df["qual_frame"] > 0)
    & (~region_df["ph_qual"].str.contains("U", na=False))
    & (region_df["cc_flags"] == "0000")
)

region_df["quality_flagged"] = ~region_df["good_quality"]

print(
    f"[OK] Good-quality detections   : "
    f"{region_df['good_quality'].sum():,}"
)

print(
    f"[OK] Quality-flagged detections: "
    f"{region_df['quality_flagged'].sum():,}"
)


# ------------------------------------------------------------
# 7. Exclude Candidate #1
# ------------------------------------------------------------

print()
print("[5/8] Excluding EXP-001 Candidate #1...")

# Small-angle approximation is sufficient for this tiny field.
cos_dec = np.cos(np.deg2rad(CANDIDATE_DEC))

candidate_dra = (
    (region_df["ra"] - CANDIDATE_RA)
    * cos_dec
)

candidate_ddec = (
    region_df["dec"] - CANDIDATE_DEC
)

candidate_sep_arcsec = (
    np.sqrt(
        candidate_dra**2
        + candidate_ddec**2
    )
    * 3600.0
)

region_df["candidate1_sep_arcsec"] = candidate_sep_arcsec

candidate_exclusion_mask = (
    candidate_sep_arcsec <= CANDIDATE_EXCLUSION_ARCSEC
)

excluded_count = int(candidate_exclusion_mask.sum())

region_df = region_df[
    ~candidate_exclusion_mask
].copy()

region_df.reset_index(drop=True, inplace=True)

print(
    f"[OK] Excluded detections near Candidate #1: "
    f"{excluded_count:,}"
)

print(
    f"[OK] Remaining detections: "
    f"{len(region_df):,}"
)


# ------------------------------------------------------------
# 8. Spatial source-like grouping
# ------------------------------------------------------------

print()
print("[6/8] Constructing source-like groups...")

# Convert RA/Dec to local tangent-plane-like coordinates
# in arcseconds.
#
# x = RA difference corrected by cos(dec)
# y = Dec difference
#
# This is only for this small field and this experiment.

mean_dec_rad = np.deg2rad(
    region_df["dec"].median()
)

x_arcsec = (
    (region_df["ra"] - RA_CENTER)
    * np.cos(mean_dec_rad)
    * 3600.0
)

y_arcsec = (
    (region_df["dec"] - DEC_CENTER)
    * 3600.0
)

coords = np.column_stack([
    x_arcsec.to_numpy(),
    y_arcsec.to_numpy()
])

# DBSCAN groups detections that are spatially close.
#
# min_samples=1 means every detection gets assigned to a group.
# Later we require >= MIN_OBSERVATIONS.

clusterer = DBSCAN(
    eps=GROUP_RADIUS_ARCSEC,
    min_samples=1,
    metric="euclidean"
)

labels = clusterer.fit_predict(coords)

region_df["group_id"] = labels

n_groups = region_df["group_id"].nunique()

print(
    f"[OK] Spatial groups constructed: "
    f"{n_groups:,}"
)


# ------------------------------------------------------------
# 9. Build group-level measurements
# ------------------------------------------------------------

print()
print("[7/8] Computing group-level variability features...")


def weighted_mean(values, errors):
    values = np.asarray(values, dtype=float)
    errors = np.asarray(errors, dtype=float)

    valid = (
        np.isfinite(values)
        & np.isfinite(errors)
        & (errors > 0)
    )

    if valid.sum() == 0:
        return np.nan

    values = values[valid]
    errors = errors[valid]

    weights = 1.0 / (errors ** 2)

    return np.sum(weights * values) / np.sum(weights)


def reduced_chi_square(values, errors):
    values = np.asarray(values, dtype=float)
    errors = np.asarray(errors, dtype=float)

    valid = (
        np.isfinite(values)
        & np.isfinite(errors)
        & (errors > 0)
    )

    values = values[valid]
    errors = errors[valid]

    n = len(values)

    if n < 2:
        return np.nan

    mean_value = weighted_mean(
        values,
        errors
    )

    chi2 = np.sum(
        ((values - mean_value) / errors) ** 2
    )

    dof = n - 1

    return chi2 / dof


group_records = []

for group_id, g in region_df.groupby("group_id"):

    n_obs = len(g)

    if n_obs < MIN_OBSERVATIONS:
        continue

    w1 = g["w1mpro"].to_numpy(dtype=float)
    w1err = g["w1sigmpro"].to_numpy(dtype=float)

    valid_w1 = (
        np.isfinite(w1)
        & np.isfinite(w1err)
        & (w1err > 0)
    )

    w1 = w1[valid_w1]
    w1err = w1err[valid_w1]

    if len(w1) < MIN_OBSERVATIONS:
        continue

    w1_mean = weighted_mean(
        w1,
        w1err
    )

    w1_std = float(
        np.std(w1, ddof=1)
    )

    w1_median = float(
        np.median(w1)
    )

    w1_range = float(
        np.max(w1) - np.min(w1)
    )

    w1_rchi2 = reduced_chi_square(
        w1,
        w1err
    )

    # Robust scatter using MAD.
    w1_mad = float(
        np.median(
            np.abs(
                w1 - np.median(w1)
            )
        )
    )

    w1_mad_sigma = (
        1.4826 * w1_mad
    )

    # W2 statistics where available.
    w2 = pd.to_numeric(
        g["w2mpro"],
        errors="coerce"
    ).to_numpy(dtype=float)

    w2_valid = np.isfinite(w2)

    if w2_valid.sum() > 0:
        w2_mean = float(
            np.mean(w2[w2_valid])
        )

        w2_std = (
            float(np.std(w2[w2_valid], ddof=1))
            if w2_valid.sum() > 1
            else np.nan
        )
    else:
        w2_mean = np.nan
        w2_std = np.nan

    # Position statistics.
    ra_median = float(
        np.median(g["ra"])
    )

    dec_median = float(
        np.median(g["dec"])
    )

    # Distance of each detection from the group's median position.
    dra = (
        (g["ra"].to_numpy(dtype=float) - ra_median)
        * np.cos(mean_dec_rad)
        * 3600.0
    )

    ddec = (
        g["dec"].to_numpy(dtype=float) - dec_median
    ) * 3600.0

    position_offsets = np.sqrt(
        dra**2 + ddec**2
    )

    # Temporal coverage.
    mjd = g["mjd"].to_numpy(dtype=float)

    time_span_days = (
        np.max(mjd) - np.min(mjd)
    )

    # Quality information.
    good_fraction = float(
        g["good_quality"].mean()
    )

    flagged_count = int(
        g["quality_flagged"].sum()
    )

    median_w1_snr = float(
        np.nanmedian(
            pd.to_numeric(
                g["w1snr"],
                errors="coerce"
            )
        )
    )

    group_records.append({
        "group_id": int(group_id),
        "ra": ra_median,
        "dec": dec_median,
        "n_observations": int(n_obs),

        "w1_weighted_mean": w1_mean,
        "w1_std": w1_std,
        "w1_median": w1_median,
        "w1_range": w1_range,
        "w1_reduced_chi2": w1_rchi2,
        "w1_mad_sigma": w1_mad_sigma,

        "w2_mean": w2_mean,
        "w2_std": w2_std,

        "median_w1_snr": median_w1_snr,

        "time_span_days": time_span_days,

        "median_position_offset_arcsec": float(
            np.median(position_offsets)
        ),

        "max_position_offset_arcsec": float(
            np.max(position_offsets)
        ),

        "good_quality_fraction": good_fraction,
        "flagged_count": flagged_count,
    })


groups_df = pd.DataFrame(
    group_records
)

print(
    f"[OK] Source-like groups with "
    f">={MIN_OBSERVATIONS} observations: "
    f"{len(groups_df):,}"
)


# ------------------------------------------------------------
# 10. Remove pathological group statistics
# ------------------------------------------------------------

groups_df = groups_df[
    np.isfinite(
        groups_df["w1_reduced_chi2"]
    )
    & np.isfinite(
        groups_df["w1_std"]
    )
].copy()

groups_df.reset_index(drop=True, inplace=True)


# ------------------------------------------------------------
# 11. Construct operational cohorts
# ------------------------------------------------------------

print()
print("[8/8] Constructing validation cohorts...")


# ------------------------------------------------------------
# STABLE CONTROL
# ------------------------------------------------------------
#
# Select sources with:
#   - enough observations
#   - low reduced chi-square
#   - good measurement quality
#
# Sorted by increasing chi-square so the most stable sources
# are selected first.

stable_pool = groups_df[
    (groups_df["w1_reduced_chi2"]
        <= STABLE_REDUCED_CHI2_MAX)
    & (groups_df["good_quality_fraction"] >= 0.90)
    & (groups_df["median_w1_snr"] >= 5)
].copy()

stable_pool = stable_pool.sort_values(
    [
        "w1_reduced_chi2",
        "w1_std"
    ],
    ascending=True
)

stable_df = stable_pool.head(
    N_STABLE
).copy()

stable_df["population"] = "STABLE_CONTROL"
stable_df["label_provenance"] = (
    "internal_operational_selection"
)


# ------------------------------------------------------------
# VARIABLE CONTROL
# ------------------------------------------------------------
#
# Strong measured variability.
#
# We require reasonable data quality so that this cohort
# is not simply a collection of bad measurements.

variable_pool = groups_df[
    (groups_df["w1_reduced_chi2"]
        >= VARIABLE_REDUCED_CHI2_MIN)
    & (groups_df["good_quality_fraction"] >= 0.90)
    & (groups_df["median_w1_snr"] >= 5)
].copy()

variable_pool = variable_pool.sort_values(
    [
        "w1_reduced_chi2",
        "w1_range"
    ],
    ascending=False
)

variable_df = variable_pool.head(
    N_VARIABLE
).copy()

variable_df["population"] = "VARIABLE_CONTROL"
variable_df["label_provenance"] = (
    "internal_operational_selection"
)


# ------------------------------------------------------------
# QUALITY-FLAGGED CONTROL
# ------------------------------------------------------------
#
# These are NOT confirmed artifacts.
#
# They are source groups containing a substantial fraction
# of quality-flagged measurements and therefore provide a
# useful stress-test population for the validator.

flagged_pool = groups_df[
    (groups_df["flagged_count"] >= 2)
    & (groups_df["good_quality_fraction"] < 0.90)
].copy()

flagged_pool = flagged_pool.sort_values(
    [
        "good_quality_fraction",
        "flagged_count"
    ],
    ascending=[True, False]
)

flagged_df = flagged_pool.head(
    N_FLAGGED
).copy()

flagged_df["population"] = "QUALITY_FLAGGED_CONTROL"
flagged_df["label_provenance"] = (
    "internal_quality_flag_proxy"
)


# ------------------------------------------------------------
# 12. Combine populations
# ------------------------------------------------------------

validation_population = pd.concat(
    [
        stable_df,
        variable_df,
        flagged_df
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 13. Ensure populations don't overlap
# ------------------------------------------------------------

population_counts = (
    validation_population
    .groupby("group_id")["population"]
    .nunique()
)

overlapping_groups = (
    population_counts[
        population_counts > 1
    ]
)

if len(overlapping_groups) > 0:

    print(
        "[WARNING] Overlapping groups detected."
    )

    # Keep the first assigned population.
    validation_population = (
        validation_population
        .drop_duplicates(
            subset=["group_id"],
            keep="first"
        )
        .reset_index(drop=True)
    )

else:

    print(
        "[OK] No population overlap."
    )


# ------------------------------------------------------------
# 14. Assign stable deterministic validation IDs
# ------------------------------------------------------------

validation_population[
    "validation_id"
] = [
    f"EXP005-{i:04d}"
    for i in range(
        1,
        len(validation_population) + 1
    )
]


# ------------------------------------------------------------
# 15. Sort for reproducibility
# ------------------------------------------------------------

validation_population = (
    validation_population
    .sort_values(
        [
            "population",
            "w1_reduced_chi2"
        ],
        ascending=[
            True,
            True
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 16. Save outputs
# ------------------------------------------------------------

population_path = (
    RESULTS_DIR /
    "exp005_validation_population.csv"
)

groups_path = (
    RESULTS_DIR /
    "exp005_all_source_groups.csv"
)

detections_path = (
    RESULTS_DIR /
    "exp005_region_detections.csv"
)

summary_path = (
    RESULTS_DIR /
    "exp005_population_summary.json"
)


validation_population.to_csv(
    population_path,
    index=False
)

groups_df.to_csv(
    groups_path,
    index=False
)

region_df.to_csv(
    detections_path,
    index=False
)


# ------------------------------------------------------------
# 17. Population summary
# ------------------------------------------------------------

population_summary = {
    "experiment_id": EXPERIMENT_ID,
    "project_name": PROJECT_NAME,

    "neowise_year": YEAR,

    "region": {
        "ra_min": RA_MIN,
        "ra_max": RA_MAX,
        "dec_min": DEC_MIN,
        "dec_max": DEC_MAX
    },

    "candidate_001": {
        "ra": CANDIDATE_RA,
        "dec": CANDIDATE_DEC,
        "exclusion_radius_arcsec":
            CANDIDATE_EXCLUSION_ARCSEC
    },

    "retrieved_detections": int(
        len(region_df)
    ),

    "source_like_groups": int(
        len(groups_df)
    ),

    "requested_population_sizes": {
        "stable": N_STABLE,
        "variable": N_VARIABLE,
        "quality_flagged": N_FLAGGED
    },

    "actual_population_sizes": {
        "stable": int(
            (validation_population["population"]
             == "STABLE_CONTROL").sum()
        ),

        "variable": int(
            (validation_population["population"]
             == "VARIABLE_CONTROL").sum()
        ),

        "quality_flagged": int(
            (validation_population["population"]
             == "QUALITY_FLAGGED_CONTROL").sum()
        )
    },

    "operational_thresholds": {
        "stable_reduced_chi2_max":
            STABLE_REDUCED_CHI2_MAX,

        "variable_reduced_chi2_min":
            VARIABLE_REDUCED_CHI2_MIN,

        "minimum_observations":
            MIN_OBSERVATIONS,

        "group_radius_arcsec":
            GROUP_RADIUS_ARCSEC
    },

    "methodological_warning": (
        "Population labels are internal operational cohorts, "
        "not independent astronomical truth labels."
    )
}

with open(
    summary_path,
    "w"
) as f:

    json.dump(
        population_summary,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 18. Print final experiment report
# ------------------------------------------------------------

print()
print("=" * 70)
print("EXP-005 COMPLETE")
print("=" * 70)

print()
print("DATASET")
print(
    f"  Retrieved detections       : "
    f"{len(region_df):,}"
)

print(
    f"  Source-like groups         : "
    f"{len(groups_df):,}"
)

print()
print("VALIDATION POPULATION")

for population_name in [
    "STABLE_CONTROL",
    "VARIABLE_CONTROL",
    "QUALITY_FLAGGED_CONTROL"
]:

    subset = validation_population[
        validation_population["population"]
        == population_name
    ]

    print(
        f"  {population_name:<24}: "
        f"{len(subset):>3}"
    )

print()
print("STATISTICAL RANGES")

for population_name in [
    "STABLE_CONTROL",
    "VARIABLE_CONTROL",
    "QUALITY_FLAGGED_CONTROL"
]:

    subset = validation_population[
        validation_population["population"]
        == population_name
    ]

    if len(subset) == 0:
        continue

    print()
    print(population_name)

    print(
        "  reduced χ² median : "
        f"{subset['w1_reduced_chi2'].median():.3f}"
    )

    print(
        "  reduced χ² max    : "
        f"{subset['w1_reduced_chi2'].max():.3f}"
    )

    print(
        "  W1 range median   : "
        f"{subset['w1_range'].median():.4f} mag"
    )

    print(
        "  observations median: "
        f"{subset['n_observations'].median():.0f}"
    )

print()
print("OUTPUT FILES")

print(
    f"  {population_path}"
)

print(
    f"  {groups_path}"
)

print(
    f"  {detections_path}"
)

print(
    f"  {summary_path}"
)

print()
print("=" * 70)
print("IMPORTANT:")
print(
    "These cohorts are internal validation groups. "
    "They are NOT independent scientific labels."
)
print(
    "EXP-006+ will introduce externally characterized "
    "reference objects before we claim validator performance."
)
print("=" * 70)

EXP-005 — INTERNAL VALIDATION POPULATION
Project directory : astronomy_exp005
NEOWISE release   : year11
RA range          : 179.9 → 180.1
Dec range         : -0.1 → 0.1

[1/8] Connecting to NEOWISE...
[OK] NEOWISE dataset opened.

[2/8] Querying local validation region...
[OK] Retrieved detections: 8,652

[3/8] Cleaning measurements...
[OK] Usable W1 detections: 7,308

[4/8] Assigning measurement-quality status...
[OK] Good-quality detections   : 2,498
[OK] Quality-flagged detections: 4,810

[5/8] Excluding EXP-001 Candidate #1...
[OK] Excluded detections near Candidate #1: 33
[OK] Remaining detections: 7,275

[6/8] Constructing source-like groups...
[OK] Spatial groups constructed: 3,371

[7/8] Computing group-level variability features...
[OK] Source-like groups with >=10 observations: 175

[8/8] Constructing validation cohorts...
[OK] No population overlap.

EXP-005 COMPLETE

DATASET
  Retrieved detections       : 7,275
  Source-like groups         : 175

VALIDATION POPULATION
  ST

In [3]:
# ============================================================
# EXP-006 — INDEPENDENT REFERENCE POPULATION
# Corrected complete experiment
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import requests
import io
import json
import re
from astropy.coordinates import Angle

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_DIR = Path("astronomy_exp005")
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EXP_ID = "EXP-006"

RA_MIN, RA_MAX = 179.9, 180.1
DEC_MIN, DEC_MAX = -0.1, 0.1
MATCH_RADIUS_ARCSEC = 5.0

print("=" * 70)
print("EXP-006 — INDEPENDENT REFERENCE POPULATION")
print("=" * 70)

# ------------------------------------------------------------
# 2. Download independent GCVS catalog
# ------------------------------------------------------------

print("\n[1/9] Downloading GCVS reference catalog...")

url = (
    "https://vizier.cds.unistra.fr/viz-bin/asu-tsv"
    "?-source=B/gcvs/gcvs_cat"
    "&-out=GCVS,RAJ2000,DEJ2000,VarType"
    "&-out.max=100000"
)

response = requests.get(url, timeout=180)
response.raise_for_status()

raw_text = response.text

if not raw_text.strip():
    raise RuntimeError("VizieR returned an empty response.")

print("[OK] Response received.")
print("Response size:", len(raw_text), "characters")

# ------------------------------------------------------------
# 3. Parse VizieR response
# ------------------------------------------------------------

print("\n[2/9] Parsing catalog...")

lines = [
    line for line in raw_text.splitlines()
    if line.strip() and not line.startswith("#")
]

if len(lines) < 2:
    raise RuntimeError("No usable catalog table found.")

gcvs = pd.read_csv(
    io.StringIO("\n".join(lines)),
    sep="\t",
    dtype=str
)

gcvs.columns = [
    str(c).strip() for c in gcvs.columns
]

print("[OK] Rows:", len(gcvs))
print("[OK] Columns:", list(gcvs.columns))

required = ["GCVS", "RAJ2000", "DEJ2000", "VarType"]

missing = [
    c for c in required
    if c not in gcvs.columns
]

if missing:
    raise RuntimeError(
        f"Missing required columns: {missing}"
    )

# ------------------------------------------------------------
# 4. Robust coordinate conversion
# ------------------------------------------------------------

print("\n[3/9] Converting J2000 coordinates...")

def parse_ra(value):
    try:
        s = str(value).strip()

        if not s or s.lower() in {"nan", "none"}:
            return np.nan

        # Normalize separators and whitespace.
        s = re.sub(r"[hH]", ":", s)
        s = re.sub(r"[mM]", ":", s)
        s = re.sub(r"[sS]", "", s)
        s = re.sub(r"\s+", ":", s)
        s = re.sub(r":+", ":", s)
        s = s.strip(":")

        # Try sexagesimal first.
        parts = s.split(":")

        if len(parts) == 3:
            h, m, sec = map(float, parts)
            return 15.0 * (
                h + m / 60.0 + sec / 3600.0
            )

        # Fallback: numeric degrees.
        value_float = float(s)
        return value_float

    except Exception:
        return np.nan


def parse_dec(value):
    try:
        s = str(value).strip()

        if not s or s.lower() in {"nan", "none"}:
            return np.nan

        sign = -1.0 if s.startswith("-") else 1.0
        s = s.lstrip("+-")

        s = re.sub(r"[dD]", ":", s)
        s = re.sub(r"[mM]", ":", s)
        s = re.sub(r"[sS]", "", s)
        s = re.sub(r"\s+", ":", s)
        s = re.sub(r":+", ":", s)
        s = s.strip(":")

        parts = s.split(":")

        if len(parts) == 3:
            d, m, sec = map(float, parts)
            return sign * (
                d + m / 60.0 + sec / 3600.0
            )

        value_float = float(s)
        return sign * abs(value_float)

    except Exception:
        return np.nan


gcvs["ra_deg"] = gcvs["RAJ2000"].apply(parse_ra)
gcvs["dec_deg"] = gcvs["DEJ2000"].apply(parse_dec)

valid_coordinates = gcvs[
    gcvs["ra_deg"].notna() &
    gcvs["dec_deg"].notna()
].copy()

print("[OK] Valid coordinates:", len(valid_coordinates))

if len(valid_coordinates) == 0:
    print("\nExample raw RA values:")
    print(gcvs["RAJ2000"].head(10).tolist())

    print("\nExample raw Dec values:")
    print(gcvs["DEJ2000"].head(10).tolist())

    raise RuntimeError(
        "Coordinate conversion still produced zero valid rows."
    )

# ------------------------------------------------------------
# 5. Restrict to EXP-005 field
# ------------------------------------------------------------

print("\n[4/9] Selecting GCVS objects inside NEOWISE field...")

reference = valid_coordinates[
    (valid_coordinates["ra_deg"] >= RA_MIN) &
    (valid_coordinates["ra_deg"] <= RA_MAX) &
    (valid_coordinates["dec_deg"] >= DEC_MIN) &
    (valid_coordinates["dec_deg"] <= DEC_MAX)
].copy()

reference = reference.reset_index(drop=True)

print(
    "[OK] GCVS objects in field:",
    len(reference)
)

# ------------------------------------------------------------
# 6. Build independent reference population
# ------------------------------------------------------------

print("\n[5/9] Building independent reference population...")

reference_population = pd.DataFrame({
    "object_id": reference["GCVS"].astype(str),
    "ra": reference["ra_deg"].astype(float),
    "dec": reference["dec_deg"].astype(float),
    "vartype": reference["VarType"].astype(str).str.strip(),
    "reference_label": "KNOWN_VARIABLE",
    "reference_source": "GCVS"
})

reference_population = (
    reference_population
    .drop_duplicates("object_id")
    .reset_index(drop=True)
)

print(
    "[OK] Unique known variables:",
    len(reference_population)
)

if len(reference_population) > 0:
    print("\nVariability types:")
    print(
        reference_population["vartype"]
        .value_counts()
        .head(20)
    )

# ------------------------------------------------------------
# 7. Load our NEOWISE validation detections
# ------------------------------------------------------------

print("\n[6/9] Loading EXP-005 NEOWISE detections...")

region_path = (
    RESULTS_DIR /
    "exp005_region_detections.csv"
)

if not region_path.exists():
    raise FileNotFoundError(
        f"Missing EXP-005 data: {region_path}"
    )

neowise = pd.read_csv(region_path)

print(
    "[OK] NEOWISE detections:",
    len(neowise)
)

# ------------------------------------------------------------
# 8. Cross-match independent objects with NEOWISE
# ------------------------------------------------------------

print("\n[7/9] Cross-matching reference objects...")

ra_values = neowise["ra"].to_numpy(float)
dec_values = neowise["dec"].to_numpy(float)

dec0 = np.deg2rad(
    float(np.nanmedian(dec_values))
)

matches = []

for _, ref in reference_population.iterrows():

    dra = np.deg2rad(
        ra_values - float(ref["ra"])
    )

    ddec = np.deg2rad(
        dec_values - float(ref["dec"])
    )

    x = dra * np.cos(dec0)
    y = ddec

    separation_arcsec = (
        np.rad2deg(
            np.sqrt(x * x + y * y)
        ) * 3600.0
    )

    best_idx = int(
        np.nanargmin(separation_arcsec)
    )

    best_sep = float(
        separation_arcsec[best_idx]
    )

    if best_sep <= MATCH_RADIUS_ARCSEC:

        row = neowise.iloc[best_idx]

        matches.append({
            "object_id": ref["object_id"],
            "reference_ra": float(ref["ra"]),
            "reference_dec": float(ref["dec"]),
            "vartype": ref["vartype"],
            "reference_label": ref["reference_label"],
            "reference_source": ref["reference_source"],
            "neowise_ra": float(row["ra"]),
            "neowise_dec": float(row["dec"]),
            "separation_arcsec": best_sep
        })

reference_matches = pd.DataFrame(matches)

if len(reference_matches) > 0:
    reference_matches = (
        reference_matches
        .sort_values("separation_arcsec")
        .drop_duplicates("object_id")
        .reset_index(drop=True)
    )

print(
    "[OK] Matched reference objects:",
    len(reference_matches)
)

# ------------------------------------------------------------
# 9. Save EXP-006 results
# ------------------------------------------------------------

print("\n[8/9] Saving results...")

population_path = (
    RESULTS_DIR /
    "exp006_independent_reference_population.csv"
)

matches_path = (
    RESULTS_DIR /
    "exp006_neowise_reference_matches.csv"
)

summary_path = (
    RESULTS_DIR /
    "exp006_reference_population_summary.json"
)

reference_population.to_csv(
    population_path,
    index=False
)

reference_matches.to_csv(
    matches_path,
    index=False
)

summary = {
    "experiment": EXP_ID,
    "reference_catalog": "GCVS",
    "access_service": "VizieR",
    "field": {
        "ra_min": RA_MIN,
        "ra_max": RA_MAX,
        "dec_min": DEC_MIN,
        "dec_max": DEC_MAX
    },
    "match_radius_arcsec": MATCH_RADIUS_ARCSEC,
    "catalog_rows_downloaded": int(len(gcvs)),
    "valid_coordinate_rows": int(len(valid_coordinates)),
    "reference_objects_in_field": int(
        len(reference_population)
    ),
    "matched_reference_objects": int(
        len(reference_matches)
    ),
    "reference_label": "KNOWN_VARIABLE",
    "independent_label": True,
    "circular_validation": False,
    "note": (
        "Reference labels originate from the external "
        "GCVS catalog rather than the EXP-005 variability "
        "criteria."
    )
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("[OK] Results saved.")

# ------------------------------------------------------------
# 10. Final scientific summary
# ------------------------------------------------------------

print("\n[9/9] Population summary")

print("\n" + "=" * 70)
print("EXP-006 COMPLETE")
print("=" * 70)

print("\nEXTERNAL REFERENCE")
print(
    "GCVS rows downloaded       :",
    len(gcvs)
)

print(
    "Valid coordinate rows      :",
    len(valid_coordinates)
)

print(
    "Objects inside field       :",
    len(reference_population)
)

print(
    "Matched to NEOWISE         :",
    len(reference_matches)
)

if len(reference_matches) > 0:

    print(
        "Median separation          :",
        round(
            reference_matches[
                "separation_arcsec"
            ].median(),
            4
        ),
        "arcsec"
    )

    print("\nMatched variability types:")
    print(
        reference_matches[
            "vartype"
        ].value_counts().head(20)
    )

else:
    print(
        "\nNo external variables matched "
        "the current NEOWISE field."
    )

print("\nOUTPUT FILES")
print(" ", population_path)
print(" ", matches_path)
print(" ", summary_path)

print("\nSCIENTIFIC INTERPRETATION")
print(
    "These labels are external reference labels."
)
print(
    "They were not generated by our validator."
)
print(
    "EXP-007 will compare our validator's behavior "
    "against these independently characterized objects."
)

print("=" * 70)

EXP-006 — INDEPENDENT REFERENCE POPULATION

[1/9] Downloading GCVS reference catalog...
[OK] Response received.
Response size: 2356930 characters

[2/9] Parsing catalog...
[OK] Rows: 60896
[OK] Columns: ['GCVS', 'RAJ2000', 'DEJ2000', 'VarType']

[3/9] Converting J2000 coordinates...
[OK] Valid coordinates: 60715

[4/9] Selecting GCVS objects inside NEOWISE field...
[OK] GCVS objects in field: 1

[5/9] Building independent reference population...
[OK] Unique known variables: 1

Variability types:
vartype
RRAB    1
Name: count, dtype: int64

[6/9] Loading EXP-005 NEOWISE detections...
[OK] NEOWISE detections: 7275

[7/9] Cross-matching reference objects...
[OK] Matched reference objects: 1

[8/9] Saving results...
[OK] Results saved.

[9/9] Population summary

EXP-006 COMPLETE

EXTERNAL REFERENCE
GCVS rows downloaded       : 60896
Valid coordinate rows      : 60715
Objects inside field       : 1
Matched to NEOWISE         : 1
Median separation          : 1.2322 arcsec

Matched variabilit

In [4]:
# ======================================================================
# EXP-007 — EXTERNAL-LABEL VALIDATOR TEST
# ======================================================================
#
# PURPOSE
# -------
# Test the current Phase-III validator against an independently
# characterized astronomical variable from GCVS.
#
# EXP-006 supplied the external label.
# EXP-007 now asks whether our validator independently identifies
# variability in the corresponding NEOWISE observations.
#
# IMPORTANT:
# This is NOT a final performance evaluation.
# EXP-006 currently contains only ONE matched GCVS object.
# Therefore precision/recall/accuracy claims are scientifically invalid.
#
# ======================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import math

# ----------------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------------

PROJECT_DIR = Path("astronomy_exp005")
RESULTS_DIR = PROJECT_DIR / "results"

REFERENCE_FILE = RESULTS_DIR / "exp006_independent_reference_population.csv"
MATCH_FILE = RESULTS_DIR / "exp006_neowise_reference_matches.csv"
DETECTIONS_FILE = RESULTS_DIR / "exp005_region_detections.csv"
GROUPS_FILE = RESULTS_DIR / "exp005_all_source_groups.csv"

OUTPUT_FILE = RESULTS_DIR / "exp007_external_validator_test.csv"
SUMMARY_FILE = RESULTS_DIR / "exp007_external_validator_summary.json"

print("=" * 70)
print("EXP-007 — EXTERNAL-LABEL VALIDATOR TEST")
print("=" * 70)


# ======================================================================
# 1. LOAD EXP-006 EXTERNAL REFERENCE
# ======================================================================

print("\n[1/9] Loading EXP-006 external reference population...")

if not REFERENCE_FILE.exists():
    raise FileNotFoundError(
        f"Missing EXP-006 reference population:\n{REFERENCE_FILE}"
    )

if not MATCH_FILE.exists():
    raise FileNotFoundError(
        f"Missing EXP-006 NEOWISE matches:\n{MATCH_FILE}"
    )

reference = pd.read_csv(REFERENCE_FILE)
matches = pd.read_csv(MATCH_FILE)

print("[OK] External reference rows:", len(reference))
print("[OK] NEOWISE match rows:", len(matches))

if len(reference) == 0:
    raise RuntimeError("EXP-006 reference population is empty.")

if len(matches) == 0:
    raise RuntimeError("EXP-006 produced no NEOWISE matches.")


# ======================================================================
# 2. IDENTIFY THE EXTERNAL OBJECT
# ======================================================================

print("\n[2/9] Inspecting independently labelled object...")

print("Reference columns:")
print(reference.columns.tolist())

print("\nMatch columns:")
print(matches.columns.tolist())

# ----------------------------------------------------------------------
# Locate likely external label column.
# ----------------------------------------------------------------------

label_candidates = [
    "vartype",
    "VarType",
    "variable_type",
    "variability_type",
    "label"
]

label_col = None

for col in label_candidates:
    if col in reference.columns:
        label_col = col
        break

if label_col is None:
    raise RuntimeError(
        "Could not identify the external variability-type column."
    )

# ----------------------------------------------------------------------
# Locate coordinates.
# ----------------------------------------------------------------------

ra_candidates = ["ra", "RA", "RAJ2000", "ra_deg", "RA_deg"]
dec_candidates = ["dec", "DEC", "DEJ2000", "dec_deg", "DEC_deg"]

ref_ra_col = next(
    (c for c in ra_candidates if c in reference.columns),
    None
)

ref_dec_col = next(
    (c for c in dec_candidates if c in reference.columns),
    None
)

# ----------------------------------------------------------------------
# The current EXP-006 result should contain exactly one object.
# ----------------------------------------------------------------------

reference = reference.copy()

print("\nExternal labels:")
print(reference[label_col].value_counts(dropna=False))

print("\nNumber of independently labelled objects:", len(reference))

if len(reference) != 1:
    print(
        "[WARNING] EXP-006 currently contains more than one object. "
        "EXP-007 will evaluate all available objects."
    )


# ======================================================================
# 3. LOAD THE ORIGINAL NEOWISE DETECTIONS
# ======================================================================

print("\n[3/9] Loading EXP-005 NEOWISE detections...")

if not DETECTIONS_FILE.exists():
    raise FileNotFoundError(
        f"Missing EXP-005 detections:\n{DETECTIONS_FILE}"
    )

detections = pd.read_csv(DETECTIONS_FILE)

print("[OK] NEOWISE detection rows:", len(detections))
print("Detection columns:")
print(detections.columns.tolist())


# ======================================================================
# 4. NORMALIZE COLUMN NAMES
# ======================================================================

print("\n[4/9] Detecting required photometric columns...")

# ----------------------------------------------------------------------
# Helper for case-insensitive column discovery.
# ----------------------------------------------------------------------

def find_column(df, candidates):
    lower_map = {str(c).lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    # substring fallback
    for c in df.columns:
        cl = str(c).lower()
        for candidate in candidates:
            if candidate.lower() in cl:
                return c

    return None


ra_col = find_column(
    detections,
    ["ra", "ra_deg", "RAJ2000"]
)

dec_col = find_column(
    detections,
    ["dec", "dec_deg", "DEJ2000"]
)

w1_col = find_column(
    detections,
    ["w1mpro", "w1_mag", "w1", "w1m"]
)

w1err_col = find_column(
    detections,
    ["w1sigmpro", "w1_err", "w1_error", "w1sigm"]
)

qual_col = find_column(
    detections,
    ["qual_frame", "qual", "quality", "qi_fact", "qi"]
)

mjd_col = find_column(
    detections,
    ["mjd", "mjd_obs", "mjdmean", "mjdstart"]
)

print("RA column       :", ra_col)
print("DEC column      :", dec_col)
print("W1 column       :", w1_col)
print("W1 error column :", w1err_col)
print("Quality column  :", qual_col)
print("Time column     :", mjd_col)

if ra_col is None or dec_col is None:
    raise RuntimeError("RA/DEC columns could not be identified.")

if w1_col is None:
    raise RuntimeError("W1 photometry column could not be identified.")

if w1err_col is None:
    raise RuntimeError("W1 uncertainty column could not be identified.")

if mjd_col is None:
    raise RuntimeError("Observation-time column could not be identified.")


# ======================================================================
# 5. CROSS-MATCH EXTERNAL OBJECTS TO NEOWISE OBSERVATIONS
# ======================================================================

print("\n[5/9] Recovering NEOWISE observations for external objects...")

# ----------------------------------------------------------------------
# Use the same small-angle tangent-plane approximation employed in
# EXP-005/EXP-006.
#
# 1 degree = 3600 arcsec.
# RA distances are corrected by cos(dec).
# ----------------------------------------------------------------------

def angular_distance_arcsec(ra1, dec1, ra2, dec2):
    ra1 = np.asarray(ra1, dtype=float)
    dec1 = np.asarray(dec1, dtype=float)
    ra2 = np.asarray(ra2, dtype=float)
    dec2 = np.asarray(dec2, dtype=float)

    mean_dec = np.deg2rad((dec1 + dec2) / 2.0)

    dra = (ra1 - ra2) * np.cos(mean_dec)
    ddec = dec1 - dec2

    return np.sqrt(dra**2 + ddec**2) * 3600.0


# ----------------------------------------------------------------------
# Convert detection coordinates to numeric.
# ----------------------------------------------------------------------

detections = detections.copy()

detections["_ra_numeric"] = pd.to_numeric(
    detections[ra_col],
    errors="coerce"
)

detections["_dec_numeric"] = pd.to_numeric(
    detections[dec_col],
    errors="coerce"
)

detections["_w1_numeric"] = pd.to_numeric(
    detections[w1_col],
    errors="coerce"
)

detections["_w1err_numeric"] = pd.to_numeric(
    detections[w1err_col],
    errors="coerce"
)

detections["_time_numeric"] = pd.to_numeric(
    detections[mjd_col],
    errors="coerce"
)

# ----------------------------------------------------------------------
# Quality handling.
#
# EXP-005 used a quality-aware population. We attempt to reconstruct
# a usable quality indicator from the available column.
# ----------------------------------------------------------------------

if qual_col is not None:

    detections["_quality_numeric"] = pd.to_numeric(
        detections[qual_col],
        errors="coerce"
    )

    # NEOWISE quality factors are treated conservatively:
    # positive/nonzero values are retained as potentially usable.
    detections["_quality_good"] = (
        detections["_quality_numeric"].fillna(0) > 0
    )

else:

    # If the quality field is unavailable in the saved CSV,
    # retain the photometric observations but mark quality as unknown.
    detections["_quality_good"] = True


# ----------------------------------------------------------------------
# Remove unusable photometry.
# ----------------------------------------------------------------------

valid_detection_mask = (
    np.isfinite(detections["_ra_numeric"])
    & np.isfinite(detections["_dec_numeric"])
    & np.isfinite(detections["_w1_numeric"])
    & np.isfinite(detections["_w1err_numeric"])
    & np.isfinite(detections["_time_numeric"])
    & (detections["_w1err_numeric"] > 0)
)

detections_valid = detections.loc[
    valid_detection_mask
].copy()

print(
    "[OK] Usable coordinate/photometry/time rows:",
    len(detections_valid)
)


# ----------------------------------------------------------------------
# Match each external object.
# ----------------------------------------------------------------------

matched_records = []

for ref_idx, ref_row in reference.iterrows():

    # Prefer reference coordinates directly.
    if ref_ra_col is None or ref_dec_col is None:
        raise RuntimeError(
            "EXP-006 reference file does not contain usable RA/DEC columns."
        )

    ref_ra = pd.to_numeric(
        ref_row[ref_ra_col],
        errors="coerce"
    )

    ref_dec = pd.to_numeric(
        ref_row[ref_dec_col],
        errors="coerce"
    )

    if not np.isfinite(ref_ra) or not np.isfinite(ref_dec):
        print(
            f"[WARNING] Reference object {ref_idx} has invalid coordinates."
        )
        continue

    distances = angular_distance_arcsec(
        detections_valid["_ra_numeric"].values,
        detections_valid["_dec_numeric"].values,
        ref_ra,
        ref_dec
    )

    # Conservative 5 arcsec match radius.
    mask = distances <= 5.0

    obj = detections_valid.loc[mask].copy()
    obj["_separation_arcsec"] = distances[mask]

    if len(obj) == 0:
        print(
            f"[WARNING] No NEOWISE detections within 5 arcsec "
            f"for reference object {ref_idx}."
        )
        continue

    obj["_reference_index"] = ref_idx
    obj["_external_label"] = str(ref_row[label_col])

    matched_records.append(obj)

if not matched_records:
    raise RuntimeError(
        "EXP-007 could not recover any NEOWISE observations "
        "for the external reference object."
    )

matched = pd.concat(
    matched_records,
    ignore_index=True
)

print(
    "[OK] Matched NEOWISE observations:",
    len(matched)
)

print(
    "[OK] External objects with observations:",
    matched["_reference_index"].nunique()
)


# ======================================================================
# 6. APPLY THE CURRENT VALIDATOR
# ======================================================================

print("\n[6/9] Applying current variability validator...")


# ----------------------------------------------------------------------
# Current validator principles from EXP-005:
#
# 1. Enough observations
# 2. Sufficient signal-to-noise
# 3. Estimate scatter relative to reported errors
# 4. Compute reduced chi-square against a constant-flux model
# 5. Require high-quality observations when making the strongest claim
#
# We intentionally do NOT use the external GCVS label to modify the
# score. The label is used only AFTER scoring for comparison.
# ----------------------------------------------------------------------

def calculate_validator_metrics(group):

    group = group.copy()

    mags = pd.to_numeric(
        group["_w1_numeric"],
        errors="coerce"
    ).values

    errs = pd.to_numeric(
        group["_w1err_numeric"],
        errors="coerce"
    ).values

    finite = (
        np.isfinite(mags)
        & np.isfinite(errs)
        & (errs > 0)
    )

    mags = mags[finite]
    errs = errs[finite]

    n = len(mags)

    if n == 0:
        return {
            "n_obs": 0,
            "median_snr_proxy": np.nan,
            "median_w1_err": np.nan,
            "w1_mean": np.nan,
            "w1_std": np.nan,
            "w1_range": np.nan,
            "reduced_chi2": np.nan,
            "quality_fraction": np.nan,
        }

    mean_mag = np.mean(mags)

    # Constant-magnitude chi-square.
    chi2 = np.sum(
        ((mags - mean_mag) / errs) ** 2
    )

    dof = max(n - 1, 1)

    reduced_chi2 = chi2 / dof

    # For magnitude data, inverse uncertainty is a useful SNR-like
    # proxy when no explicit flux/SNR column is available.
    snr_proxy = 1.0 / errs

    if qual_col is not None and "_quality_good" in group.columns:
        quality_fraction = float(
            np.mean(group["_quality_good"].values)
        )
    else:
        quality_fraction = np.nan

    return {
        "n_obs": int(n),
        "median_snr_proxy": float(np.median(snr_proxy)),
        "median_w1_err": float(np.median(errs)),
        "w1_mean": float(mean_mag),
        "w1_std": float(np.std(mags, ddof=1)) if n > 1 else 0.0,
        "w1_range": float(np.max(mags) - np.min(mags)),
        "reduced_chi2": float(reduced_chi2),
        "quality_fraction": quality_fraction,
    }


# ----------------------------------------------------------------------
# Validator thresholds.
#
# These are deliberately based on the Phase-III population construction
# rather than optimized against GCVS. This prevents label leakage.
# ----------------------------------------------------------------------

MIN_OBS = 10
MIN_SNR_PROXY = 5.0
VARIABLE_RCHI2_THRESHOLD = 2.0
GOOD_QUALITY_FRACTION = 0.90


results = []

for ref_idx, group in matched.groupby("_reference_index"):

    metrics = calculate_validator_metrics(group)

    external_label = str(
        group["_external_label"].iloc[0]
    )

    variable_signal = (
        np.isfinite(metrics["reduced_chi2"])
        and metrics["reduced_chi2"] >= VARIABLE_RCHI2_THRESHOLD
    )

    enough_observations = (
        metrics["n_obs"] >= MIN_OBS
    )

    sufficient_snr = (
        np.isfinite(metrics["median_snr_proxy"])
        and metrics["median_snr_proxy"] >= MIN_SNR_PROXY
    )

    good_quality = (
        np.isnan(metrics["quality_fraction"])
        or metrics["quality_fraction"] >= GOOD_QUALITY_FRACTION
    )

    # --------------------------------------------------------------
    # Main validator decision.
    #
    # "VARIABLE" means the source shows variability according to
    # the current internal statistical criterion and has enough
    # observations/SNR.
    #
    # "VARIABLE_HIGH_QUALITY" additionally requires >=90% good
    # observations.
    # --------------------------------------------------------------

    if (
        variable_signal
        and enough_observations
        and sufficient_snr
    ):
        validator_class = "VARIABLE"

        if good_quality:
            validator_class = "VARIABLE_HIGH_QUALITY"

    elif (
        enough_observations
        and sufficient_snr
    ):
        validator_class = "NOT_VARIABLE_BY_CURRENT_GATE"

    else:
        validator_class = "INSUFFICIENT_DATA"

    row = {
        "reference_index": ref_idx,
        "external_vartype": external_label,
        **metrics,
        "variable_signal": bool(variable_signal),
        "enough_observations": bool(enough_observations),
        "sufficient_snr": bool(sufficient_snr),
        "good_quality": bool(good_quality),
        "validator_class": validator_class,
    }

    results.append(row)


results_df = pd.DataFrame(results)

print("\nValidator results:")
print(results_df.to_string(index=False))


# ======================================================================
# 7. INDEPENDENT-LABEL COMPARISON
# ======================================================================

print("\n[7/9] Comparing validator output with external labels...")


# ----------------------------------------------------------------------
# GCVS is a known-variable catalog.
#
# We therefore expect the validator to show VARIABLE behavior for
# these objects if their NEOWISE observations contain sufficient
# information.
# ----------------------------------------------------------------------

results_df["external_known_variable"] = True

results_df["external_label_agrees"] = (
    results_df["validator_class"].isin([
        "VARIABLE",
        "VARIABLE_HIGH_QUALITY"
    ])
)

results_df["external_label_disagrees"] = (
    ~results_df["external_label_agrees"]
)


# ----------------------------------------------------------------------
# Do NOT call disagreement a "false negative" in a final scientific
# sense yet. There are several possible explanations:
#
# - insufficient NEOWISE sampling
# - low SNR
# - quality flags
# - wavelength-dependent behavior
# - GCVS/NEOWISE epoch differences
# - imperfect current validator thresholds
#
# Therefore use the scientifically neutral terminology below.
# ----------------------------------------------------------------------

results_df["interpretation"] = np.where(
    results_df["external_label_agrees"],
    "KNOWN_VARIABLE_RECOGNIZED",
    "KNOWN_VARIABLE_NOT_RECOGNIZED_BY_CURRENT_GATE"
)


# ======================================================================
# 8. SAVE EXP-007 RESULTS
# ======================================================================

print("\n[8/9] Saving EXP-007 results...")

results_df.to_csv(
    OUTPUT_FILE,
    index=False
)

# ----------------------------------------------------------------------
# Summary statistics.
# ----------------------------------------------------------------------

n_objects = len(results_df)
n_recognized = int(
    results_df["external_label_agrees"].sum()
)

n_not_recognized = int(
    results_df["external_label_disagrees"].sum()
)

recognition_fraction = (
    n_recognized / n_objects
    if n_objects > 0
    else np.nan
)

summary = {
    "experiment": "EXP-007",
    "purpose": (
        "Test current variability validator against "
        "independently labelled GCVS variables."
    ),
    "external_catalog": "GCVS",
    "external_objects_tested": int(n_objects),
    "known_variables_recognized": n_recognized,
    "known_variables_not_recognized": n_not_recognized,
    "recognition_fraction": (
        float(recognition_fraction)
        if np.isfinite(recognition_fraction)
        else None
    ),
    "validator_thresholds": {
        "minimum_observations": MIN_OBS,
        "minimum_snr_proxy": MIN_SNR_PROXY,
        "variable_reduced_chi2_threshold": VARIABLE_RCHI2_THRESHOLD,
        "good_quality_fraction": GOOD_QUALITY_FRACTION,
    },
    "scientific_status": (
        "PIPELINE_CONSISTENCY_TEST_ONLY"
    ),
    "statistical_warning": (
        "The current EXP-006 external reference population "
        "contains only one matched GCVS object. This sample is "
        "insufficient for estimating general validatoraccuracy, "
        "precision, recall, completeness, or false-positive rates."
    ),
}

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print("[OK] Results saved.")
print("  ", OUTPUT_FILE)
print("  ", SUMMARY_FILE)


# ======================================================================
# 9. SCIENTIFIC REPORT
# ======================================================================

print("\n[9/9] EXP-007 scientific report")

print("=" * 70)
print("EXP-007 COMPLETE")
print("=" * 70)

print("\nEXTERNAL REFERENCE")
print("Objects tested              :", n_objects)
print("Known variables recognized  :", n_recognized)
print("Not recognized              :", n_not_recognized)

if np.isfinite(recognition_fraction):
    print(
        "Recognition fraction        :",
        f"{recognition_fraction:.3f}"
    )

print("\nVALIDATOR THRESHOLDS")
print("Minimum observations        :", MIN_OBS)
print("Minimum SNR proxy           :", MIN_SNR_PROXY)
print("Variable reduced chi2       :", VARIABLE_RCHI2_THRESHOLD)
print("Good-quality fraction       :", GOOD_QUALITY_FRACTION)

print("\nOBJECT-LEVEL RESULTS")

for _, row in results_df.iterrows():

    print("-" * 70)

    print(
        "External GCVS type         :",
        row["external_vartype"]
    )

    print(
        "NEOWISE observations       :",
        row["n_obs"]
    )

    print(
        "W1 reduced chi2            :",
        f"{row['reduced_chi2']:.4f}"
        if np.isfinite(row["reduced_chi2"])
        else "NaN"
    )

    print(
        "W1 scatter                 :",
        f"{row['w1_std']:.5f}"
        if np.isfinite(row["w1_std"])
        else "NaN"
    )

    print(
        "W1 range                   :",
        f"{row['w1_range']:.5f}"
        if np.isfinite(row["w1_range"])
        else "NaN"
    )

    print(
        "Quality fraction           :",
        f"{row['quality_fraction']:.3f}"
        if np.isfinite(row["quality_fraction"])
        else "unknown"
    )

    print(
        "Validator classification   :",
        row["validator_class"]
    )

    print(
        "External-label agreement   :",
        "YES" if row["external_label_agrees"] else "NO"
    )

    print(
        "Interpretation             :",
        row["interpretation"]
    )


print("\nSCIENTIFIC INTERPRETATION")
print(
    "EXP-007 evaluates the validator against an external catalog "
    "label rather than an internally generated label."
)

print(
    "The GCVS label was NOT used to calculate the validator score."
)

if n_recognized == n_objects:
    print(
        "The independently known variable object(s) were recognized "
        "as variable by the current NEOWISE gate."
    )
elif n_recognized == 0:
    print(
        "The current validator did not recognize the independently "
        "known variable object(s). This does NOT prove the validator "
        "is wrong; the result requires investigation of sampling, "
        "SNR, quality flags, wavelength behavior, and threshold choice."
    )
else:
    print(
        "The validator recognized some but not all external variables."
    )

print(
    "\nIMPORTANT: Because only one GCVS object is currently matched, "
    "this experiment is a consistency test, NOT a statistical "
    "validation benchmark."
)

print(
    "\nEXP-008 should expand the independently labelled population "
    "and/or introduce an independently characterized non-variable "
    "control population before we calculate meaningful "
    "completeness, false-positive rate, precision, or recall."
)

print("=" * 70)

EXP-007 — EXTERNAL-LABEL VALIDATOR TEST

[1/9] Loading EXP-006 external reference population...
[OK] External reference rows: 1
[OK] NEOWISE match rows: 1

[2/9] Inspecting independently labelled object...
Reference columns:
['object_id', 'ra', 'dec', 'vartype', 'reference_label', 'reference_source']

Match columns:
['object_id', 'reference_ra', 'reference_dec', 'vartype', 'reference_label', 'reference_source', 'neowise_ra', 'neowise_dec', 'separation_arcsec']

External labels:
vartype
RRAB    1
Name: count, dtype: int64

Number of independently labelled objects: 1

[3/9] Loading EXP-005 NEOWISE detections...
[OK] NEOWISE detection rows: 7275
Detection columns:
['ra', 'dec', 'mjd', 'w1mpro', 'w1sigmpro', 'w2mpro', 'w2sigmpro', 'w1snr', 'w2snr', 'qual_frame', 'ph_qual', 'cc_flags', 'scan_id', 'frame_num', 'good_quality', 'quality_flagged', 'candidate1_sep_arcsec', 'group_id']

[4/9] Detecting required photometric columns...
RA column       : ra
DEC column      : dec
W1 column       : w1

In [5]:
import io
import json
import numpy as np
import pandas as pd
import requests
from astropy.coordinates import SkyCoord
import astropy.units as u

print("EXP-008 — EXPANDED EXTERNAL VALIDATION")

BASE = "astronomy_exp005/results"
NEOWISE_FILE = f"{BASE}/exp005_region_detections.csv"

RA_MIN, RA_MAX = 179.9, 180.1
DEC_MIN, DEC_MAX = -0.1, 0.1
MATCH_RADIUS = 3.0

MIN_OBS = 10
MIN_SNR = 5.0
VARIABLE_RCHI2 = 2.0
GOOD_QUALITY_FRACTION = 0.90

# Load EXP-005 data
neowise = pd.read_csv(NEOWISE_FILE)

ra_col = "ra"
dec_col = "dec"
time_col = "mjd"
w1_col = "w1mpro"
w1err_col = "w1sigmpro"
snr_col = "w1snr"
quality_col = "good_quality"

neowise = neowise.copy()

for col in [ra_col, dec_col, time_col, w1_col, w1err_col]:
    neowise[col] = pd.to_numeric(neowise[col], errors="coerce")

neowise = neowise.dropna(
    subset=[ra_col, dec_col, time_col, w1_col, w1err_col]
)

print(f"[OK] NEOWISE rows: {len(neowise)}")

# Query Gaia DR3 variability summary
url = "https://vizier.cds.unistra.fr/viz-bin/asu-tsv"

params = {
    "-source": "I/358/varisum",
    "-out": "Source,RA_ICRS,DE_ICRS",
    "RA_ICRS": f"{RA_MIN}..{RA_MAX}",
    "DE_ICRS": f"{DEC_MIN}..{DEC_MAX}",
    "-out.max": "100000",
}

print("[INFO] Querying Gaia DR3...")

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

text = response.text
lines = text.splitlines()

print(f"[OK] Response lines: {len(lines)}")

# Find actual table header.
header_idx = None

for i, line in enumerate(lines):
    clean = line.strip().lstrip("#").strip()

    if (
        "Source" in clean
        and "RA_ICRS" in clean
        and "DE_ICRS" in clean
        and "\t" in line
    ):
        header_idx = i
        break

if header_idx is None:
    print("\nVizieR response preview:")
    print("\n".join(lines[:40]))
    raise RuntimeError("Could not locate the VizieR data header.")

print(f"[OK] Table header found at line {header_idx}")

# Remove VizieR metadata and parse only the table.
table_lines = []

for line in lines[header_idx:]:
    if not line.strip():
        continue

    if line.startswith("#"):
        continue

    table_lines.append(line)

if len(table_lines) < 2:
    raise RuntimeError("VizieR returned no Gaia data rows.")

table_text = "\n".join(table_lines)

gaia = pd.read_csv(
    io.StringIO(table_text),
    sep="\t",
    dtype=str
)

print(f"[OK] Gaia rows: {len(gaia)}")
print("Gaia columns:", list(gaia.columns))

# Clean Gaia coordinates
gaia["RA_ICRS"] = pd.to_numeric(gaia["RA_ICRS"], errors="coerce")
gaia["DE_ICRS"] = pd.to_numeric(gaia["DE_ICRS"], errors="coerce")

gaia = gaia.dropna(subset=["RA_ICRS", "DE_ICRS"]).copy()

print(f"[OK] Gaia rows with coordinates: {len(gaia)}")

if len(gaia) == 0:
    raise RuntimeError("No Gaia sources were returned in the requested field.")

# Coordinate matching
neowise_coord = SkyCoord(
    ra=neowise[ra_col].to_numpy() * u.deg,
    dec=neowise[dec_col].to_numpy() * u.deg
)

gaia_coord = SkyCoord(
    ra=gaia["RA_ICRS"].to_numpy() * u.deg,
    dec=gaia["DE_ICRS"].to_numpy() * u.deg
)

idx, sep2d, _ = gaia_coord.match_to_catalog_sky(neowise_coord)

gaia["neowise_index"] = idx
gaia["separation_arcsec"] = sep2d.arcsec

matches = gaia[gaia["separation_arcsec"] <= MATCH_RADIUS].copy()

print(f"[OK] Gaia/NEOWISE matches within {MATCH_RADIUS:.1f} arcsec: {len(matches)}")

if len(matches) == 0:
    print("No external Gaia sources matched the current NEOWISE field.")

    summary = {
        "experiment": "EXP-008",
        "gaia_sources": int(len(gaia)),
        "matched_sources": 0,
        "status": "NO_MATCHES"
    }

    with open(
        f"{BASE}/exp008_expanded_external_validator_summary.json",
        "w"
    ) as f:
        json.dump(summary, f, indent=2)

else:
    # Apply validator independently of Gaia labels.
    results = []

    for i, row in matches.iterrows():
        nw_idx = int(row["neowise_index"])

        # Use the matched NEOWISE position to recover its source-like group.
        target = neowise.iloc[nw_idx]

        sep = SkyCoord(
            ra=target[ra_col] * u.deg,
            dec=target[dec_col] * u.deg
        ).separation(neowise_coord).arcsec

        # Same local source region.
        obs = neowise[sep <= MATCH_RADIUS].copy()

        obs = obs.sort_values(time_col)

        if len(obs) == 0:
            continue

        mags = obs[w1_col].to_numpy(float)
        errs = obs[w1err_col].to_numpy(float)

        valid = np.isfinite(mags) & np.isfinite(errs) & (errs > 0)

        mags = mags[valid]
        errs = errs[valid]

        if len(mags) == 0:
            continue

        mean_mag = np.mean(mags)
        std_mag = np.std(mags, ddof=1) if len(mags) > 1 else np.nan
        mag_range = np.ptp(mags) if len(mags) > 1 else 0.0

        chi2 = np.sum(((mags - mean_mag) / errs) ** 2)

        dof = max(len(mags) - 1, 1)
        reduced_chi2 = chi2 / dof

        if snr_col in obs.columns:
            snr = pd.to_numeric(obs[snr_col], errors="coerce")
            median_snr = float(snr.median())
        else:
            median_snr = float(np.nanmedian(1.0 / errs))

        if quality_col in obs.columns:
            quality_fraction = float(
                pd.to_numeric(obs[quality_col], errors="coerce").mean()
            )
        else:
            quality_fraction = np.nan

        variable_signal = reduced_chi2 >= VARIABLE_RCHI2
        enough_observations = len(mags) >= MIN_OBS
        sufficient_snr = median_snr >= MIN_SNR
        good_quality = (
            quality_fraction >= GOOD_QUALITY_FRACTION
            if np.isfinite(quality_fraction)
            else True
        )

        validator_positive = (
            variable_signal
            and enough_observations
            and sufficient_snr
            and good_quality
        )

        if validator_positive:
            validator_class = "VARIABLE_HIGH_QUALITY"
        elif variable_signal:
            validator_class = "VARIABLE_LOW_CONFIDENCE"
        else:
            validator_class = "NOT_VARIABLE"

        results.append({
            "source": row["Source"],
            "gaia_ra": row["RA_ICRS"],
            "gaia_dec": row["DE_ICRS"],
            "separation_arcsec": row["separation_arcsec"],
            "n_obs": len(mags),
            "median_snr": median_snr,
            "w1_mean": mean_mag,
            "w1_std": std_mag,
            "w1_range": mag_range,
            "reduced_chi2": reduced_chi2,
            "quality_fraction": quality_fraction,
            "variable_signal": variable_signal,
            "enough_observations": enough_observations,
            "sufficient_snr": sufficient_snr,
            "good_quality": good_quality,
            "validator_class": validator_class
        })

    results = pd.DataFrame(results)

    print("\nValidator results:")
    print(results.to_string(index=False))

    # Save results.
    results.to_csv(
        f"{BASE}/exp008_expanded_external_validator_test.csv",
        index=False
    )

    matches.to_csv(
        f"{BASE}/exp008_gaia_neowise_matches.csv",
        index=False
    )

    gaia.to_csv(
        f"{BASE}/exp008_gaia_external_population.csv",
        index=False
    )

    summary = {
        "experiment": "EXP-008",
        "gaia_sources_in_field": int(len(gaia)),
        "gaia_neowise_matches": int(len(matches)),
        "validator_objects_tested": int(len(results)),
        "validator_positive": int(
            results["validator_class"].isin(
                ["VARIABLE_HIGH_QUALITY", "VARIABLE_LOW_CONFIDENCE"]
            ).sum()
        ),
        "validator_negative": int(
            (results["validator_class"] == "NOT_VARIABLE").sum()
        ),
        "note": (
            "Gaia varisum is an external variability-related reference "
            "catalog. It is not by itself a statistically complete set "
            "of positive and negative labels."
        )
    }

    with open(
        f"{BASE}/exp008_expanded_external_validator_summary.json",
        "w"
    ) as f:
        json.dump(summary, f, indent=2)

    print("\nEXP-008 COMPLETE")
    print("=" * 60)
    print(f"Gaia sources in field : {len(gaia)}")
    print(f"Matched sources       : {len(matches)}")
    print(f"Objects tested        : {len(results)}")
    print(
        "Validator positive    :",
        int(results["validator_class"].isin(
            ["VARIABLE_HIGH_QUALITY", "VARIABLE_LOW_CONFIDENCE"]
        ).sum())
    )
    print(
        "Validator negative    :",
        int((results["validator_class"] == "NOT_VARIABLE").sum())
    )
    print("\nImportant:")
    print("This is still not a precision/recall benchmark.")
    print("EXP-008 expands the external reference population.")

EXP-008 — EXPANDED EXTERNAL VALIDATION
[OK] NEOWISE rows: 7275
[INFO] Querying Gaia DR3...
[OK] Response lines: 41
[OK] Table header found at line 9
[OK] Gaia rows: 5
Gaia columns: ['Source', 'RA_ICRS', 'DE_ICRS']
[OK] Gaia rows with coordinates: 3
[OK] Gaia/NEOWISE matches within 3.0 arcsec: 3

Validator results:
             source    gaia_ra  gaia_dec  separation_arcsec  n_obs  median_snr   w1_mean   w1_std  w1_range  reduced_chi2  quality_fraction  variable_signal  enough_observations  sufficient_snr  good_quality       validator_class
3795032587349812480 179.921149 -0.055561           0.016740     28        17.7 14.152679 0.068298     0.263      1.034698          1.000000            False                 True            True          True          NOT_VARIABLE
3891110460301684864 179.978921  0.049972           0.011936     22        34.9 12.934273 0.037003     0.161      1.312392          1.000000            False                 True            True          True          NOT_VAR

In [6]:
import io
import json
import numpy as np
import pandas as pd
import requests
from astropy.coordinates import SkyCoord
import astropy.units as u

print("EXP-009 — EXTERNAL CLASSIFICATION BENCHMARK")

BASE = "astronomy_exp005/results"
NEOWISE_FILE = f"{BASE}/exp005_region_detections.csv"

RA_MIN, RA_MAX = 179.9, 180.1
DEC_MIN, DEC_MAX = -0.1, 0.1
MATCH_RADIUS = 3.0

MIN_OBS = 10
MIN_SNR = 5.0
VARIABLE_RCHI2 = 2.0
GOOD_QUALITY_FRACTION = 0.90

# Load NEOWISE data
neowise = pd.read_csv(NEOWISE_FILE)

for col in ["ra", "dec", "mjd", "w1mpro", "w1sigmpro"]:
    neowise[col] = pd.to_numeric(neowise[col], errors="coerce")

neowise = neowise.dropna(
    subset=["ra", "dec", "mjd", "w1mpro", "w1sigmpro"]
).copy()

print(f"[OK] NEOWISE rows: {len(neowise)}")

# Query Gaia DR3 variability classification results.
url = "https://vizier.cds.unistra.fr/viz-bin/asu-tsv"

params = {
    "-source": "I/358/vclassre",
    "-out": "Source,RA_ICRS,DE_ICRS,Classifier,Class,ClassSc",
    "RA_ICRS": f"{RA_MIN}..{RA_MAX}",
    "DE_ICRS": f"{DEC_MIN}..{DEC_MAX}",
    "-out.max": "100000"
}

print("[INFO] Querying Gaia DR3 vclassre...")

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

lines = response.text.splitlines()

print(f"[OK] Response lines: {len(lines)}")

# Find the actual VizieR table header.
header_idx = None

for i, line in enumerate(lines):
    clean = line.strip().lstrip("#").strip()

    if (
        "Source" in clean
        and "RA_ICRS" in clean
        and "DE_ICRS" in clean
        and "Class" in clean
        and "\t" in line
    ):
        header_idx = i
        break

if header_idx is None:
    print("\nVizieR response preview:")
    print("\n".join(lines[:40]))
    raise RuntimeError("Could not locate Gaia vclassre table header.")

print(f"[OK] Table header found at line {header_idx}")

table_lines = []

for line in lines[header_idx:]:
    if not line.strip():
        continue
    if line.startswith("#"):
        continue
    table_lines.append(line)

if len(table_lines) < 2:
    raise RuntimeError("Gaia vclassre returned no data rows.")

table_text = "\n".join(table_lines)

gaia = pd.read_csv(
    io.StringIO(table_text),
    sep="\t",
    dtype=str
)

print(f"[OK] Gaia classification rows: {len(gaia)}")
print("Columns:", list(gaia.columns))

# Clean fields.
gaia["RA_ICRS"] = pd.to_numeric(gaia["RA_ICRS"], errors="coerce")
gaia["DE_ICRS"] = pd.to_numeric(gaia["DE_ICRS"], errors="coerce")
gaia["ClassSc"] = pd.to_numeric(gaia["ClassSc"], errors="coerce")

gaia["Class"] = gaia["Class"].astype(str).str.strip()
gaia["Classifier"] = gaia["Classifier"].astype(str).str.strip()

gaia = gaia.dropna(
    subset=["RA_ICRS", "DE_ICRS"]
).copy()

gaia = gaia[
    (gaia["Class"].notna()) &
    (gaia["Class"].str.lower() != "nan") &
    (gaia["Class"].str.strip() != "")
].copy()

print(f"[OK] Gaia rows with classifications: {len(gaia)}")

if len(gaia) == 0:
    raise RuntimeError("No usable Gaia classifications returned.")

print("\nGaia class distribution:")
print(gaia["Class"].value_counts().head(20))

# Cross-match Gaia to NEOWISE.
neowise_coord = SkyCoord(
    ra=neowise["ra"].to_numpy() * u.deg,
    dec=neowise["dec"].to_numpy() * u.deg
)

gaia_coord = SkyCoord(
    ra=gaia["RA_ICRS"].to_numpy() * u.deg,
    dec=gaia["DE_ICRS"].to_numpy() * u.deg
)

idx, sep2d, _ = gaia_coord.match_to_catalog_sky(neowise_coord)

gaia["neowise_index"] = idx
gaia["separation_arcsec"] = sep2d.arcsec

matches = gaia[
    gaia["separation_arcsec"] <= MATCH_RADIUS
].copy()

print(
    f"[OK] Gaia/NEOWISE matches within "
    f"{MATCH_RADIUS:.1f} arcsec: {len(matches)}"
)

if len(matches) == 0:
    raise RuntimeError(
        "No Gaia classified sources matched the NEOWISE field."
    )

# Run our validator independently.
results = []

for _, row in matches.iterrows():
    target = neowise.iloc[int(row["neowise_index"])]

    target_coord = SkyCoord(
        ra=float(target["ra"]) * u.deg,
        dec=float(target["dec"]) * u.deg
    )

    separations = target_coord.separation(neowise_coord).arcsec

    obs = neowise[separations <= MATCH_RADIUS].copy()

    if len(obs) == 0:
        continue

    mags = obs["w1mpro"].to_numpy(float)
    errs = obs["w1sigmpro"].to_numpy(float)

    valid = (
        np.isfinite(mags) &
        np.isfinite(errs) &
        (errs > 0)
    )

    mags = mags[valid]
    errs = errs[valid]

    if len(mags) == 0:
        continue

    mean_mag = np.mean(mags)

    std_mag = (
        np.std(mags, ddof=1)
        if len(mags) > 1
        else np.nan
    )

    mag_range = (
        np.ptp(mags)
        if len(mags) > 1
        else 0.0
    )

    chi2 = np.sum(
        ((mags - mean_mag) / errs) ** 2
    )

    dof = max(len(mags) - 1, 1)

    reduced_chi2 = chi2 / dof

    if "w1snr" in obs.columns:
        snr_values = pd.to_numeric(
            obs["w1snr"],
            errors="coerce"
        )
        median_snr = float(snr_values.median())
    else:
        median_snr = float(
            np.nanmedian(1.0 / errs)
        )

    if "good_quality" in obs.columns:
        quality_values = pd.to_numeric(
            obs["good_quality"],
            errors="coerce"
        )
        quality_fraction = float(
            quality_values.mean()
        )
    else:
        quality_fraction = np.nan

    variable_signal = (
        reduced_chi2 >= VARIABLE_RCHI2
    )

    enough_observations = (
        len(mags) >= MIN_OBS
    )

    sufficient_snr = (
        median_snr >= MIN_SNR
    )

    good_quality = (
        quality_fraction >= GOOD_QUALITY_FRACTION
        if np.isfinite(quality_fraction)
        else True
    )

    validator_positive = (
        variable_signal
        and enough_observations
        and sufficient_snr
        and good_quality
    )

    if validator_positive:
        validator_class = "VARIABLE_HIGH_QUALITY"
    elif variable_signal:
        validator_class = "VARIABLE_LOW_CONFIDENCE"
    else:
        validator_class = "NOT_VARIABLE"

    results.append({
        "source": row["Source"],
        "gaia_ra": row["RA_ICRS"],
        "gaia_dec": row["DE_ICRS"],
        "gaia_classifier": row["Classifier"],
        "gaia_class": row["Class"],
        "gaia_class_score": row["ClassSc"],
        "separation_arcsec": row["separation_arcsec"],
        "n_obs": len(mags),
        "median_snr": median_snr,
        "w1_mean": mean_mag,
        "w1_std": std_mag,
        "w1_range": mag_range,
        "reduced_chi2": reduced_chi2,
        "quality_fraction": quality_fraction,
        "validator_class": validator_class
    })

results = pd.DataFrame(results)

if len(results) == 0:
    raise RuntimeError(
        "No matched objects produced validator measurements."
    )

# Define Gaia-positive labels only from actual classification results.
# We do NOT treat an arbitrary Gaia source as a non-variable.
results["gaia_variable_label"] = True

results["validator_variable_label"] = results[
    "validator_class"
].isin([
    "VARIABLE_HIGH_QUALITY",
    "VARIABLE_LOW_CONFIDENCE"
])

# Agreement.
results["agreement"] = (
    results["gaia_variable_label"]
    == results["validator_variable_label"]
)

print("\nExternal classification benchmark:")
print(
    results[
        [
            "source",
            "gaia_classifier",
            "gaia_class",
            "gaia_class_score",
            "n_obs",
            "reduced_chi2",
            "quality_fraction",
            "validator_class",
            "agreement"
        ]
    ].to_string(index=False)
)

recognized = int(
    results["validator_variable_label"].sum()
)

not_recognized = int(
    (~results["validator_variable_label"]).sum()
)

total = len(results)

recognition_fraction = (
    recognized / total
    if total > 0
    else np.nan
)

summary = {
    "experiment": "EXP-009",
    "gaia_classification_rows": int(len(gaia)),
    "matched_sources": int(len(matches)),
    "objects_tested": int(total),
    "gaia_positive_objects": int(total),
    "validator_positive": recognized,
    "validator_negative": not_recognized,
    "recognition_fraction": recognition_fraction,
    "note": (
        "Gaia vclassre provides external variability "
        "classification results. Because this benchmark "
        "does not contain independently confirmed negative "
        "labels, false-positive and specificity metrics "
        "are not calculated."
    )
}

results.to_csv(
    f"{BASE}/exp009_external_classification_benchmark.csv",
    index=False
)

matches.to_csv(
    f"{BASE}/exp009_gaia_classification_matches.csv",
    index=False
)

gaia.to_csv(
    f"{BASE}/exp009_gaia_classification_population.csv",
    index=False
)

with open(
    f"{BASE}/exp009_external_classification_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=2)

print("\nEXP-009 COMPLETE")
print("=" * 60)
print(f"Gaia classified sources : {len(gaia)}")
print(f"Matched sources         : {len(matches)}")
print(f"Objects tested          : {total}")
print(f"Validator positive      : {recognized}")
print(f"Validator negative      : {not_recognized}")
print(f"Recognition fraction    : {recognition_fraction:.3f}")

print("\nImportant:")
print("Gaia labels were not used by the validator.")
print("No negative-control metrics are calculated yet.")

EXP-009 — EXTERNAL CLASSIFICATION BENCHMARK
[OK] NEOWISE rows: 7275
[INFO] Querying Gaia DR3 vclassre...
[OK] Response lines: 44
[OK] Table header found at line 9
[OK] Gaia classification rows: 5
Columns: ['Source', 'RA_ICRS', 'DE_ICRS', 'Classifier', 'Class', 'ClassSc']
[OK] Gaia rows with classifications: 3

Gaia class distribution:
Class
AGN                1
RR                 1
DSCT|GDOR|SXPHE    1
Name: count, dtype: int64
[OK] Gaia/NEOWISE matches within 3.0 arcsec: 3

External classification benchmark:
             source gaia_classifier      gaia_class  gaia_class_score  n_obs  reduced_chi2  quality_fraction       validator_class  agreement
3795032587349812480    nTransits:5+             AGN          0.709531     28      1.034698          1.000000          NOT_VARIABLE      False
3795033278840733568    nTransits:5+              RR          0.878563     23      2.211583          0.956522 VARIABLE_HIGH_QUALITY       True
3891110460301684864    nTransits:5+ DSCT|GDOR|SXPHE        

In [7]:
# ============================================================
# EXP-010 — INDEPENDENT CONTROL POPULATION
# ============================================================

import io
import json
import numpy as np
import pandas as pd
import requests
from astropy.coordinates import SkyCoord
import astropy.units as u

RESULTS_DIR = "astronomy_exp005/results"
NEOWISE_FILE = f"{RESULTS_DIR}/exp005_region_detections.csv"

MIN_OBS = 10
MIN_SNR = 5.0
VARIABLE_RCHI2 = 2.0
GOOD_QUALITY_FRACTION = 0.90

MATCH_RADIUS_ARCSEC = 1.5
MAX_CONTROLS = 30

# ------------------------------------------------------------
# 1. LOAD NEOWISE
# ------------------------------------------------------------

neowise = pd.read_csv(NEOWISE_FILE)
neowise.columns = [c.strip() for c in neowise.columns]

print(f"[OK] NEOWISE rows: {len(neowise)}")

required = ["ra", "dec", "w1mpro", "w1sigmpro"]
missing = [c for c in required if c not in neowise.columns]

if missing:
    raise ValueError(f"Missing NEOWISE columns: {missing}")

ra_min = float(neowise["ra"].min())
ra_max = float(neowise["ra"].max())
dec_min = float(neowise["dec"].min())
dec_max = float(neowise["dec"].max())

print("[INFO] NEOWISE field:")
print(f"       RA  = {ra_min:.6f} to {ra_max:.6f}")
print(f"       Dec = {dec_min:.6f} to {dec_max:.6f}")

# ------------------------------------------------------------
# 2. QUERY GAIA DR3 MAIN CATALOG
# ------------------------------------------------------------

tap_url = "https://vizier.cds.unistra.fr/viz-bin/asu-tsv"

gaia_response = requests.get(
    tap_url,
    params={
        "-source": "I/355/gaiadr3",
        "-out": "Source,RA_ICRS,DE_ICRS",
        "-out.max": "100000",
        "-c": f"{(ra_min + ra_max)/2:.6f} "
               f"{(dec_min + dec_max)/2:.6f}",
        "-c.rm": f"{max(ra_max-ra_min, dec_max-dec_min)*60:.3f}",
        "-out.form": "tsv",
    },
    timeout=120,
)

gaia_response.raise_for_status()

lines = gaia_response.text.splitlines()

header_idx = None

for i, line in enumerate(lines):
    if line.startswith("Source") and "RA_ICRS" in line:
        header_idx = i
        break

if header_idx is None:
    raise RuntimeError("Could not locate Gaia table header.")

gaia = pd.read_csv(
    io.StringIO("\n".join(lines[header_idx:])),
    sep="\t",
    comment="#"
)

print(f"[OK] Gaia rows retrieved: {len(gaia)}")

gaia["Source"] = pd.to_numeric(
    gaia["Source"], errors="coerce"
)

gaia["RA_ICRS"] = pd.to_numeric(
    gaia["RA_ICRS"], errors="coerce"
)

gaia["DE_ICRS"] = pd.to_numeric(
    gaia["DE_ICRS"], errors="coerce"
)

gaia = gaia.dropna(
    subset=["Source", "RA_ICRS", "DE_ICRS"]
).copy()

gaia["Source"] = gaia["Source"].astype(np.int64)

print(f"[OK] Gaia rows with coordinates: {len(gaia)}")

# ------------------------------------------------------------
# 3. QUERY GAIA VARIABILITY CLASSIFICATIONS
# ------------------------------------------------------------

print("[INFO] Querying Gaia variability classifications...")

class_response = requests.get(
    tap_url,
    params={
        "-source": "I/358/vclassre",
        "-out": "Source,Class,ClassSc,RA_ICRS,DE_ICRS",
        "-out.max": "100000",
        "-c": f"{(ra_min + ra_max)/2:.6f} "
               f"{(dec_min + dec_max)/2:.6f}",
        "-c.rm": f"{max(ra_max-ra_min, dec_max-dec_min)*60:.3f}",
        "-out.form": "tsv",
    },
    timeout=120,
)

class_response.raise_for_status()

class_lines = class_response.text.splitlines()

class_header_idx = None

for i, line in enumerate(class_lines):
    if line.startswith("Source") and "Class" in line:
        class_header_idx = i
        break

if class_header_idx is None:
    raise RuntimeError(
        "Could not locate Gaia classification header."
    )

gaia_classes = pd.read_csv(
    io.StringIO(
        "\n".join(class_lines[class_header_idx:])
    ),
    sep="\t",
    comment="#"
)

gaia_classes["Source"] = pd.to_numeric(
    gaia_classes["Source"],
    errors="coerce"
)

gaia_classes = gaia_classes.dropna(
    subset=["Source"]
)

gaia_classes["Source"] = (
    gaia_classes["Source"].astype(np.int64)
)

classified_ids = set(
    gaia_classes["Source"]
)

print(
    f"[OK] Gaia classified sources in field: "
    f"{len(classified_ids)}"
)

# ------------------------------------------------------------
# 4. REMOVE GAIA-CLASSIFIED SOURCES
# ------------------------------------------------------------

controls = gaia[
    ~gaia["Source"].isin(classified_ids)
].copy()

print(
    f"[OK] Gaia unclassified control candidates: "
    f"{len(controls)}"
)

# ------------------------------------------------------------
# 5. CROSSMATCH TO NEOWISE
# ------------------------------------------------------------

gaia_coord = SkyCoord(
    ra=controls["RA_ICRS"].to_numpy() * u.deg,
    dec=controls["DE_ICRS"].to_numpy() * u.deg
)

neowise_coord = SkyCoord(
    ra=neowise["ra"].to_numpy() * u.deg,
    dec=neowise["dec"].to_numpy() * u.deg
)

idx, sep2d, _ = gaia_coord.match_to_catalog_sky(
    neowise_coord
)

controls["neowise_match_index"] = idx
controls["separation_arcsec"] = sep2d.arcsec

controls = controls[
    controls["separation_arcsec"] <= MATCH_RADIUS_ARCSEC
].copy()

print(
    f"[OK] Gaia/NEOWISE matches within "
    f"{MATCH_RADIUS_ARCSEC:.1f} arcsec: "
    f"{len(controls)}"
)

# ------------------------------------------------------------
# 6. RUN VALIDATOR ON CONTROLS
# ------------------------------------------------------------

control_results = []

for _, control in controls.iterrows():

    ra0 = float(control["RA_ICRS"])
    dec0 = float(control["DE_ICRS"])

    c0 = SkyCoord(
        ra=ra0 * u.deg,
        dec=dec0 * u.deg
    )

    separation = c0.separation(
        neowise_coord
    ).arcsec

    obj = neowise[
        separation <= MATCH_RADIUS_ARCSEC
    ].copy()

    if len(obj) < MIN_OBS:
        continue

    w1 = pd.to_numeric(
        obj["w1mpro"],
        errors="coerce"
    )

    err = pd.to_numeric(
        obj["w1sigmpro"],
        errors="coerce"
    )

    mask = (
        np.isfinite(w1)
        & np.isfinite(err)
        & (err > 0)
    )

    w1 = w1[mask]
    err = err[mask]

    if len(w1) < MIN_OBS:
        continue

    snr = 1.0857 / err

    quality_fraction = float(
        np.mean(snr >= MIN_SNR)
    )

    weights = 1.0 / (err ** 2)

    w1_mean = float(
        np.sum(weights * w1) /
        np.sum(weights)
    )

    chi2 = float(
        np.sum(
            ((w1 - w1_mean) / err) ** 2
        )
    )

    dof = max(len(w1) - 1, 1)

    reduced_chi2 = chi2 / dof

    scatter = float(
        w1.std(ddof=1)
    )

    w1_range = float(
        w1.max() - w1.min()
    )

    median_snr = float(
        np.median(snr)
    )

    # SAME validator used in previous experiments
    if (
        len(w1) >= MIN_OBS
        and median_snr >= MIN_SNR
        and reduced_chi2 >= VARIABLE_RCHI2
        and quality_fraction >= GOOD_QUALITY_FRACTION
    ):
        validator_class = "VARIABLE_HIGH_QUALITY"

    elif (
        len(w1) >= MIN_OBS
        and reduced_chi2 >= VARIABLE_RCHI2
    ):
        validator_class = "VARIABLE_LOW_QUALITY"

    else:
        validator_class = "NOT_VARIABLE"

    control_results.append({
        "source": int(control["Source"]),
        "gaia_ra": ra0,
        "gaia_dec": dec0,
        "separation_arcsec": float(
            control["separation_arcsec"]
        ),
        "n_obs": int(len(w1)),
        "median_snr": median_snr,
        "w1_mean": w1_mean,
        "w1_std": scatter,
        "w1_range": w1_range,
        "reduced_chi2": reduced_chi2,
        "quality_fraction": quality_fraction,
        "validator_class": validator_class
    })

# ------------------------------------------------------------
# 7. BUILD RESULT TABLE
# ------------------------------------------------------------

controls_df = pd.DataFrame(
    control_results
)

if controls_df.empty:
    raise RuntimeError(
        "No evaluable Gaia-unclassified controls "
        "were found."
    )

controls_df = (
    controls_df
    .sort_values(
        ["separation_arcsec", "source"]
    )
    .head(MAX_CONTROLS)
)

n_controls = len(controls_df)

n_positive = int(
    controls_df["validator_class"].isin(
        [
            "VARIABLE_HIGH_QUALITY",
            "VARIABLE_LOW_QUALITY"
        ]
    ).sum()
)

n_negative = n_controls - n_positive

high_quality_positive = int(
    (
        controls_df["validator_class"]
        == "VARIABLE_HIGH_QUALITY"
    ).sum()
)

trigger_rate = (
    n_positive / n_controls
)

# ------------------------------------------------------------
# 8. PRINT RESULTS
# ------------------------------------------------------------

print()
print("=" * 60)
print("EXP-010 — INDEPENDENT CONTROL POPULATION")
print("=" * 60)

print()
print("CONTROL POPULATION")
print(f"Controls tested              : {n_controls}")
print(f"Validator positive           : {n_positive}")
print(f"Validator negative           : {n_negative}")
print(
    f"Control trigger rate         : "
    f"{trigger_rate:.3f}"
)

print()
print("HIGH-QUALITY VARIABLE GATE")
print(
    f"High-quality positives       : "
    f"{high_quality_positive}"
)

print()
print("VALIDATOR THRESHOLDS")
print(f"Minimum observations         : {MIN_OBS}")
print(f"Minimum median SNR           : {MIN_SNR}")
print(f"Variable reduced chi2        : {VARIABLE_RCHI2}")
print(
    f"Good-quality fraction        : "
    f"{GOOD_QUALITY_FRACTION}"
)

print()
print("OBJECT-LEVEL RESULTS")

print(
    controls_df[
        [
            "source",
            "separation_arcsec",
            "n_obs",
            "median_snr",
            "w1_std",
            "w1_range",
            "reduced_chi2",
            "quality_fraction",
            "validator_class"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 9. SAVE
# ------------------------------------------------------------

controls_file = (
    f"{RESULTS_DIR}/exp010_independent_controls.csv"
)

summary_file = (
    f"{RESULTS_DIR}/exp010_control_summary.json"
)

controls_df.to_csv(
    controls_file,
    index=False
)

summary = {
    "experiment": "EXP-010",
    "control_population": "GAIA_UNCLASSIFIED_CONTROL",
    "n_controls": n_controls,
    "validator_positive": n_positive,
    "validator_negative": n_negative,
    "control_trigger_rate": trigger_rate,
    "high_quality_positive": high_quality_positive,
    "match_radius_arcsec": MATCH_RADIUS_ARCSEC,
    "minimum_observations": MIN_OBS,
    "minimum_median_snr": MIN_SNR,
    "variable_reduced_chi2": VARIABLE_RCHI2,
    "good_quality_fraction": GOOD_QUALITY_FRACTION
}

with open(summary_file, "w") as f:
    json.dump(summary, f, indent=2)

print()
print("OUTPUT FILES")
print(f"[OK] {controls_file}")
print(f"[OK] {summary_file}")

print()
print("EXP-010 COMPLETE")
print(
    "IMPORTANT: Gaia-unclassified controls are NOT "
    "proven non-variable objects."
)

[OK] NEOWISE rows: 7275
[INFO] NEOWISE field:
       RA  = 179.900005 to 180.099973
       Dec = -0.099993 to 0.099988
[OK] Gaia rows retrieved: 475
[OK] Gaia rows with coordinates: 473
[INFO] Querying Gaia variability classifications...
[OK] Gaia classified sources in field: 7
[OK] Gaia unclassified control candidates: 465
[OK] Gaia/NEOWISE matches within 1.5 arcsec: 115

EXP-010 — INDEPENDENT CONTROL POPULATION

CONTROL POPULATION
Controls tested              : 30
Validator positive           : 7
Validator negative           : 23
Control trigger rate         : 0.233

HIGH-QUALITY VARIABLE GATE
High-quality positives       : 6

VALIDATOR THRESHOLDS
Minimum observations         : 10
Minimum median SNR           : 5.0
Variable reduced chi2        : 2.0
Good-quality fraction        : 0.9

OBJECT-LEVEL RESULTS
             source  separation_arcsec  n_obs  median_snr   w1_std  w1_range  reduced_chi2  quality_fraction       validator_class
3602879431314935808           0.002907     24   10

In [8]:
# ============================================================
# EXP-011 — CONTROL TRIGGER FORENSICS
# Corrected for the actual EXP-005 detection schema
# ============================================================

import os
import json
import numpy as np
import pandas as pd

BASE = "astronomy_exp005/results"
CONTROL_FILE = os.path.join(BASE, "exp010_independent_controls.csv")
DETECTION_FILE = os.path.join(BASE, "exp005_region_detections.csv")

controls = pd.read_csv(CONTROL_FILE)
detections = pd.read_csv(DETECTION_FILE)

print("EXP-011 — CONTROL TRIGGER FORENSICS")
print("=" * 60)
print(f"[OK] Controls loaded: {len(controls)}")
print(f"[OK] Detection rows loaded: {len(detections)}")

# ------------------------------------------------------------
# Required columns confirmed from EXP-011 output
# ------------------------------------------------------------

ra_col = "ra"
dec_col = "dec"
w1_col = "w1mpro"
w1err_col = "w1sigmpro"
w2_col = "w2mpro"
w2err_col = "w2sigmpro"
time_col = "mjd"
qual_col = "qual_frame"

required = [
    ra_col, dec_col, w1_col, w1err_col,
    w2_col, w2err_col, time_col, qual_col
]

missing = [c for c in required if c not in detections.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# Identify the validator-positive controls
# ------------------------------------------------------------

positive_classes = {
    "VARIABLE_HIGH_QUALITY",
    "VARIABLE_LOW_QUALITY"
}

positive_controls = controls[
    controls["validator_class"].isin(positive_classes)
].copy()

print(f"[OK] Validator-positive controls: {len(positive_controls)}")

if len(positive_controls) == 0:
    raise ValueError("No validator-positive controls found.")

# ------------------------------------------------------------
# Coordinate columns in EXP-010
# ------------------------------------------------------------

if "gaia_ra" not in positive_controls.columns or \
   "gaia_dec" not in positive_controls.columns:
    raise ValueError(
        "EXP-010 output does not contain gaia_ra/gaia_dec."
    )

# Convert NEOWISE coordinates once.
det_ra = pd.to_numeric(detections[ra_col], errors="coerce").to_numpy()
det_dec = pd.to_numeric(detections[dec_col], errors="coerce").to_numpy()

results = []

for idx, ctrl in positive_controls.iterrows():

    source = ctrl["source"] if "source" in ctrl else f"control_{idx}"

    gra = float(ctrl["gaia_ra"])
    gde = float(ctrl["gaia_dec"])

    # --------------------------------------------------------
    # Match all NEOWISE detections within 1.5 arcsec
    # --------------------------------------------------------

    dra = (det_ra - gra) * np.cos(np.deg2rad(gde))
    ddec = det_dec - gde

    separation_arcsec = np.sqrt(
        dra**2 + ddec**2
    ) * 3600.0

    mask = (
        np.isfinite(separation_arcsec)
        & (separation_arcsec <= 1.5)
    )

    obj = detections.loc[mask].copy()
    obj["_sep_arcsec"] = separation_arcsec[mask]

    # --------------------------------------------------------
    # W1 data
    # --------------------------------------------------------

    w1 = pd.to_numeric(obj[w1_col], errors="coerce")
    w1err = pd.to_numeric(obj[w1err_col], errors="coerce")

    valid_w1 = (
        np.isfinite(w1)
        & np.isfinite(w1err)
        & (w1err > 0)
    )

    w1 = w1[valid_w1]
    w1err = w1err[valid_w1]

    n_obs = len(w1)

    if n_obs < 2:
        results.append({
            "source": source,
            "gaia_ra": gra,
            "gaia_dec": gde,
            "n_obs": n_obs,
            "forensic_verdict": "INSUFFICIENT_DATA"
        })
        continue

    w1_mean = float(w1.mean())
    w1_std = float(w1.std(ddof=1))
    w1_range = float(w1.max() - w1.min())

    chi2 = float(
        np.sum(((w1 - w1_mean) / w1err) ** 2)
    )

    reduced_chi2 = chi2 / max(n_obs - 1, 1)

    z = np.abs((w1 - w1_mean) / w1err)
    max_abs_z = float(z.max())

    # --------------------------------------------------------
    # Time analysis
    # --------------------------------------------------------

    times = pd.to_numeric(
        obj.loc[w1.index, time_col],
        errors="coerce"
    )

    valid_t = np.isfinite(times)

    times = times[valid_t]

    if len(times) >= 2:

        times_sorted = np.sort(times.to_numpy())

        epoch_span_days = float(
            times_sorted[-1] - times_sorted[0]
        )

        gaps = np.diff(times_sorted)

        # NEOWISE roughly revisits the sky every ~6 months.
        # >30 days is therefore used only as a rough epoch separator.
        epoch_count = int(
            np.sum(gaps > 30.0) + 1
        )

        close_pair_count = int(
            np.sum(gaps < 60.0 / 86400.0)
        )

        largest_gap_days = float(
            gaps.max()
        ) if len(gaps) else np.nan

    else:
        epoch_span_days = np.nan
        epoch_count = np.nan
        close_pair_count = np.nan
        largest_gap_days = np.nan

    # --------------------------------------------------------
    # W2 analysis
    # --------------------------------------------------------

    w2 = pd.to_numeric(
        obj[w2_col],
        errors="coerce"
    )

    w2err = pd.to_numeric(
        obj[w2err_col],
        errors="coerce"
    )

    valid_w2 = (
        np.isfinite(w2)
        & np.isfinite(w2err)
        & (w2err > 0)
    )

    w2 = w2[valid_w2]
    w2err = w2err[valid_w2]

    if len(w2) >= 2:

        w2_mean = float(w2.mean())
        w2_std = float(w2.std(ddof=1))
        w2_range = float(w2.max() - w2.min())

        w2_chi2 = float(
            np.sum(((w2 - w2_mean) / w2err) ** 2)
        )

        w2_rchi2 = w2_chi2 / max(len(w2) - 1, 1)

    else:
        w2_std = np.nan
        w2_range = np.nan
        w2_rchi2 = np.nan

    # --------------------------------------------------------
    # Quality fraction
    # --------------------------------------------------------

    quality = pd.to_numeric(
        obj.loc[w1.index, qual_col],
        errors="coerce"
    )

    quality = quality[np.isfinite(quality)]

    if len(quality):
        quality_fraction = float(
            np.mean(quality > 0)
        )
    else:
        quality_fraction = np.nan

    # --------------------------------------------------------
    # Separation statistics
    # --------------------------------------------------------

    sep_values = pd.to_numeric(
        obj["_sep_arcsec"],
        errors="coerce"
    )

    median_sep = float(
        np.nanmedian(sep_values)
    )

    max_sep = float(
        np.nanmax(sep_values)
    )

    # --------------------------------------------------------
    # Inspect whether variability is dominated by one point
    # --------------------------------------------------------

    sorted_z = np.sort(z.to_numpy())[::-1]

    if len(sorted_z) >= 2:
        second_largest_z = float(sorted_z[1])
    else:
        second_largest_z = np.nan

    # --------------------------------------------------------
    # Forensic interpretation
    # --------------------------------------------------------

    reasons = []

    if max_abs_z >= 4:
        reasons.append("EXTREME_POINT")

    if reduced_chi2 >= 2:
        reasons.append("EXCESS_SCATTER")

    if np.isfinite(epoch_span_days):
        if epoch_span_days < 30:
            reasons.append("SHORT_BASELINE")
        elif epoch_span_days > 100:
            reasons.append("LONG_BASELINE")

    if np.isfinite(quality_fraction) and quality_fraction < 0.9:
        reasons.append("QUALITY_CONCERN")

    if close_pair_count >= 2:
        reasons.append("CLOSE_PAIR_ACTIVITY")

    # --------------------------------------------------------
    # Preliminary verdict
    # --------------------------------------------------------

    if (
        reduced_chi2 >= 2
        and quality_fraction < 0.9
    ):
        verdict = "LIKELY_SYSTEMATIC"

    elif (
        reduced_chi2 >= 2
        and quality_fraction >= 0.9
        and epoch_span_days > 100
        and second_largest_z >= 2
    ):
        verdict = "LIKELY_REAL_VARIABILITY"

    else:
        verdict = "AMBIGUOUS"

    results.append({
        "source": source,
        "gaia_ra": gra,
        "gaia_dec": gde,
        "n_obs": n_obs,
        "median_sep_arcsec": median_sep,
        "max_sep_arcsec": max_sep,
        "w1_mean": w1_mean,
        "w1_std": w1_std,
        "w1_range": w1_range,
        "reduced_chi2": reduced_chi2,
        "max_abs_z": max_abs_z,
        "second_largest_z": second_largest_z,
        "epoch_span_days": epoch_span_days,
        "epoch_count": epoch_count,
        "largest_gap_days": largest_gap_days,
        "close_pair_count": close_pair_count,
        "w2_std": w2_std,
        "w2_range": w2_range,
        "w2_reduced_chi2": w2_rchi2,
        "quality_fraction": quality_fraction,
        "diagnostic_reasons": ";".join(reasons),
        "forensic_verdict": verdict
    })

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

forensic = pd.DataFrame(results)

print("\n" + "=" * 60)
print("OBJECT-LEVEL FORENSIC RESULTS")
print("=" * 60)

display_cols = [
    "source",
    "n_obs",
    "w1_std",
    "w1_range",
    "reduced_chi2",
    "max_abs_z",
    "second_largest_z",
    "epoch_span_days",
    "epoch_count",
    "close_pair_count",
    "w2_reduced_chi2",
    "quality_fraction",
    "forensic_verdict"
]

print(forensic[display_cols].to_string(index=False))

print("\n" + "=" * 60)
print("FORENSIC VERDICT DISTRIBUTION")
print("=" * 60)

print(
    forensic["forensic_verdict"]
    .value_counts(dropna=False)
)

print("\n" + "=" * 60)
print("SCIENTIFIC INTERPRETATION")
print("=" * 60)

print(
    """
LIKELY_SYSTEMATIC:
The trigger is associated with quality problems or other
measurement-related warning signs.

LIKELY_REAL_VARIABILITY:
The excess scatter persists over a long baseline, the data
are predominantly good quality, and more than one observation
contributes substantially to the variability.

AMBIGUOUS:
The light curve alone cannot distinguish astrophysical
variability from measurement/systematic effects.

IMPORTANT:
These are forensic classifications, not confirmed
astrophysical classifications.
"""
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

output_csv = os.path.join(
    BASE,
    "exp011_control_trigger_forensics.csv"
)

output_json = os.path.join(
    BASE,
    "exp011_control_trigger_forensics_summary.json"
)

forensic.to_csv(output_csv, index=False)

summary = {
    "experiment": "EXP-011",
    "positive_controls_examined": int(len(positive_controls)),
    "analyzed": int(len(forensic)),
    "verdict_distribution": {
        str(k): int(v)
        for k, v in forensic["forensic_verdict"]
        .value_counts(dropna=False)
        .to_dict()
        .items()
    },
    "purpose": (
        "Forensic investigation of validator-positive "
        "Gaia-unclassified controls."
    )
}

with open(output_json, "w") as f:
    json.dump(summary, f, indent=2)

print("\n[OK] Saved:")
print(output_csv)
print(output_json)

print("\nEXP-011 COMPLETE")

EXP-011 — CONTROL TRIGGER FORENSICS
[OK] Controls loaded: 30
[OK] Detection rows loaded: 7275
[OK] Validator-positive controls: 7

OBJECT-LEVEL FORENSIC RESULTS
             source  n_obs   w1_std  w1_range  reduced_chi2  max_abs_z  second_largest_z  epoch_span_days  epoch_count  close_pair_count  w2_reduced_chi2  quality_fraction        forensic_verdict
3795032797804395008     25 0.085382     0.296      2.116296   2.884444          2.335000       160.419201            2                 3         1.206987               1.0 LIKELY_REAL_VARIABILITY
3602878572320135168     24 0.059269     0.310      2.525131   4.403080          3.358073       160.290207            2                 2         1.166835               1.0 LIKELY_REAL_VARIABILITY
3602878915918859776     30 0.142246     0.619      2.093741   3.776357          2.878878       160.419201            2                 8         1.092798               1.0 LIKELY_REAL_VARIABILITY
3795032591646256128     28 0.136923     0.666      3.94

In [9]:
# ============================================================
# EXP-012 — EPOCH-TO-EPOCH SYSTEMATICS TEST
# Determine whether EXP-010 triggers are dominated by
# differences between observing epochs.
# ============================================================

import os
import json
import numpy as np
import pandas as pd

BASE = "astronomy_exp005/results"

CONTROL_FILE = os.path.join(
    BASE,
    "exp010_independent_controls.csv"
)

DETECTION_FILE = os.path.join(
    BASE,
    "exp005_region_detections.csv"
)

controls = pd.read_csv(CONTROL_FILE)
detections = pd.read_csv(DETECTION_FILE)

# ------------------------------------------------------------
# Columns confirmed from EXP-011
# ------------------------------------------------------------

RA = "ra"
DEC = "dec"
W1 = "w1mpro"
W1ERR = "w1sigmpro"
W2 = "w2mpro"
W2ERR = "w2sigmpro"
TIME = "mjd"

positive_classes = {
    "VARIABLE_HIGH_QUALITY",
    "VARIABLE_LOW_QUALITY"
}

positive_controls = controls[
    controls["validator_class"].isin(positive_classes)
].copy()

print("EXP-012 — EPOCH-TO-EPOCH SYSTEMATICS TEST")
print("=" * 60)
print(f"[OK] Positive controls: {len(positive_controls)}")
print(f"[OK] NEOWISE detections: {len(detections)}")

det_ra = pd.to_numeric(
    detections[RA],
    errors="coerce"
).to_numpy()

det_dec = pd.to_numeric(
    detections[DEC],
    errors="coerce"
).to_numpy()

results = []

for _, ctrl in positive_controls.iterrows():

    source = ctrl["source"]
    gra = float(ctrl["gaia_ra"])
    gde = float(ctrl["gaia_dec"])

    # --------------------------------------------------------
    # Match NEOWISE detections within 1.5 arcsec
    # --------------------------------------------------------

    dra = (
        det_ra - gra
    ) * np.cos(np.deg2rad(gde))

    ddec = det_dec - gde

    separation = np.sqrt(
        dra**2 + ddec**2
    ) * 3600.0

    mask = (
        np.isfinite(separation)
        & (separation <= 1.5)
    )

    obj = detections.loc[mask].copy()

    if len(obj) < 4:
        continue

    # --------------------------------------------------------
    # Convert data
    # --------------------------------------------------------

    obj["w1"] = pd.to_numeric(
        obj[W1],
        errors="coerce"
    )

    obj["w1err"] = pd.to_numeric(
        obj[W1ERR],
        errors="coerce"
    )

    obj["w2"] = pd.to_numeric(
        obj[W2],
        errors="coerce"
    )

    obj["w2err"] = pd.to_numeric(
        obj[W2ERR],
        errors="coerce"
    )

    obj["mjd_num"] = pd.to_numeric(
        obj[TIME],
        errors="coerce"
    )

    obj = obj[
        np.isfinite(obj["mjd_num"])
        & np.isfinite(obj["w1"])
        & np.isfinite(obj["w1err"])
        & (obj["w1err"] > 0)
    ].copy()

    if len(obj) < 4:
        continue

    # --------------------------------------------------------
    # Define epochs
    #
    # Large gaps separate observing visits.
    # For these objects EXP-011 showed ~160-day span.
    # --------------------------------------------------------

    obj = obj.sort_values("mjd_num").reset_index(drop=True)

    gaps = np.diff(obj["mjd_num"].to_numpy())

    epoch_id = np.zeros(len(obj), dtype=int)

    current_epoch = 0

    for i, gap in enumerate(gaps, start=1):

        if gap > 30:
            current_epoch += 1

        epoch_id[i] = current_epoch

    obj["epoch"] = epoch_id

    epoch_counts = obj["epoch"].value_counts().sort_index()

    if len(epoch_counts) < 2:
        continue

    # Keep the two main epochs.
    main_epochs = (
        epoch_counts
        .sort_values(ascending=False)
        .head(2)
        .index
        .tolist()
    )

    main_epochs = sorted(main_epochs)

    if len(main_epochs) != 2:
        continue

    e1, e2 = main_epochs

    a = obj[obj["epoch"] == e1].copy()
    b = obj[obj["epoch"] == e2].copy()

    # --------------------------------------------------------
    # W1 epoch statistics
    # --------------------------------------------------------

    w1_e1 = float(a["w1"].mean())
    w1_e2 = float(b["w1"].mean())

    w1_e1_std = float(
        a["w1"].std(ddof=1)
    ) if len(a) > 1 else np.nan

    w1_e2_std = float(
        b["w1"].std(ddof=1)
    ) if len(b) > 1 else np.nan

    w1_epoch_delta = w1_e2 - w1_e1

    # --------------------------------------------------------
    # W1 within-epoch reduced chi2
    # --------------------------------------------------------

    def reduced_chi2(values, errors):

        values = np.asarray(values, dtype=float)
        errors = np.asarray(errors, dtype=float)

        good = (
            np.isfinite(values)
            & np.isfinite(errors)
            & (errors > 0)
        )

        values = values[good]
        errors = errors[good]

        if len(values) < 2:
            return np.nan

        mean = np.mean(values)

        chi2 = np.sum(
            ((values - mean) / errors) ** 2
        )

        return float(
            chi2 / max(len(values) - 1, 1)
        )

    w1_rchi2_e1 = reduced_chi2(
        a["w1"],
        a["w1err"]
    )

    w1_rchi2_e2 = reduced_chi2(
        b["w1"],
        b["w1err"]
    )

    # --------------------------------------------------------
    # W2 epoch statistics
    # --------------------------------------------------------

    a_w2 = a[
        np.isfinite(a["w2"])
        & np.isfinite(a["w2err"])
        & (a["w2err"] > 0)
    ]

    b_w2 = b[
        np.isfinite(b["w2"])
        & np.isfinite(b["w2err"])
        & (b["w2err"] > 0)
    ]

    if len(a_w2) >= 2 and len(b_w2) >= 2:

        w2_e1 = float(a_w2["w2"].mean())
        w2_e2 = float(b_w2["w2"].mean())

        w2_epoch_delta = w2_e2 - w2_e1

        w2_rchi2_e1 = reduced_chi2(
            a_w2["w2"],
            a_w2["w2err"]
        )

        w2_rchi2_e2 = reduced_chi2(
            b_w2["w2"],
            b_w2["w2err"]
        )

    else:

        w2_e1 = np.nan
        w2_e2 = np.nan
        w2_epoch_delta = np.nan
        w2_rchi2_e1 = np.nan
        w2_rchi2_e2 = np.nan

    # --------------------------------------------------------
    # W1/W2 direction agreement
    #
    # Magnitudes: positive delta = became fainter.
    # --------------------------------------------------------

    if np.isfinite(w2_epoch_delta):

        same_direction = (
            np.sign(w1_epoch_delta)
            == np.sign(w2_epoch_delta)
        )

    else:
        same_direction = np.nan

    # --------------------------------------------------------
    # Within-epoch vs between-epoch variability
    # --------------------------------------------------------

    within_values = []

    if np.isfinite(w1_rchi2_e1):
        within_values.append(w1_rchi2_e1)

    if np.isfinite(w1_rchi2_e2):
        within_values.append(w1_rchi2_e2)

    if within_values:
        median_within_rchi2 = float(
            np.median(within_values)
        )
    else:
        median_within_rchi2 = np.nan

    # Overall validator chi2 from EXP-011.
    overall_rchi2 = float(
        ctrl.get("reduced_chi2", np.nan)
    )

    # --------------------------------------------------------
    # Preliminary interpretation
    # --------------------------------------------------------

    if (
        np.isfinite(overall_rchi2)
        and overall_rchi2 >= 2
        and np.isfinite(median_within_rchi2)
        and median_within_rchi2 < 1.5
    ):
        epoch_dominated = True
    else:
        epoch_dominated = False

    if (
        epoch_dominated
        and np.isfinite(w2_epoch_delta)
        and not same_direction
    ):
        diagnostic = "W1_EPOCH_SYSTEMATIC_SUSPECTED"

    elif (
        epoch_dominated
        and (
            not np.isfinite(w2_epoch_delta)
            or same_direction
        )
    ):
        diagnostic = "EPOCH_OFFSET_NEEDS_CROSS_SURVEY_CHECK"

    elif (
        np.isfinite(median_within_rchi2)
        and median_within_rchi2 >= 1.5
    ):
        diagnostic = "WITHIN_EPOCH_VARIABILITY_PRESENT"

    else:
        diagnostic = "AMBIGUOUS"

    results.append({
        "source": source,
        "n_epoch1": len(a),
        "n_epoch2": len(b),
        "w1_epoch1_mean": w1_e1,
        "w1_epoch2_mean": w1_e2,
        "w1_epoch_delta": w1_epoch_delta,
        "w1_epoch1_std": w1_e1_std,
        "w1_epoch2_std": w1_e2_std,
        "w1_rchi2_epoch1": w1_rchi2_e1,
        "w1_rchi2_epoch2": w1_rchi2_e2,
        "median_within_epoch_rchi2": median_within_rchi2,
        "overall_w1_rchi2": overall_rchi2,
        "w2_epoch1_mean": w2_e1,
        "w2_epoch2_mean": w2_e2,
        "w2_epoch_delta": w2_epoch_delta,
        "w2_rchi2_epoch1": w2_rchi2_e1,
        "w2_rchi2_epoch2": w2_rchi2_e2,
        "w1_w2_same_direction": same_direction,
        "epoch_dominated": epoch_dominated,
        "diagnostic": diagnostic
    })

results_df = pd.DataFrame(results)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EXP-012 RESULTS")
print("=" * 60)

cols = [
    "source",
    "n_epoch1",
    "n_epoch2",
    "w1_epoch_delta",
    "w1_rchi2_epoch1",
    "w1_rchi2_epoch2",
    "median_within_epoch_rchi2",
    "overall_w1_rchi2",
    "w2_epoch_delta",
    "w2_rchi2_epoch1",
    "w2_rchi2_epoch2",
    "w1_w2_same_direction",
    "epoch_dominated",
    "diagnostic"
]

print(
    results_df[cols].to_string(index=False)
)

print("\n" + "=" * 60)
print("DIAGNOSTIC DISTRIBUTION")
print("=" * 60)

print(
    results_df["diagnostic"]
    .value_counts(dropna=False)
)

print("\n" + "=" * 60)
print("SCIENTIFIC INTERPRETATION")
print("=" * 60)

print(
    """
The key comparison is:

OVERALL W1 REDUCED CHI2
        versus
WITHIN-EPOCH W1 REDUCED CHI2

If overall chi2 >= 2 but both individual epochs
have low chi2, the apparent variability is dominated
by an offset between epochs.

If both individual epochs also show excess scatter,
the source has stronger evidence for genuine
within-epoch variability.

The W2 epoch direction provides an additional
cross-band diagnostic, but agreement is not required
for genuine astrophysical variability.

These diagnostics are not final astrophysical labels.
"""
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

output_csv = os.path.join(
    BASE,
    "exp012_epoch_systematics.csv"
)

output_json = os.path.join(
    BASE,
    "exp012_epoch_systematics_summary.json"
)

results_df.to_csv(
    output_csv,
    index=False
)

summary = {
    "experiment": "EXP-012",
    "objects_tested": int(len(results_df)),
    "epoch_dominated": int(
        results_df["epoch_dominated"].sum()
    ),
    "diagnostic_distribution": {
        str(k): int(v)
        for k, v in results_df["diagnostic"]
        .value_counts(dropna=False)
        .to_dict()
        .items()
    }
}

with open(output_json, "w") as f:
    json.dump(summary, f, indent=2)

print("\n[OK] Saved:")
print(output_csv)
print(output_json)

print("\nEXP-012 COMPLETE")

EXP-012 — EPOCH-TO-EPOCH SYSTEMATICS TEST
[OK] Positive controls: 7
[OK] NEOWISE detections: 7275

EXP-012 RESULTS
             source  n_epoch1  n_epoch2  w1_epoch_delta  w1_rchi2_epoch1  w1_rchi2_epoch2  median_within_epoch_rchi2  overall_w1_rchi2  w2_epoch_delta  w2_rchi2_epoch1  w2_rchi2_epoch2  w1_w2_same_direction  epoch_dominated                       diagnostic
3795032797804395008        10        15       -0.007433         1.978631         2.359127                   2.168879          2.115925       -0.079300         2.117679         0.777630                  True            False WITHIN_EPOCH_VARIABILITY_PRESENT
3602878572320135168        11        13        0.020315         2.003070         3.132052                   2.567561          2.493238        0.072049         0.819160         1.252209                  True            False WITHIN_EPOCH_VARIABILITY_PRESENT
3602878915918859776        17        13       -0.036647         2.336822         1.804624                   2.0707

In [10]:
# EXP-013 — WITHIN-EPOCH PHOTOMETRIC CONSISTENCY
import json
import numpy as np
import pandas as pd

BASE = "astronomy_exp005/results"

controls = pd.read_csv(f"{BASE}/exp010_independent_controls.csv")
det = pd.read_csv(f"{BASE}/exp005_region_detections.csv")

print("EXP-013 — WITHIN-EPOCH PHOTOMETRIC CONSISTENCY")
print("=" * 60)

positive = controls[controls["validator_class"].astype(str).str.startswith("VARIABLE")].copy()

print(f"[OK] Positive controls: {len(positive)}")
print(f"[OK] NEOWISE detections: {len(det)}")

rows = []

for _, c in positive.iterrows():
    source = str(c["source"])
    ra = float(c["gaia_ra"])
    dec = float(c["gaia_dec"])

    x = det.copy()
    sep = np.sqrt(
        ((x["ra"] - ra) * np.cos(np.deg2rad(dec)))**2 +
        (x["dec"] - dec)**2
    ) * 3600.0

    x = x[sep <= 1.5].copy()

    if len(x) < 10:
        continue

    x["mjd"] = pd.to_numeric(x["mjd"], errors="coerce")
    x["w1mpro"] = pd.to_numeric(x["w1mpro"], errors="coerce")
    x["w1sigmpro"] = pd.to_numeric(x["w1sigmpro"], errors="coerce")
    x["w2mpro"] = pd.to_numeric(x["w2mpro"], errors="coerce")
    x["w2sigmpro"] = pd.to_numeric(x["w2sigmpro"], errors="coerce")

    x = x.dropna(subset=["mjd", "w1mpro", "w1sigmpro"])
    x = x[x["w1sigmpro"] > 0].sort_values("mjd").copy()

    if len(x) < 10:
        continue

    # Split observing visits using a 30-day gap.
    gaps = x["mjd"].diff().fillna(0)
    epoch = (gaps > 30).cumsum()
    x["epoch"] = epoch

    for ep, g in x.groupby("epoch"):
        if len(g) < 3:
            continue

        w1 = g["w1mpro"].to_numpy()
        e1 = g["w1sigmpro"].to_numpy()

        mean_w1 = np.average(w1, weights=1 / e1**2)
        residual = w1 - mean_w1
        z = residual / e1

        order = np.arange(len(g))

        # Correlation between measurement order and W1 residual.
        if len(g) >= 4 and np.std(order) > 0 and np.std(residual) > 0:
            order_corr = np.corrcoef(order, residual)[0, 1]
        else:
            order_corr = np.nan

        # Largest and second-largest standardized residual.
        abs_z = np.abs(z)
        max_z = float(np.max(abs_z))
        second_z = float(np.sort(abs_z)[-2]) if len(abs_z) >= 2 else np.nan

        # Fraction of points exceeding 2 sigma.
        frac_2sigma = float(np.mean(abs_z >= 2))

        # W2 consistency if available.
        w2_rchi2 = np.nan
        w2_mean = np.nan
        w2_std = np.nan

        g2 = g.dropna(subset=["w2mpro", "w2sigmpro"]).copy()
        g2 = g2[g2["w2sigmpro"] > 0]

        if len(g2) >= 3:
            w2 = g2["w2mpro"].to_numpy()
            e2 = g2["w2sigmpro"].to_numpy()
            mean_w2 = np.average(w2, weights=1 / e2**2)
            r2 = (w2 - mean_w2) / e2
            w2_rchi2 = float(np.sum(r2**2) / (len(w2) - 1))
            w2_mean = float(np.mean(w2))
            w2_std = float(np.std(w2, ddof=1))

        rows.append({
            "source": source,
            "epoch": int(ep),
            "n_obs": len(g),
            "w1_mean": float(np.mean(w1)),
            "w1_std": float(np.std(w1, ddof=1)),
            "w1_weighted_mean": float(mean_w1),
            "w1_max_abs_z": max_z,
            "w1_second_abs_z": second_z,
            "w1_frac_abs_z_ge_2": frac_2sigma,
            "w1_order_residual_corr": float(order_corr) if np.isfinite(order_corr) else np.nan,
            "w2_mean": w2_mean,
            "w2_std": w2_std,
            "w2_reduced_chi2": w2_rchi2
        })

result = pd.DataFrame(rows)

print("\n============================================================")
print("EXP-013 RESULTS")
print("============================================================")

if len(result):
    print(result.to_string(index=False))

    print("\n============================================================")
    print("SUMMARY")
    print("============================================================")

    print(f"Epochs analyzed                 : {len(result)}")
    print(f"Median W1 within-epoch scatter  : {result['w1_std'].median():.5f}")
    print(f"Median max |W1 z|               : {result['w1_max_abs_z'].median():.3f}")
    print(f"Median fraction |z| >= 2        : {result['w1_frac_abs_z_ge_2'].median():.3f}")

    corr = result["w1_order_residual_corr"].dropna()
    if len(corr):
        print(f"Median order/residual correlation: {corr.median():.3f}")

    print("\nDiagnostic guidance:")
    print("  - One extreme point dominating each epoch -> outlier/systematic suspect")
    print("  - Smooth residual trend with observation order -> time-dependent systematic suspect")
    print("  - Repeated deviations across many exposures -> stronger variability evidence")
    print("  - W1 variability without W2 support -> measurement/systematic remains plausible")
else:
    print("[WARN] No usable epoch-level results.")

out_csv = f"{BASE}/exp013_within_epoch_consistency.csv"
out_json = f"{BASE}/exp013_within_epoch_consistency_summary.json"

result.to_csv(out_csv, index=False)

summary = {
    "experiment": "EXP-013",
    "positive_controls": int(len(positive)),
    "epochs_analyzed": int(len(result)),
    "purpose": "Test whether W1 excess scatter persists across repeated exposures within individual observing epochs."
}

with open(out_json, "w") as f:
    json.dump(summary, f, indent=2)

print("\n[OK] Saved:")
print(out_csv)
print(out_json)

print("\nEXP-013 COMPLETE")

EXP-013 — WITHIN-EPOCH PHOTOMETRIC CONSISTENCY
[OK] Positive controls: 7
[OK] NEOWISE detections: 7275

EXP-013 RESULTS
             source  epoch  n_obs   w1_mean   w1_std  w1_weighted_mean  w1_max_abs_z  w1_second_abs_z  w1_frac_abs_z_ge_2  w1_order_residual_corr   w2_mean   w2_std  w2_reduced_chi2
3795032797804395008      0     10 13.967700 0.089771         13.963384      2.272447         2.135223            0.200000                0.214826 14.086900 0.261288         1.617361
3795032797804395008      1     15 13.960267 0.085399         13.961397      2.918565         2.367902            0.133333                0.347872 14.007600 0.176333         0.722661
3602878572320135168      0     11 13.005455 0.046595         13.004380      2.980640         2.577044            0.181818               -0.341018 12.899182 0.074342         0.818948
3602878572320135168      1     13 13.025769 0.068694         13.016235      4.407945         2.461420            0.307692                0.094384 12.971

In [11]:
# EXP-014 — CROSS-BAND RESIDUAL COHERENCE
import json
import numpy as np
import pandas as pd

BASE = "astronomy_exp005/results"

controls = pd.read_csv(f"{BASE}/exp010_independent_controls.csv")
det = pd.read_csv(f"{BASE}/exp005_region_detections.csv")

print("EXP-014 — CROSS-BAND RESIDUAL COHERENCE")
print("=" * 60)

positive = controls[
    controls["validator_class"].astype(str).str.startswith("VARIABLE")
].copy()

print(f"[OK] Positive controls: {len(positive)}")
print(f"[OK] NEOWISE detections: {len(det)}")

rows = []

for _, c in positive.iterrows():
    source = str(c["source"])
    ra = float(c["gaia_ra"])
    dec = float(c["gaia_dec"])

    x = det.copy()

    sep = np.sqrt(
        ((x["ra"] - ra) * np.cos(np.deg2rad(dec)))**2 +
        (x["dec"] - dec)**2
    ) * 3600.0

    x = x[sep <= 1.5].copy()

    for col in ["mjd", "w1mpro", "w1sigmpro", "w2mpro", "w2sigmpro"]:
        x[col] = pd.to_numeric(x[col], errors="coerce")

    x = x.dropna(
        subset=["mjd", "w1mpro", "w1sigmpro", "w2mpro", "w2sigmpro"]
    )

    x = x[
        (x["w1sigmpro"] > 0) &
        (x["w2sigmpro"] > 0)
    ].sort_values("mjd")

    if len(x) < 5:
        continue

    # Weighted mean in each band.
    w1 = x["w1mpro"].to_numpy()
    e1 = x["w1sigmpro"].to_numpy()
    w2 = x["w2mpro"].to_numpy()
    e2 = x["w2sigmpro"].to_numpy()

    mean1 = np.average(w1, weights=1 / e1**2)
    mean2 = np.average(w2, weights=1 / e2**2)

    # Residuals from each band's own mean.
    r1 = w1 - mean1
    r2 = w2 - mean2

    z1 = r1 / e1
    z2 = r2 / e2

    # Pearson correlation of observation-level residuals.
    if np.std(r1) > 0 and np.std(r2) > 0:
        corr = np.corrcoef(r1, r2)[0, 1]
    else:
        corr = np.nan

    # Same-direction fraction.
    same_sign = np.mean(np.sign(r1) == np.sign(r2))

    # Ignore points extremely close to zero for a stricter directional test.
    meaningful = (np.abs(z1) >= 1) & (np.abs(z2) >= 1)

    if np.sum(meaningful) >= 2:
        meaningful_same_sign = np.mean(
            np.sign(r1[meaningful]) == np.sign(r2[meaningful])
        )
    else:
        meaningful_same_sign = np.nan

    # How often both bands independently exceed 2 sigma.
    joint_2sigma = np.mean(
        (np.abs(z1) >= 2) & (np.abs(z2) >= 2)
    )

    # W1-only and W2-only significant fractions.
    w1_2sigma = np.mean(np.abs(z1) >= 2)
    w2_2sigma = np.mean(np.abs(z2) >= 2)

    rows.append({
        "source": source,
        "n_paired_obs": len(x),
        "w1_std": float(np.std(w1, ddof=1)),
        "w2_std": float(np.std(w2, ddof=1)),
        "residual_correlation": float(corr) if np.isfinite(corr) else np.nan,
        "same_direction_fraction": float(same_sign),
        "meaningful_same_direction_fraction": (
            float(meaningful_same_sign)
            if np.isfinite(meaningful_same_sign)
            else np.nan
        ),
        "w1_abs_z_ge_2_fraction": float(w1_2sigma),
        "w2_abs_z_ge_2_fraction": float(w2_2sigma),
        "joint_abs_z_ge_2_fraction": float(joint_2sigma)
    })

result = pd.DataFrame(rows)

print("\n============================================================")
print("EXP-014 RESULTS")
print("============================================================")

if len(result):
    print(result.to_string(index=False))

    print("\n============================================================")
    print("SUMMARY")
    print("============================================================")

    print(
        f"Median W1/W2 residual correlation : "
        f"{result['residual_correlation'].median():.3f}"
    )

    print(
        f"Median same-direction fraction    : "
        f"{result['same_direction_fraction'].median():.3f}"
    )

    print(
        f"Median joint |z|>=2 fraction      : "
        f"{result['joint_abs_z_ge_2_fraction'].median():.3f}"
    )

    print("\nInterpretation:")
    print("  Strong positive residual correlation -> stronger cross-band support.")
    print("  Near-zero/negative correlation -> W1 behavior may be band-specific.")
    print("  High joint 2-sigma fraction -> repeated multi-band support.")
    print("  High W1-only fraction with weak W2 -> systematic remains plausible.")
else:
    print("[WARN] No usable paired observations.")

out_csv = f"{BASE}/exp014_cross_band_coherence.csv"
out_json = f"{BASE}/exp014_cross_band_coherence_summary.json"

result.to_csv(out_csv, index=False)

summary = {
    "experiment": "EXP-014",
    "positive_controls": int(len(positive)),
    "objects_analyzed": int(len(result)),
    "purpose": "Measure observation-level coherence between W1 and W2 residual variability."
}

with open(out_json, "w") as f:
    json.dump(summary, f, indent=2)

print("\n[OK] Saved:")
print(out_csv)
print(out_json)

print("\nEXP-014 COMPLETE")

EXP-014 — CROSS-BAND RESIDUAL COHERENCE
[OK] Positive controls: 7
[OK] NEOWISE detections: 7275

EXP-014 RESULTS
             source  n_paired_obs   w1_std   w2_std  residual_correlation  same_direction_fraction  meaningful_same_direction_fraction  w1_abs_z_ge_2_fraction  w2_abs_z_ge_2_fraction  joint_abs_z_ge_2_fraction
3795032797804395008            25 0.085382 0.212866              0.248461                 0.680000                            0.333333                0.160000                0.040000                   0.000000
3602878572320135168            24 0.059269 0.087714              0.597470                 0.708333                            1.000000                0.208333                0.041667                   0.041667
3602878915918859776            28 0.133309 0.293813             -0.039056                 0.428571                                 NaN                0.107143                0.035714                   0.000000
3795032591646256128            28 0.136923 0.25

In [12]:
# EXP-015 — KNOWN-VARIABLE CROSS-BAND REFERENCE
import json
import numpy as np
import pandas as pd

BASE = "astronomy_exp005/results"

det = pd.read_csv(f"{BASE}/exp005_region_detections.csv")
gaia = pd.read_csv(f"{BASE}/exp009_external_classification_benchmark.csv")

print("EXP-015 — KNOWN-VARIABLE CROSS-BAND REFERENCE")
print("=" * 60)

print(f"[OK] NEOWISE detections: {len(det)}")
print(f"[OK] Gaia benchmark rows: {len(gaia)}")

# Keep Gaia-classified sources only.
gaia = gaia.dropna(subset=["gaia_ra", "gaia_dec"]).copy()

rows = []

for _, g in gaia.iterrows():
    source = str(g["source"])
    ra = float(g["gaia_ra"])
    dec = float(g["gaia_dec"])

    x = det.copy()

    sep = np.sqrt(
        ((x["ra"] - ra) * np.cos(np.deg2rad(dec)))**2 +
        (x["dec"] - dec)**2
    ) * 3600.0

    x = x[sep <= 1.5].copy()

    for col in ["mjd", "w1mpro", "w1sigmpro", "w2mpro", "w2sigmpro"]:
        x[col] = pd.to_numeric(x[col], errors="coerce")

    x = x.dropna(
        subset=["mjd", "w1mpro", "w1sigmpro", "w2mpro", "w2sigmpro"]
    )

    x = x[
        (x["w1sigmpro"] > 0) &
        (x["w2sigmpro"] > 0)
    ].sort_values("mjd")

    if len(x) < 5:
        continue

    w1 = x["w1mpro"].to_numpy()
    e1 = x["w1sigmpro"].to_numpy()
    w2 = x["w2mpro"].to_numpy()
    e2 = x["w2sigmpro"].to_numpy()

    mean1 = np.average(w1, weights=1 / e1**2)
    mean2 = np.average(w2, weights=1 / e2**2)

    r1 = w1 - mean1
    r2 = w2 - mean2

    z1 = r1 / e1
    z2 = r2 / e2

    corr = (
        np.corrcoef(r1, r2)[0, 1]
        if np.std(r1) > 0 and np.std(r2) > 0
        else np.nan
    )

    same_sign = np.mean(np.sign(r1) == np.sign(r2))

    joint_2sigma = np.mean(
        (np.abs(z1) >= 2) & (np.abs(z2) >= 2)
    )

    w1_2sigma = np.mean(np.abs(z1) >= 2)
    w2_2sigma = np.mean(np.abs(z2) >= 2)

    rows.append({
        "source": source,
        "gaia_class": str(g["gaia_class"]),
        "gaia_class_score": float(g["gaia_class_score"]),
        "n_paired_obs": len(x),
        "w1_std": float(np.std(w1, ddof=1)),
        "w2_std": float(np.std(w2, ddof=1)),
        "residual_correlation": float(corr),
        "same_direction_fraction": float(same_sign),
        "w1_abs_z_ge_2_fraction": float(w1_2sigma),
        "w2_abs_z_ge_2_fraction": float(w2_2sigma),
        "joint_abs_z_ge_2_fraction": float(joint_2sigma)
    })

result = pd.DataFrame(rows)

print("\n============================================================")
print("RESULTS")
print("============================================================")

if len(result):
    print(result.to_string(index=False))
else:
    print("[WARN] No usable Gaia reference sources.")

out_csv = f"{BASE}/exp015_known_variable_cross_band.csv"
out_json = f"{BASE}/exp015_known_variable_cross_band_summary.json"

result.to_csv(out_csv, index=False)

summary = {
    "experiment": "EXP-015",
    "objects_analyzed": int(len(result)),
    "purpose": "Establish cross-band behavior of independently Gaia-classified sources."
}

with open(out_json, "w") as f:
    json.dump(summary, f, indent=2)

print("\n[OK] Saved:")
print(out_csv)
print(out_json)

print("\nEXP-015 COMPLETE")

EXP-015 — KNOWN-VARIABLE CROSS-BAND REFERENCE
[OK] NEOWISE detections: 7275
[OK] Gaia benchmark rows: 3

RESULTS
             source      gaia_class  gaia_class_score  n_paired_obs   w1_std   w2_std  residual_correlation  same_direction_fraction  w1_abs_z_ge_2_fraction  w2_abs_z_ge_2_fraction  joint_abs_z_ge_2_fraction
3795032587349812480             AGN          0.709531            28 0.068298 0.094438             -0.006676                 0.464286                0.071429                0.035714                        0.0
3891110460301684864 DSCT|GDOR|SXPHE          0.141831            22 0.037003 0.098670             -0.031970                 0.454545                0.090909                0.136364                        0.0

[OK] Saved:
astronomy_exp005/results/exp015_known_variable_cross_band.csv
astronomy_exp005/results/exp015_known_variable_cross_band_summary.json

EXP-015 COMPLETE


In [13]:
# EXP-016 — RANDOMIZED INDEPENDENT CONTROL SAMPLE
import json
import numpy as np
import pandas as pd

BASE = "astronomy_exp005/results"

det = pd.read_csv(f"{BASE}/exp005_region_detections.csv")

print("EXP-016 — RANDOMIZED INDEPENDENT CONTROL SAMPLE")
print("=" * 60)

print(f"[OK] NEOWISE detections: {len(det)}")

# Reconstruct the Gaia source population used in EXP-010.
# Query result is stored by EXP-010 only as the final matched controls,
# so use the complete Gaia source table if it exists.
gaia_file = f"{BASE}/exp010_gaia_unclassified_candidates.csv"

try:
    gaia = pd.read_csv(gaia_file)
    print(f"[OK] Gaia control candidates loaded: {len(gaia)}")
except FileNotFoundError:
    print("[WARN] Full Gaia control candidate file not found.")
    print("Using EXP-010 control table as available population.")
    gaia = pd.read_csv(f"{BASE}/exp010_independent_controls.csv")

gaia = gaia.dropna(subset=["gaia_ra", "gaia_dec"]).copy()

# Deterministic random sampling.
sample_n = min(100, len(gaia))
gaia_sample = gaia.sample(
    n=sample_n,
    random_state=42
).reset_index(drop=True)

print(f"[OK] Randomized controls selected: {len(gaia_sample)}")

rows = []

for _, g in gaia_sample.iterrows():
    source = str(g["source"])
    ra = float(g["gaia_ra"])
    dec = float(g["gaia_dec"])

    x = det.copy()

    sep = np.sqrt(
        ((x["ra"] - ra) * np.cos(np.deg2rad(dec)))**2 +
        (x["dec"] - dec)**2
    ) * 3600.0

    x = x[sep <= 1.5].copy()

    for col in ["mjd", "w1mpro", "w1sigmpro", "w2mpro", "w2sigmpro"]:
        x[col] = pd.to_numeric(x[col], errors="coerce")

    x = x.dropna(subset=["mjd", "w1mpro", "w1sigmpro"])
    x = x[x["w1sigmpro"] > 0].sort_values("mjd")

    if len(x) < 10:
        continue

    w1 = x["w1mpro"].to_numpy()
    e1 = x["w1sigmpro"].to_numpy()

    mean1 = np.average(w1, weights=1 / e1**2)
    residual = w1 - mean1
    z = residual / e1

    rchi2 = np.sum(z**2) / (len(w1) - 1)

    q = np.mean(
        pd.to_numeric(x["qual_frame"], errors="coerce").fillna(0).to_numpy() > 0
    )

    median_snr = float(
        np.median(
            np.abs(
                1.0857 / np.maximum(e1, 1e-6)
            )
        )
    )

    if median_snr >= 5 and q >= 0.9 and rchi2 >= 2:
        cls = "VARIABLE_HIGH_QUALITY"
    elif rchi2 >= 2:
        cls = "VARIABLE_LOW_QUALITY"
    else:
        cls = "NOT_VARIABLE"

    rows.append({
        "source": source,
        "gaia_ra": ra,
        "gaia_dec": dec,
        "n_obs": len(x),
        "median_snr": median_snr,
        "w1_std": float(np.std(w1, ddof=1)),
        "w1_range": float(np.ptp(w1)),
        "reduced_chi2": float(rchi2),
        "quality_fraction": float(q),
        "validator_class": cls
    })

result = pd.DataFrame(rows)

print("\n============================================================")
print("RANDOMIZED CONTROL RESULTS")
print("============================================================")

if len(result):
    print(result.to_string(index=False))

    positive = result[
        result["validator_class"].astype(str).str.startswith("VARIABLE")
    ]

    high = result[
        result["validator_class"] == "VARIABLE_HIGH_QUALITY"
    ]

    print("\n============================================================")
    print("SUMMARY")
    print("============================================================")

    print(f"Controls analyzed          : {len(result)}")
    print(f"Validator positives        : {len(positive)}")
    print(f"High-quality positives     : {len(high)}")
    print(
        f"Overall trigger rate       : "
        f"{len(positive) / len(result):.3f}"
    )
    print(
        f"High-quality trigger rate  : "
        f"{len(high) / len(result):.3f}"
    )
    print(
        f"Median control reduced chi2: "
        f"{result['reduced_chi2'].median():.3f}"
    )
else:
    print("[WARN] No usable randomized controls.")

out_csv = f"{BASE}/exp016_randomized_controls.csv"
out_json = f"{BASE}/exp016_randomized_controls_summary.json"

result.to_csv(out_csv, index=False)

summary = {
    "experiment": "EXP-016",
    "random_seed": 42,
    "requested_controls": 100,
    "controls_selected": int(len(gaia_sample)),
    "controls_analyzed": int(len(result)),
    "validator_positives": int(
        result["validator_class"].astype(str).str.startswith("VARIABLE").sum()
    ) if len(result) else 0,
    "purpose": "Remove the nearest-separation sampling bias from EXP-010 and estimate validator trigger behavior on a reproducible random control sample."
}

with open(out_json, "w") as f:
    json.dump(summary, f, indent=2)

print("\n[OK] Saved:")
print(out_csv)
print(out_json)

print("\nEXP-016 COMPLETE")

EXP-016 — RANDOMIZED INDEPENDENT CONTROL SAMPLE
[OK] NEOWISE detections: 7275
[WARN] Full Gaia control candidate file not found.
Using EXP-010 control table as available population.
[OK] Randomized controls selected: 30

RANDOMIZED CONTROL RESULTS
             source    gaia_ra  gaia_dec  n_obs  median_snr   w1_std  w1_range  reduced_chi2  quality_fraction       validator_class
3795033484999465472 179.933707  0.012212     23   12.198876 0.128507     0.429      1.659471               1.0          NOT_VARIABLE
3795033240184894976 179.975607 -0.007116     14    4.583053 0.198131     0.589      0.728751               1.0          NOT_VARIABLE
3602879500034765824 179.962725 -0.034452     26   10.696811 0.144512     0.669      1.614192               1.0          NOT_VARIABLE
3795033244481286144 179.983896 -0.013487     24   14.102379 0.120494     0.624      2.278855               1.0 VARIABLE_HIGH_QUALITY
3602879122077290496 179.953193 -0.051031     27   36.190000 0.035134     0.158      1.0

In [14]:
!pip -q install astroquery

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.5 MB/s eta 0:00:00


In [15]:
import json,numpy as np,pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.vizier import Vizier

base="astronomy_exp005/results"
groups=pd.read_csv(f"{base}/exp005_all_source_groups.csv")
det=pd.read_csv(f"{base}/exp005_region_detections.csv")

print("1/5 Gaia field query...")
V=Vizier(columns=["Source","RA_ICRS","DE_ICRS"],row_limit=1000)
t=V.query_region(SkyCoord(ra=180*u.deg,dec=0*u.deg),width=0.2*u.deg,height=0.2*u.deg,catalog="I/355/gaiadr3")[0]
gaia=t.to_pandas().rename(columns={"Source":"source_id","RA_ICRS":"ra","DE_ICRS":"dec"})
gaia["source_id"]=gaia["source_id"].astype(str)
gaia["ra"]=pd.to_numeric(gaia["ra"],errors="coerce")
gaia["dec"]=pd.to_numeric(gaia["dec"],errors="coerce")
gaia=gaia.dropna(subset=["ra","dec"]).drop_duplicates("source_id").reset_index(drop=True)
print("Gaia sources:",len(gaia))

print("2/5 Gaia classification query...")
classified=set()
for sid in gaia["source_id"]:
    try:
        q=Vizier(columns=["Source","ClassSc"],row_limit=20)
        r=q.query_constraints(catalog="I/358/vclassre",Source=str(sid))
        if len(r):
            x=r[0].to_pandas()
            if len(x):
                classified.add(str(sid))
    except Exception:
        pass
gaia["classified"]=gaia["source_id"].isin(classified)
unclassified=gaia[~gaia["classified"]].copy().reset_index(drop=True)
print("Classified:",len(classified))
print("Unclassified:",len(unclassified))

print("3/5 NEOWISE crossmatch...")
gc=SkyCoord(groups["ra"].values*u.deg,groups["dec"].values*u.deg)
uc=SkyCoord(unclassified["ra"].values*u.deg,unclassified["dec"].values*u.deg)
i1,i2,sep,_=gc.search_around_sky(uc,1.5*u.arcsec)
matches=unclassified.iloc[i1].copy()
matches["group_id"]=groups.iloc[i2]["group_id"].values
matches["separation_arcsec"]=sep.arcsec
matches=matches.sort_values("separation_arcsec").drop_duplicates("source_id").reset_index(drop=True)
print("Matched unclassified:",len(matches))

print("4/5 Random sampling + validator...")
rng=np.random.default_rng(42)
n=min(100,len(matches))
sample=matches.iloc[rng.choice(len(matches),size=n,replace=False)].copy().reset_index(drop=True)

rows=[]
for _,c in sample.iterrows():
    g=groups[groups["group_id"]==c["group_id"]]
    d=det[det["group_id"]==c["group_id"]].copy()
    d=d[d["qual_frame"]>0].copy()
    e=pd.to_numeric(d["w1sigmpro"],errors="coerce")
    m=pd.to_numeric(d["w1mpro"],errors="coerce")
    ok=np.isfinite(e)&np.isfinite(m)&(e>0)
    m=m[ok].values
    e=e[ok].values
    if len(m)<10:
        status="NOT_VARIABLE"
    else:
        snr=1.0857/e
        med_snr=float(np.median(snr))
        w=1/e**2
        mean=float(np.sum(w*m)/np.sum(w))
        chi=float(np.sum(w*(m-mean)**2)/(len(m)-1))
        qfrac=float(g["good_quality_fraction"].iloc[0])
        if med_snr>=5 and chi>=2 and qfrac>=0.9:
            status="VARIABLE_HIGH_QUALITY"
        elif chi>=2:
            status="VARIABLE_LOW_QUALITY"
        else:
            status="NOT_VARIABLE"
    rows.append({**c.to_dict(),"validator":status})

controls=pd.DataFrame(rows)

print("5/5 Saving...")
gaia[["source_id","ra","dec","classified"]].to_csv(f"{base}/exp017_gaia_field_population.csv",index=False)
controls.to_csv(f"{base}/exp017_full_randomized_controls.csv",index=False)

summary={
    "gaia_field_sources":int(len(gaia)),
    "classified_sources":int(gaia["classified"].sum()),
    "unclassified_sources":int(len(unclassified)),
    "neowise_matched_unclassified":int(len(matches)),
    "sampled_controls":int(len(controls)),
    "validated_controls":int(len(controls)),
    "positive_controls":int((controls["validator"]!="NOT_VARIABLE").sum()),
    "high_quality_controls":int((controls["validator"]=="VARIABLE_HIGH_QUALITY").sum()),
    "trigger_rate":float((controls["validator"]!="NOT_VARIABLE").mean()) if len(controls) else None,
    "high_quality_rate":float((controls["validator"]=="VARIABLE_HIGH_QUALITY").mean()) if len(controls) else None,
    "random_seed":42,
    "match_radius_arcsec":1.5
}
with open(f"{base}/exp017_full_randomized_controls_summary.json","w") as f:
    json.dump(summary,f,indent=2)

print(json.dumps(summary,indent=2))
print("\nValidator counts:")
print(controls["validator"].value_counts())
print("\nRequired columns present:", "group_id" in controls.columns)

1/5 Gaia field query...
Gaia sources: 159
2/5 Gaia classification query...
Classified: 3
Unclassified: 156
3/5 NEOWISE crossmatch...
Matched unclassified: 84
4/5 Random sampling + validator...
5/5 Saving...
{
  "gaia_field_sources": 159,
  "classified_sources": 3,
  "unclassified_sources": 156,
  "neowise_matched_unclassified": 84,
  "sampled_controls": 84,
  "validated_controls": 84,
  "positive_controls": 17,
  "high_quality_controls": 7,
  "trigger_rate": 0.20238095238095238,
  "high_quality_rate": 0.08333333333333333,
  "random_seed": 42,
  "match_radius_arcsec": 1.5
}

Validator counts:
validator
NOT_VARIABLE             67
VARIABLE_LOW_QUALITY     10
VARIABLE_HIGH_QUALITY     7
Name: count, dtype: int64

Required columns present: True


In [16]:
import json,numpy as np,pandas as pd

base="astronomy_exp005/results"
controls=pd.read_csv(f"{base}/exp017_full_randomized_controls.csv")
det=pd.read_csv(f"{base}/exp005_region_detections.csv")
groups=pd.read_csv(f"{base}/exp005_all_source_groups.csv")

assert "group_id" in controls.columns
assert len(controls)==84
assert controls["group_id"].notna().all()

rows=[]
for _,c in controls.iterrows():
    d=det[det["group_id"]==c["group_id"]].copy()
    d=d[d["qual_frame"]>0].copy()
    e1=pd.to_numeric(d["w1sigmpro"],errors="coerce")
    e2=pd.to_numeric(d["w2sigmpro"],errors="coerce")
    m1=pd.to_numeric(d["w1mpro"],errors="coerce")
    m2=pd.to_numeric(d["w2mpro"],errors="coerce")
    ok=np.isfinite(e1)&np.isfinite(e2)&np.isfinite(m1)&np.isfinite(m2)&(e1>0)&(e2>0)
    e1,e2,m1,m2=e1[ok].values,e2[ok].values,m1[ok].values,m2[ok].values
    if len(m1)<3:
        continue
    w1=1/e1**2
    w2=1/e2**2
    mean1=np.sum(w1*m1)/np.sum(w1)
    mean2=np.sum(w2*m2)/np.sum(w2)
    z1=(m1-mean1)/e1
    z2=(m2-mean2)/e2
    rows.append({
        "source_id":c["source_id"],
        "group_id":c["group_id"],
        "validator":c["validator"],
        "n_obs":len(m1),
        "w1_std":float(np.std(m1,ddof=1)),
        "w2_std":float(np.std(m2,ddof=1)),
        "w1_frac_abs_z2":float(np.mean(np.abs(z1)>=2)),
        "w2_frac_abs_z2":float(np.mean(np.abs(z2)>=2)),
        "joint_abs_z2":float(np.mean((np.abs(z1)>=2)&(np.abs(z2)>=2))),
        "same_direction":float(np.mean(np.sign(z1)==np.sign(z2))),
        "residual_corr":float(np.corrcoef(z1,z2)[0,1]) if len(z1)>2 and np.std(z1)>0 and np.std(z2)>0 else np.nan
    })

out=pd.DataFrame(rows)
out.to_csv(f"{base}/exp018_control_cross_band_behavior.csv",index=False)

summary={
    "controls_input":int(len(controls)),
    "controls_reconstructed":int(len(out)),
    "controls_missing":int(len(controls)-len(out)),
    "validator_positive":int((controls["validator"]!="NOT_VARIABLE").sum()),
    "validator_high_quality":int((controls["validator"]=="VARIABLE_HIGH_QUALITY").sum()),
    "overall_medians":out[["w1_std","w2_std","w1_frac_abs_z2","w2_frac_abs_z2","joint_abs_z2","same_direction","residual_corr"]].median(numeric_only=True).to_dict()
}
with open(f"{base}/exp018_control_cross_band_summary.json","w") as f:
    json.dump(summary,f,indent=2)

print(json.dumps(summary,indent=2))
print("\nGROUP MEDIANS:")
print(out.groupby("validator")[["w1_std","w2_std","w1_frac_abs_z2","w2_frac_abs_z2","joint_abs_z2","same_direction","residual_corr"]].median(numeric_only=True))
print("\nCOUNTS:")
print(out["validator"].value_counts())

{
  "controls_input": 84,
  "controls_reconstructed": 76,
  "controls_missing": 8,
  "validator_positive": 17,
  "validator_high_quality": 7,
  "overall_medians": {
    "w1_std": 0.13800756240885625,
    "w2_std": 0.25296450794132364,
    "w1_frac_abs_z2": 0.09090909090909091,
    "w2_frac_abs_z2": 0.0,
    "joint_abs_z2": 0.0,
    "same_direction": 0.5,
    "residual_corr": -0.03337049312950732
  }
}

GROUP MEDIANS:
                         w1_std    w2_std  w1_frac_abs_z2  w2_frac_abs_z2  \
validator                                                                   
NOT_VARIABLE           0.138586  0.253518        0.071429        0.000000   
VARIABLE_HIGH_QUALITY  0.085382  0.129355        0.160000        0.043478   
VARIABLE_LOW_QUALITY   0.168084  0.270762        0.129167        0.000000   

                       joint_abs_z2  same_direction  residual_corr  
validator                                                           
NOT_VARIABLE               0.000000        0.500000    

In [17]:
import numpy as np,pandas as pd

base="astronomy_exp005/results"
controls=pd.read_csv(f"{base}/exp017_full_randomized_controls.csv")
det=pd.read_csv(f"{base}/exp005_region_detections.csv")

rows=[]
for _,c in controls.iterrows():
    d=det[det["group_id"]==c["group_id"]].copy()
    d=d[d["qual_frame"]>0].copy()
    w1=pd.to_numeric(d["w1mpro"],errors="coerce")
    e1=pd.to_numeric(d["w1sigmpro"],errors="coerce")
    w2=pd.to_numeric(d["w2mpro"],errors="coerce")
    e2=pd.to_numeric(d["w2sigmpro"],errors="coerce")
    valid_w1=np.isfinite(w1)&np.isfinite(e1)&(e1>0)
    valid_w2=np.isfinite(w2)&np.isfinite(e2)&(e2>0)
    valid_both=valid_w1&valid_w2
    rows.append({
        "source_id":c["source_id"],
        "group_id":c["group_id"],
        "validator":c["validator"],
        "total_quality_obs":len(d),
        "valid_w1":int(valid_w1.sum()),
        "valid_w2":int(valid_w2.sum()),
        "valid_both":int(valid_both.sum())
    })

audit=pd.DataFrame(rows)
missing=audit[audit["valid_both"]<3].copy()

print("TOTAL CONTROLS:",len(audit))
print("RECONSTRUCTABLE:",int((audit["valid_both"]>=3).sum()))
print("MISSING:",len(missing))
print("\nMISSING CONTROLS:")
print(missing.to_string(index=False))
print("\nVALID-BOTH DISTRIBUTION:")
print(audit["valid_both"].describe())

TOTAL CONTROLS: 84
RECONSTRUCTABLE: 76
MISSING: 8

MISSING CONTROLS:
          source_id  group_id            validator  total_quality_obs  valid_w1  valid_w2  valid_both
3698956539760415360      1167         NOT_VARIABLE                 11        11         2           2
3602878842903277824        90         NOT_VARIABLE                 13        13         1           1
3602875235131187200       257         NOT_VARIABLE                 15        15         1           1
3698955066588101888      1282         NOT_VARIABLE                 14        14         2           2
3698956990733333760      1090 VARIABLE_LOW_QUALITY                 20        20         1           1
3698956784575326592      1110 VARIABLE_LOW_QUALITY                 13        13         1           1
3795033411983592064      1995         NOT_VARIABLE                 24        24         0           0
3795034408416148480      1787         NOT_VARIABLE                 15        15         2           2

VALID-BOTH D

In [18]:
import pandas as pd,numpy as np
base="astronomy_exp005/results"
controls=pd.read_csv(f"{base}/exp017_full_randomized_controls.csv")
det=pd.read_csv(f"{base}/exp005_region_detections.csv")
rows=[]
for _,c in controls.iterrows():
    d=det[(det["group_id"]==c["group_id"])&(det["qual_frame"]>0)].copy()
    w1=pd.to_numeric(d["w1mpro"],errors="coerce")
    e1=pd.to_numeric(d["w1sigmpro"],errors="coerce")
    w2=pd.to_numeric(d["w2mpro"],errors="coerce")
    e2=pd.to_numeric(d["w2sigmpro"],errors="coerce")
    v1=np.isfinite(w1)&np.isfinite(e1)&(e1>0)
    v2=np.isfinite(w2)&np.isfinite(e2)&(e2>0)
    both=v1&v2
    rows.append({"source_id":c["source_id"],"group_id":c["group_id"],"validator":c["validator"],"quality_obs":len(d),"valid_w1":int(v1.sum()),"valid_w2":int(v2.sum()),"valid_both":int(both.sum()),"w2_coverage":float(v2.sum()/len(d)) if len(d) else np.nan})
audit=pd.DataFrame(rows)
print("TOTAL:",len(audit))
print("W2 >=1:",int((audit.valid_w2>=1).sum()))
print("W2 >=3:",int((audit.valid_w2>=3).sum()))
print("W1+W2 >=3:",int((audit.valid_both>=3).sum()))
print("\nBY VALIDATOR:")
print(audit.groupby("validator")[["quality_obs","valid_w2","valid_both","w2_coverage"]].agg(["count","median","min","max"]).round(3).to_string())
print("\nMISSING FROM EXP-018 (paired W1+W2 <3):")
print(audit[audit.valid_both<3].sort_values("valid_both").to_string(index=False))
print("\nW2 COVERAGE DISTRIBUTION:")
print(audit["w2_coverage"].describe().round(3).to_string())
print("\nW2 VALID COUNT DISTRIBUTION:")
print(audit["valid_w2"].describe().round(3).to_string())

TOTAL: 84
W2 >=1: 83
W2 >=3: 76
W1+W2 >=3: 76

BY VALIDATOR:
                      quality_obs                valid_w2                valid_both                w2_coverage                    
                            count median min max    count median min max      count median min max       count median    min   max
validator                                                                                                                         
NOT_VARIABLE                   67   22.0  11  32       67   15.0   0  31         67   15.0   0  31          67  0.625  0.000  1.00
VARIABLE_HIGH_QUALITY           7   24.0  22  28        7   24.0  22  28          7   24.0  22  28           7  1.000  0.958  1.00
VARIABLE_LOW_QUALITY           10   22.0  13  32       10   15.5   1  28         10   15.5   1  28          10  0.696  0.050  0.88

MISSING FROM EXP-018 (paired W1+W2 <3):
          source_id  group_id            validator  quality_obs  valid_w1  valid_w2  valid_both  w2_coverage
379

In [19]:
import pandas as pd,numpy as np,json
base="astronomy_exp005/results"
controls=pd.read_csv(f"{base}/exp017_full_randomized_controls.csv")
det=pd.read_csv(f"{base}/exp005_region_detections.csv")
rows=[]
for _,c in controls.iterrows():
    d=det[(det["group_id"]==c["group_id"])&(det["qual_frame"]>0)].copy()
    w=pd.to_numeric(d["w1mpro"],errors="coerce")
    e=pd.to_numeric(d["w1sigmpro"],errors="coerce")
    snr=1.0857/e
    v=np.isfinite(w)&np.isfinite(e)&(e>0)
    w=w[v].to_numpy(); e=e[v].to_numpy(); snr=snr[v].to_numpy()
    n=len(w)
    if n<2:
        continue
    wt=1/e**2
    mean=np.sum(w*wt)/np.sum(wt)
    chi2=np.sum(((w-mean)/e)**2)
    rchi2=chi2/(n-1)
    med_snr=float(np.median(snr))
    qfrac=float(len(d)/len(det[det["group_id"]==c["group_id"]])) if len(det[det["group_id"]==c["group_id"]]) else 0
    rows.append({"source_id":c["source_id"],"group_id":c["group_id"],"n_obs":n,"median_snr":med_snr,"rchi2":rchi2,"quality_fraction":qfrac})
m=pd.DataFrame(rows)
tests=[
("baseline",10,5,2,0.90),
("nobs_15",15,5,2,0.90),
("nobs_20",20,5,2,0.90),
("snr_7",10,7,2,0.90),
("snr_10",10,10,2,0.90),
("chi2_3",10,5,3,0.90),
("chi2_4",10,5,4,0.90),
("quality_0.95",10,5,2,0.95),
("quality_1.00",10,5,2,1.00)
]
out=[]
for name,nmin,smin,cmin,qmin in tests:
    x=m[(m.n_obs>=nmin)&(m.median_snr>=smin)&(m.rchi2>=cmin)&(m.quality_fraction>=qmin)]
    out.append({"test":name,"n_controls":len(m),"positive":len(x),"trigger_rate":len(x)/len(m),"high_quality":int(((x.rchi2>=cmin)&(x.quality_fraction>=qmin)&(x.median_snr>=smin)&(x.n_obs>=nmin)).sum())})
res=pd.DataFrame(out)
print(res.to_string(index=False))
res.to_csv(f"{base}/exp019a_threshold_sensitivity.csv",index=False)
json.dump({"experiment":"EXP-019A","n_controls":len(m),"tests":out},open(f"{base}/exp019a_threshold_sensitivity_summary.json","w"),indent=2)

        test  n_controls  positive  trigger_rate  high_quality
    baseline          84        13      0.154762            13
     nobs_15          84        13      0.154762            13
     nobs_20          84        13      0.154762            13
       snr_7          84        13      0.154762            13
      snr_10          84        10      0.119048            10
      chi2_3          84         2      0.023810             2
      chi2_4          84         0      0.000000             0
quality_0.95          84        13      0.154762            13
quality_1.00          84        13      0.154762            13


In [20]:
import pandas as pd,numpy as np
base="astronomy_exp005/results"
c=pd.read_csv(f"{base}/exp017_full_randomized_controls.csv")
d=pd.read_csv(f"{base}/exp005_region_detections.csv")
rows=[]
for _,x in c.iterrows():
    g=d[(d.group_id==x.group_id)&(d.qual_frame>0)].copy()
    w=pd.to_numeric(g.w1mpro,errors="coerce")
    e=pd.to_numeric(g.w1sigmpro,errors="coerce")
    v=np.isfinite(w)&np.isfinite(e)&(e>0)
    w=w[v].to_numpy();e=e[v].to_numpy()
    if len(w)<2: continue
    wt=1/e**2
    mean=np.sum(w*wt)/np.sum(wt)
    rchi2=np.sum(((w-mean)/e)**2)/(len(w)-1)
    snr=1.0857/e
    rows.append({"source_id":x.source_id,"group_id":x.group_id,"old_validator":x.validator,"n_obs_new":len(w),"snr_new":np.median(snr),"rchi2_new":rchi2,"quality_fraction_new":len(g)/len(d[d.group_id==x.group_id])})
m=pd.DataFrame(rows)
m["new_positive"]=(m.n_obs_new>=10)&(m.snr_new>=5)&(m.rchi2_new>=2)&(m.quality_fraction_new>=0.90)
m["changed"]=m.old_validator.ne(np.where(m.new_positive,"VARIABLE_HIGH_QUALITY","NOT_VARIABLE"))
print("CONTROLS:",len(m))
print("OLD POSITIVE:",int((m.old_validator=="VARIABLE_HIGH_QUALITY").sum()))
print("NEW POSITIVE:",int(m.new_positive.sum()))
print("\nCHANGED SOURCES:")
print(m[m.changed].to_string(index=False))
print("\nOLD vs NEW:")
print(pd.crosstab(m.old_validator,m.new_positive))
print("\nRCHI2 RANGE OF CHANGED:")
print(m[m.changed][["source_id","group_id","old_validator","rchi2_new","n_obs_new","snr_new","quality_fraction_new"]].to_string(index=False))

CONTROLS: 84
OLD POSITIVE: 7
NEW POSITIVE: 13

CHANGED SOURCES:
          source_id  group_id        old_validator  n_obs_new   snr_new  rchi2_new  quality_fraction_new  new_positive  changed
3795033450638418176      1778 VARIABLE_LOW_QUALITY         23  8.225000   2.211039                   1.0          True     True
3891112551949280512      2551 VARIABLE_LOW_QUALITY         18  4.710217   2.776213                   1.0         False     True
3891111349358877696      2623 VARIABLE_LOW_QUALITY         23  7.385714   2.034919                   1.0          True     True
3602875239427221888       236 VARIABLE_LOW_QUALITY         25 10.966667   2.278061                   1.0          True     True
3602878915918859776         0 VARIABLE_LOW_QUALITY         32 11.192784   2.221670                   1.0          True     True
3795033450639424384      1778 VARIABLE_LOW_QUALITY         23  8.225000   2.211039                   1.0          True     True
3891111521157570688      2627 VARIABLE_L

In [21]:
import pandas as pd
base="astronomy_exp005/results"
c=pd.read_csv(f"{base}/exp017_full_randomized_controls.csv")
dups=c[c.duplicated("group_id",False)].sort_values("group_id")
print("CONTROLS:",len(c))
print("UNIQUE GROUPS:",c.group_id.nunique())
print("DUPLICATE GROUP IDS:",dups.group_id.nunique())
print("\nDUPLICATES:")
print(dups[["source_id","group_id","validator","separation_arcsec"]].to_string(index=False))
print("\nGAIA SOURCES PER GROUP:")
print(c.groupby("group_id").source_id.nunique().value_counts().sort_index().to_string())

CONTROLS: 84
UNIQUE GROUPS: 82
DUPLICATE GROUP IDS: 2

DUPLICATES:
          source_id  group_id             validator  separation_arcsec
3602878572321477248       234 VARIABLE_HIGH_QUALITY           0.656449
3602878572320135296       234 VARIABLE_HIGH_QUALITY           0.057815
3795033450639424384      1778  VARIABLE_LOW_QUALITY           0.237093
3795033450638418176      1778  VARIABLE_LOW_QUALITY           0.719637

GAIA SOURCES PER GROUP:
source_id
1    80
2     2


In [22]:
import pandas as pd
import numpy as np
base="astronomy_exp005/results"
c=pd.read_csv(f"{base}/exp017_full_randomized_controls.csv")
g=pd.read_csv(f"{base}/exp005_all_source_groups.csv")
d=pd.read_csv(f"{base}/exp005_region_detections.csv")
dup=c[c.duplicated("group_id",False)].sort_values("group_id")
for gid,x in dup.groupby("group_id"):
    print("\nGROUP",gid)
    print(x[["source_id","ra","dec","group_id","validator","separation_arcsec"]].to_string(index=False))
    print("GAIA SOURCE SEPARATION:")
    a=x.iloc[0]
    b=x.iloc[1]
    dra=(b.ra-a.ra)*3600*np.cos(np.deg2rad((a.dec+b.dec)/2))
    ddec=(b.dec-a.dec)*3600
    print(np.sqrt(dra**2+ddec**2),"arcsec")
    print("NEOWISE GROUP:")
    print(g[g.group_id==gid][["group_id","ra","dec","n_observations","w1_weighted_mean","w1_std","w1_range","w1_reduced_chi2","median_w1_snr","good_quality_fraction"]].to_string(index=False))
print("\nTOTAL CONTROLS:",len(c))
print("UNIQUE NEOWISE GROUPS:",c.group_id.nunique())
print("DUPLICATE GROUPS:",c[c.duplicated("group_id",False)].group_id.nunique())
print("\nVALIDATOR BY UNIQUE GROUP:")
u=c.sort_values("separation_arcsec").drop_duplicates("group_id")
print(u.validator.value_counts().to_string())


GROUP 234
          source_id         ra       dec  group_id             validator  separation_arcsec
3602878572321477248 180.040150 -0.045516       234 VARIABLE_HIGH_QUALITY           0.656449
3602878572320135296 180.040133 -0.045332       234 VARIABLE_HIGH_QUALITY           0.057815
GAIA SOURCE SEPARATION:
0.6637267040799542 arcsec
NEOWISE GROUP:
 group_id         ra       dec  n_observations  w1_weighted_mean   w1_std  w1_range  w1_reduced_chi2  median_w1_snr  good_quality_fraction
      234 180.040117 -0.045336              24         13.010557 0.059269      0.31         2.493238          33.35                    1.0

GROUP 1778
          source_id         ra      dec  group_id            validator  separation_arcsec
3795033450639424384 179.921100 0.002220      1778 VARIABLE_LOW_QUALITY           0.237093
3795033450638418176 179.921355 0.002174      1778 VARIABLE_LOW_QUALITY           0.719637
GAIA SOURCE SEPARATION:
0.9304864512174505 arcsec
NEOWISE GROUP:
 group_id         ra   

In [23]:
import pandas as pd
base="astronomy_exp005/results"
c=pd.read_csv(f"{base}/exp017_full_randomized_controls.csv")
print("ENTRY LEVEL")
print("N:",len(c))
print("POSITIVE:",c.validator.ne("NOT_VARIABLE").sum())
print("HIGH QUALITY:",(c.validator=="VARIABLE_HIGH_QUALITY").sum())
print("\nUNIQUE NEOWISE GROUP LEVEL")
u=c.sort_values("separation_arcsec").drop_duplicates("group_id")
print("N:",len(u))
print("POSITIVE:",u.validator.ne("NOT_VARIABLE").sum())
print("HIGH QUALITY:",(u.validator=="VARIABLE_HIGH_QUALITY").sum())
print("\nVALIDATOR COUNTS")
print(u.validator.value_counts().to_string())
print("\nDUPLICATE GROUPS")
print(c[c.duplicated("group_id",False)][["source_id","group_id","validator","separation_arcsec"]].sort_values("group_id").to_string(index=False))

ENTRY LEVEL
N: 84
POSITIVE: 17
HIGH QUALITY: 7

UNIQUE NEOWISE GROUP LEVEL
N: 82
POSITIVE: 15
HIGH QUALITY: 6

VALIDATOR COUNTS
validator
NOT_VARIABLE             67
VARIABLE_LOW_QUALITY      9
VARIABLE_HIGH_QUALITY     6

DUPLICATE GROUPS
          source_id  group_id             validator  separation_arcsec
3602878572321477248       234 VARIABLE_HIGH_QUALITY           0.656449
3602878572320135296       234 VARIABLE_HIGH_QUALITY           0.057815
3795033450639424384      1778  VARIABLE_LOW_QUALITY           0.237093
3795033450638418176      1778  VARIABLE_LOW_QUALITY           0.719637


In [24]:
import pandas as pd
import numpy as np
base="astronomy_exp005/results"
c=pd.read_csv(f"{base}/exp017_full_randomized_controls.csv")
d=pd.read_csv(f"{base}/exp005_region_detections.csv")
c=c.sort_values("separation_arcsec").drop_duplicates("group_id").reset_index(drop=True)
rows=[]
for _,x in c.iterrows():
    z=d[d.group_id==x.group_id].copy()
    z=z[z.qual_frame>0].copy()
    z=z.dropna(subset=["w1mpro","w1sigmpro"])
    n=len(z)
    if n<2:
        continue
    e=z.w1sigmpro.to_numpy()
    m=z.w1mpro.to_numpy()
    w=1/e**2
    mean=np.sum(w*m)/np.sum(w)
    rchi2=np.sum(w*(m-mean)**2)/(n-1)
    snr=np.median(1.0857/e)
    qfrac=len(z)/len(d[d.group_id==x.group_id])
    rows.append([x.source_id,x.group_id,n,snr,rchi2,qfrac])
v=pd.DataFrame(rows,columns=["source_id","group_id","n_obs","median_snr","rchi2","quality_fraction"])
tests={
"baseline":(10,5,2,.90),
"nobs_15":(15,5,2,.90),
"nobs_20":(20,5,2,.90),
"snr_7":(10,7,2,.90),
"snr_10":(10,10,2,.90),
"chi2_3":(10,5,3,.90),
"chi2_4":(10,5,4,.90),
"quality_.95":(10,5,2,.95),
"quality_1.0":(10,5,2,1.0)
}
out=[]
for name,(nmin,smin,cmin,qmin) in tests.items():
    p=(v.n_obs>=nmin)&(v.median_snr>=smin)&(v.rchi2>=cmin)&(v.quality_fraction>=qmin)
    out.append([name,len(v),p.sum(),p.mean(),((p)&(v.quality_fraction>=.90)).sum()])
r=pd.DataFrame(out,columns=["test","n_groups","positive","trigger_rate","high_quality"])
print(r.to_string(index=False))
print("\nUNIQUE GROUPS:",len(v))
print("BASELINE POSITIVE:",((v.n_obs>=10)&(v.median_snr>=5)&(v.rchi2>=2)&(v.quality_fraction>=.90)).sum())

       test  n_groups  positive  trigger_rate  high_quality
   baseline        82        11      0.134146            11
    nobs_15        82        11      0.134146            11
    nobs_20        82        11      0.134146            11
      snr_7        82        11      0.134146            11
     snr_10        82         9      0.109756             9
     chi2_3        82         2      0.024390             2
     chi2_4        82         0      0.000000             0
quality_.95        82        11      0.134146            11
quality_1.0        82        11      0.134146            11

UNIQUE GROUPS: 82
BASELINE POSITIVE: 11


In [25]:
import pandas as pd
import numpy as np
base="astronomy_exp005/results"
c=pd.read_csv(f"{base}/exp017_full_randomized_controls.csv")
d=pd.read_csv(f"{base}/exp005_region_detections.csv")
c=c.sort_values("separation_arcsec").drop_duplicates("group_id").reset_index(drop=True)
rows=[]
for _,x in c.iterrows():
    z=d[d.group_id==x.group_id].copy()
    n_all=len(z)
    z=z[z.qual_frame>0].copy()
    z=z.dropna(subset=["w1mpro","w1sigmpro"])
    n=len(z)
    if n<2:
        continue
    e=z.w1sigmpro.to_numpy()
    m=z.w1mpro.to_numpy()
    w=1/e**2
    mean=np.sum(w*m)/np.sum(w)
    rchi2=np.sum(w*(m-mean)**2)/(n-1)
    snr=np.median(1.0857/e)
    qfrac=len(z)/n_all if n_all else np.nan
    exact=(n>=10)&(snr>=5)&(rchi2>=2)&(qfrac>=.90)
    rows.append([x.source_id,x.group_id,x.validator,n,snr,rchi2,qfrac,exact])
v=pd.DataFrame(rows,columns=["source_id","group_id","old_validator","n_obs","median_snr","rchi2","quality_fraction","exact_positive"])
v["old_positive"]=v.old_validator.ne("NOT_VARIABLE")
v["old_high_quality"]=v.old_validator.eq("VARIABLE_HIGH_QUALITY")
v["agreement"]=v.old_positive==v.exact_positive
v["reason"]=np.select([
    v.exact_positive&~v.old_positive,
    ~v.exact_positive&v.old_positive,
    v.old_validator.eq("VARIABLE_LOW_QUALITY")&v.exact_positive,
    v.old_validator.eq("VARIABLE_LOW_QUALITY")&~v.exact_positive],
    ["OLD_MISSED_THRESHOLD","OLD_TRIGGERED_BUT_NUMERIC_FAILS","LOW_LABEL_PASSES_NUMERIC","LOW_LABEL_FAILS_NUMERIC"],
    default="AGREES")
print("UNIQUE GROUPS:",len(v))
print("\nOLD LABELS")
print(v.old_validator.value_counts().to_string())
print("\nEXACT NUMERIC THRESHOLD")
print(v.exact_positive.value_counts().rename({False:"NOT_VARIABLE",True:"POSITIVE"}).to_string())
print("\nOLD POSITIVE:",v.old_positive.sum())
print("EXACT POSITIVE:",v.exact_positive.sum())
print("AGREEMENT:",v.agreement.sum(),"/",len(v))
print("DISAGREEMENT:",(~v.agreement).sum())
print("\nDISAGREEMENTS")
print(v[~v.agreement][["source_id","group_id","old_validator","n_obs","median_snr","rchi2","quality_fraction","exact_positive","reason"]].to_string(index=False))
print("\nLOW-QUALITY AUDIT")
print(v[v.old_validator=="VARIABLE_LOW_QUALITY"][["source_id","group_id","n_obs","median_snr","rchi2","quality_fraction","exact_positive","reason"]].to_string(index=False))
print("\nTHRESHOLD FAILURES AMONG OLD POSITIVES")
p=v[v.old_positive]
print("n_obs < 10:",(p.n_obs<10).sum())
print("SNR < 5:",(p.median_snr<5).sum())
print("rchi2 < 2:",(p.rchi2<2).sum())
print("quality < .90:",(p.quality_fraction<.90).sum())
print("\nFINAL AUTHORITATIVE NUMERIC RESULT")
print("N:",len(v))
print("POSITIVE:",v.exact_positive.sum())
print("RATE:",v.exact_positive.mean())
print("HIGH-QUALITY:",((v.exact_positive)&(v.quality_fraction>=.90)).sum())
print("\nCROSS-TAB")
print(pd.crosstab(v.old_validator,v.exact_positive).to_string())
v.to_csv(f"{base}/exp019_validator_reproducibility_audit.csv",index=False)

UNIQUE GROUPS: 82

OLD LABELS
old_validator
NOT_VARIABLE             67
VARIABLE_LOW_QUALITY      9
VARIABLE_HIGH_QUALITY     6

EXACT NUMERIC THRESHOLD
exact_positive
NOT_VARIABLE    71
POSITIVE        11

OLD POSITIVE: 15
EXACT POSITIVE: 11
AGREEMENT: 78 / 82
DISAGREEMENT: 4

DISAGREEMENTS
          source_id  group_id        old_validator  n_obs  median_snr    rchi2  quality_fraction  exact_positive                          reason
3698956990733333760      1090 VARIABLE_LOW_QUALITY     20    4.836104 2.273081               1.0           False OLD_TRIGGERED_BUT_NUMERIC_FAILS
3891112551949280512      2551 VARIABLE_LOW_QUALITY     18    4.710217 2.776213               1.0           False OLD_TRIGGERED_BUT_NUMERIC_FAILS
3698956784575326592      1110 VARIABLE_LOW_QUALITY     13    4.413415 2.276742               1.0           False OLD_TRIGGERED_BUT_NUMERIC_FAILS
3891111521157570688      2627 VARIABLE_LOW_QUALITY     14    4.181897 2.124471               1.0           False OLD_TRIGGERED_

In [26]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

BASE=Path("astronomy_exp005/results")
controls=pd.read_csv(BASE/"exp017_full_randomized_controls.csv")
groups=pd.read_csv(BASE/"exp005_all_source_groups.csv")
det=pd.read_csv(BASE/"exp005_region_detections.csv")

print("INPUTS")
print("controls:",len(controls))
print("groups:",len(groups))
print("detections:",len(det))
print()

required_controls={"group_id","separation_arcsec"}
required_groups={"group_id","good_quality_fraction","flagged_count","n_observations","median_w1_snr","w1_reduced_chi2"}
required_det={"group_id","good_quality","quality_flagged","qual_frame","w1mpro","w1sigmpro"}

assert required_controls.issubset(controls.columns)
assert required_groups.issubset(groups.columns)
assert required_det.issubset(det.columns)

controls["group_id"]=controls["group_id"].astype(int)
groups["group_id"]=groups["group_id"].astype(int)
det["group_id"]=det["group_id"].astype(int)

controls_unique=(controls.sort_values("separation_arcsec").drop_duplicates("group_id").copy())

print("UNIQUE CONTROL GROUPS:",len(controls_unique))
print()

audit_rows=[]

for gid in controls_unique.group_id:
    x=det[det.group_id==gid].copy()
    g=groups[groups.group_id==gid]
    if len(g)!=1:
        continue
    g=g.iloc[0]
    n=len(x)
    good_mean=pd.to_numeric(x["good_quality"],errors="coerce").mean()
    flagged_mean=pd.to_numeric(x["quality_flagged"],errors="coerce").mean()
    not_flagged=1.0-flagged_mean
    qual_frame_positive=(pd.to_numeric(x["qual_frame"],errors="coerce")>0).mean()
    stored=float(g["good_quality_fraction"])
    audit_rows.append({
        "group_id":gid,
        "n_detections":n,
        "stored_good_quality_fraction":stored,
        "mean_good_quality":good_mean,
        "fraction_not_quality_flagged":not_flagged,
        "fraction_qual_frame_positive":qual_frame_positive,
        "stored_flagged_count":g["flagged_count"],
        "computed_flagged_count":int(pd.to_numeric(x["quality_flagged"],errors="coerce").fillna(False).sum()),
        "computed_bad_good_quality":int((pd.to_numeric(x["good_quality"],errors="coerce").fillna(0)==0).sum())
    })

qa=pd.DataFrame(audit_rows)

qa["diff_good_quality"]=qa["stored_good_quality_fraction"]-qa["mean_good_quality"]
qa["diff_not_flagged"]=qa["stored_good_quality_fraction"]-qa["fraction_not_quality_flagged"]
qa["diff_qual_frame"]=qa["stored_good_quality_fraction"]-qa["fraction_qual_frame_positive"]

print("QUALITY FORMULA REPRODUCIBILITY")
print("median abs diff: mean(good_quality) =",qa.diff_good_quality.abs().median())
print("max abs diff:    mean(good_quality) =",qa.diff_good_quality.abs().max())
print("median abs diff: 1-mean(quality_flagged) =",qa.diff_not_flagged.abs().median())
print("max abs diff:    1-mean(quality_flagged) =",qa.diff_not_flagged.abs().max())
print("median abs diff: qual_frame>0 =",qa.diff_qual_frame.abs().median())
print("max abs diff:    qual_frame>0 =",qa.diff_qual_frame.abs().max())
print()

formula_scores={
    "mean_good_quality":qa.diff_good_quality.abs().max(),
    "fraction_not_quality_flagged":qa.diff_not_flagged.abs().max(),
    "fraction_qual_frame_positive":qa.diff_qual_frame.abs().max()
}
best_formula=min(formula_scores,key=formula_scores.get)

print("BEST REPRODUCING FORMULA:",best_formula)
print()

qa["quality_reproduced"]=np.select(
    [
        np.isclose(qa.stored_good_quality_fraction,qa.mean_good_quality,atol=1e-9,rtol=1e-9),
        np.isclose(qa.stored_good_quality_fraction,qa.fraction_not_quality_flagged,atol=1e-9,rtol=1e-9),
        np.isclose(qa.stored_good_quality_fraction,qa.fraction_qual_frame_positive,atol=1e-9,rtol=1e-9)
    ],
    ["mean_good_quality","fraction_not_quality_flagged","fraction_qual_frame_positive"],
    default="NONE"
)

print("FORMULA MATCH COUNTS")
print(qa.quality_reproduced.value_counts(dropna=False))
print()

if best_formula=="mean_good_quality":
    quality_source=qa.set_index("group_id")["mean_good_quality"]
elif best_formula=="fraction_not_quality_flagged":
    quality_source=qa.set_index("group_id")["fraction_not_quality_flagged"]
else:
    quality_source=qa.set_index("group_id")["fraction_qual_frame_positive"]

rows=[]

for gid in controls_unique.group_id:
    x=det[det.group_id==gid].copy()
    if len(x)==0:
        continue
    x=x[pd.to_numeric(x["w1mpro"],errors="coerce").notna()]
    x=x[pd.to_numeric(x["w1sigmpro"],errors="coerce").notna()]
    x=x[pd.to_numeric(x["w1sigmpro"],errors="coerce")>0]
    if len(x)==0:
        continue

    m=x["w1mpro"].astype(float).to_numpy()
    e=x["w1sigmpro"].astype(float).to_numpy()
    snr=1.0857/e
    w=1/(e**2)
    mean=np.sum(w*m)/np.sum(w)
    chi2=np.sum(((m-mean)/e)**2)
    rchi2=chi2/(len(m)-1) if len(m)>1 else np.nan
    med_snr=float(np.median(snr))

    q=float(quality_source.get(gid,np.nan))

    positive=(
        len(m)>=10 and
        med_snr>=5 and
        rchi2>=2 and
        q>=0.90
    )

    rows.append({
        "group_id":gid,
        "n_obs":len(m),
        "median_w1_snr_recomputed":med_snr,
        "w1_reduced_chi2_recomputed":rchi2,
        "quality_fraction_correct":q,
        "positive":bool(positive)
    })

v=pd.DataFrame(rows)

v=v.merge(
    groups[["group_id","n_observations","median_w1_snr","w1_reduced_chi2","good_quality_fraction"]],
    on="group_id",
    how="left"
)

v["old_label"]=controls_unique.set_index("group_id")["validator_label"].reindex(v.group_id).values if "validator_label" in controls_unique.columns else "UNKNOWN"

print("CORRECTED BASELINE")
print("unique groups:",len(v))
print("positive:",int(v.positive.sum()))
print("trigger rate:",float(v.positive.mean()))
print()
print(v[v.positive].sort_values("group_id")[[
    "group_id","n_obs","median_w1_snr_recomputed",
    "w1_reduced_chi2_recomputed","quality_fraction_correct"
]].to_string(index=False))
print()

print("QUALITY COUNTS")
print(pd.cut(
    v.quality_fraction_correct,
    [-np.inf,0.5,0.7,0.9,0.95,1.0,np.inf],
    labels=["<0.5","0.5-0.7","0.7-0.9","0.9-0.95","0.95-1.0",">1"]
).value_counts().sort_index())
print()

def run_validator(df,nobs=10,snr=5,chi2=2,quality=.90):
    return (
        (df.n_obs>=nobs) &
        (df.median_w1_snr_recomputed>=snr) &
        (df.w1_reduced_chi2_recomputed>=chi2) &
        (df.quality_fraction_correct>=quality)
    )

tests=[
    ("baseline",10,5,2,.90),
    ("nobs_15",15,5,2,.90),
    ("nobs_20",20,5,2,.90),
    ("snr_7",10,7,2,.90),
    ("snr_10",10,10,2,.90),
    ("chi2_3",10,5,3,.90),
    ("chi2_4",10,5,4,.90),
    ("quality_.95",10,5,2,.95),
    ("quality_1.0",10,5,2,1.0)
]

sensitivity=[]

for name,nobs,snr,chi2,q in tests:
    mask=run_validator(v,nobs,snr,chi2,q)
    sensitivity.append({
        "test":name,
        "n_groups":len(v),
        "positive":int(mask.sum()),
        "trigger_rate":float(mask.mean()),
        "nobs_threshold":nobs,
        "snr_threshold":snr,
        "chi2_threshold":chi2,
        "quality_threshold":q
    })

sens=pd.DataFrame(sensitivity)

print("CORRECTED THRESHOLD SENSITIVITY")
print(sens.to_string(index=False))
print()

base=run_validator(v)

print("BASELINE COMPONENT FAILURES")
print("fails n_obs:",int((v.n_obs<10).sum()))
print("fails SNR:",int((v.median_w1_snr_recomputed<5).sum()))
print("fails chi2:",int((v.w1_reduced_chi2_recomputed<2).sum()))
print("fails quality:",int((v.quality_fraction_correct<.90).sum()))
print()

print("GROUPS AFFECTED BY QUALITY THRESHOLD")
q90=v.quality_fraction_correct>=.90
q100=v.quality_fraction_correct>=1.0
print("positive at quality>=0.90:",int((base).sum()))
print("positive at quality>=1.00:",int((q100 & (v.n_obs>=10) & (v.median_w1_snr_recomputed>=5) & (v.w1_reduced_chi2_recomputed>=2)).sum()))
print()

print("GROUP 1778 CHECK")
print(v[v.group_id==1778][[
    "group_id","n_obs","median_w1_snr_recomputed",
    "w1_reduced_chi2_recomputed","quality_fraction_correct"
]].to_string(index=False))
print()

comparison=v.copy()
comparison["baseline_positive"]=base
comparison["old_positive"]=comparison["old_label"].isin(["VARIABLE_HIGH_QUALITY","VARIABLE_LOW_QUALITY"])
comparison["agreement"]=comparison.baseline_positive==comparison.old_positive

print("OLD VS CORRECTED")
print("old positive:",int(comparison.old_positive.sum()))
print("corrected positive:",int(comparison.baseline_positive.sum()))
print("agreement:",int(comparison.agreement.sum()),"/",len(comparison))
print("agreement rate:",float(comparison.agreement.mean()))
print()

print("DISAGREEMENTS")
print(comparison[~comparison.agreement][[
    "group_id","old_label","n_obs",
    "median_w1_snr_recomputed",
    "w1_reduced_chi2_recomputed",
    "quality_fraction_correct",
    "baseline_positive"
]].sort_values("group_id").to_string(index=False))
print()

out=BASE/"exp019D_quality_metric_reproducibility_audit.csv"
qa.to_csv(out,index=False)

vout=BASE/"exp019D_corrected_validator_results.csv"
comparison.to_csv(vout,index=False)

sout=BASE/"exp019D_corrected_threshold_sensitivity.csv"
sens.to_csv(sout,index=False)

summary={
    "experiment":"EXP-019D",
    "purpose":"Quality metric reproducibility and corrected validator audit",
    "controls_input":int(len(controls)),
    "unique_neowise_groups":int(len(v)),
    "best_quality_formula":best_formula,
    "formula_max_abs_error":float(formula_scores[best_formula]),
    "corrected_baseline_positive":int(base.sum()),
    "corrected_baseline_trigger_rate":float(base.mean()),
    "old_positive":int(comparison.old_positive.sum()),
    "label_agreement_count":int(comparison.agreement.sum()),
    "label_agreement_rate":float(comparison.agreement.mean()),
    "group1778_quality_fraction":float(v.loc[v.group_id==1778,"quality_fraction_correct"].iloc[0]) if (v.group_id==1778).any() else None,
    "threshold_sensitivity":sens.to_dict(orient="records"),
    "interpretation":"Validator quality metric audited and baseline recomputed using the empirically reproduced EXP005 quality definition."
}

with open(BASE/"exp019D_quality_metric_reproducibility_summary.json","w") as f:
    json.dump(summary,f,indent=2)

print("SAVED")
print(out)
print(vout)
print(sout)
print(BASE/"exp019D_quality_metric_reproducibility_summary.json")

INPUTS
controls: 84
groups: 175
detections: 7275

UNIQUE CONTROL GROUPS: 82

QUALITY FORMULA REPRODUCIBILITY
median abs diff: mean(good_quality) = 0.0
max abs diff:    mean(good_quality) = 1.1102230246251565e-16
median abs diff: 1-mean(quality_flagged) = 0.0
max abs diff:    1-mean(quality_flagged) = 1.1102230246251565e-16
median abs diff: qual_frame>0 = 0.2950310559006211
max abs diff:    qual_frame>0 = 1.0

BEST REPRODUCING FORMULA: mean_good_quality

FORMULA MATCH COUNTS
quality_reproduced
mean_good_quality    82
Name: count, dtype: int64

CORRECTED BASELINE
unique groups: 82
positive: 6
trigger rate: 0.07317073170731707

 group_id  n_obs  median_w1_snr_recomputed  w1_reduced_chi2_recomputed  quality_fraction_correct
      233     22                 35.022581                    2.118475                  1.000000
      234     24                 33.414062                    2.493238                  1.000000
     1974     25                 22.618750                    3.026153      

In [27]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from itertools import combinations

BASE=Path("astronomy_exp005/results")
GIDS=[233,234,1974,1976,1978,1979]

g=pd.read_csv(BASE/"exp005_all_source_groups.csv")
d=pd.read_csv(BASE/"exp005_region_detections.csv")
c=pd.read_csv(BASE/"exp017_full_randomized_controls.csv")

for x in [g,d,c]:
    if "group_id" in x.columns:
        x["group_id"]=pd.to_numeric(x["group_id"],errors="coerce").astype("Int64")

d["mjd"]=pd.to_numeric(d["mjd"],errors="coerce")
d["w1mpro"]=pd.to_numeric(d["w1mpro"],errors="coerce")
d["w1sigmpro"]=pd.to_numeric(d["w1sigmpro"],errors="coerce")
d["w2mpro"]=pd.to_numeric(d["w2mpro"],errors="coerce")
d["w2sigmpro"]=pd.to_numeric(d["w2sigmpro"],errors="coerce")
d["scan_id"]=d["scan_id"].astype(str)
d["frame_num"]=d["frame_num"].astype(str)

surv=g[g.group_id.isin(GIDS)].copy()
assert len(surv)==6

print("EXP-020 — SIX-SURVIVOR SYSTEMATICS AUDIT")
print("="*55)
print("survivors:",GIDS)
print("detections:",len(d))
print()

def epoch_id(mjd):
    return int(np.floor(mjd/150.0))

summary=[]
epoch_rows=[]
scan_rows=[]
frame_rows=[]

for gid in GIDS:
    x=d[d.group_id==gid].copy().sort_values("mjd")
    assert len(x)>0

    x["epoch"]=x.mjd.map(epoch_id)
    x["w1_snr"]=1.0857/x.w1sigmpro.replace(0,np.nan)
    x["w2_snr"]=1.0857/x.w2sigmpro.replace(0,np.nan)

    q=x["good_quality"].astype(float)
    quality=float(q.mean())

    w1=x.dropna(subset=["w1mpro","w1sigmpro"])
    w1=w1[w1.w1sigmpro>0]

    w2=x.dropna(subset=["w2mpro","w2sigmpro"])
    w2=w2[w2.w2sigmpro>0]

    w1_mean=np.average(w1.w1mpro,weights=1/(w1.w1sigmpro**2)) if len(w1) else np.nan
    w2_mean=np.average(w2.w2mpro,weights=1/(w2.w2sigmpro**2)) if len(w2) else np.nan

    w1_rchi2=np.sum(((w1.w1mpro-w1_mean)/w1.w1sigmpro)**2)/(len(w1)-1) if len(w1)>1 else np.nan
    w2_rchi2=np.sum(((w2.w2mpro-w2_mean)/w2.w2sigmpro)**2)/(len(w2)-1) if len(w2)>1 else np.nan

    epoch_stats=[]
    for ep,z in x.groupby("epoch"):
        z=z.dropna(subset=["w1mpro","w1sigmpro"])
        z=z[z.w1sigmpro>0]
        if len(z)<2:
            continue
        mu=np.average(z.w1mpro,weights=1/(z.w1sigmpro**2))
        rz=(z.w1mpro-mu)/z.w1sigmpro
        epoch_stats.append({
            "group_id":int(gid),
            "epoch":int(ep),
            "n":len(z),
            "w1_std":float(z.w1mpro.std(ddof=1)),
            "max_abs_z":float(np.max(np.abs(rz))),
            "frac_abs_z2":float(np.mean(np.abs(rz)>=2)),
            "mjd_min":float(z.mjd.min()),
            "mjd_max":float(z.mjd.max())
        })

    epoch_rows.extend(epoch_stats)

    summary.append({
        "group_id":int(gid),
        "ra":float(surv.loc[surv.group_id==gid,"ra"].iloc[0]),
        "dec":float(surv.loc[surv.group_id==gid,"dec"].iloc[0]),
        "n_obs":len(x),
        "n_epochs":x.epoch.nunique(),
        "time_span_days":float(x.mjd.max()-x.mjd.min()),
        "median_w1_snr":float(x.w1_snr.median()),
        "median_w2_snr":float(x.w2_snr.median()),
        "w1_std":float(w1.w1mpro.std(ddof=1)),
        "w2_std":float(w2.w2mpro.std(ddof=1)) if len(w2)>1 else np.nan,
        "w1_range":float(w1.w1mpro.max()-w1.w1mpro.min()),
        "w2_range":float(w2.w2mpro.max()-w2.w2mpro.min()) if len(w2) else np.nan,
        "w1_rchi2":float(w1_rchi2),
        "w2_rchi2":float(w2_rchi2),
        "w2_coverage":float(len(w2)/len(x)),
        "good_quality_fraction":quality,
        "flagged_count":int((x["quality_flagged"]==True).sum()),
        "unique_scan_ids":int(x.scan_id.nunique()),
        "unique_frame_nums":int(x.frame_num.nunique()),
        "median_position_offset_arcsec":float(surv.loc[surv.group_id==gid,"median_position_offset_arcsec"].iloc[0]),
        "max_position_offset_arcsec":float(surv.loc[surv.group_id==gid,"max_position_offset_arcsec"].iloc[0])
    })

    for scan,n in x.groupby("scan_id").size().items():
        scan_rows.append({"group_id":int(gid),"scan_id":scan,"n":int(n)})

    for frame,n in x.groupby("frame_num").size().items():
        frame_rows.append({"group_id":int(gid),"frame_num":frame,"n":int(n)})

s=pd.DataFrame(summary)
ep=pd.DataFrame(epoch_rows)
sc=pd.DataFrame(scan_rows)
fr=pd.DataFrame(frame_rows)

print("SURVIVOR SUMMARY")
print(s.to_string(index=False))
print()

print("QUALITY")
print(s[["group_id","good_quality_fraction","flagged_count"]].to_string(index=False))
print()

print("OBSERVING EPOCHS")
print(ep.to_string(index=False))
print()

print("SHARED SCAN IDs")
scan_sets={gid:set(sc.loc[sc.group_id==gid,"scan_id"]) for gid in GIDS}
shared_scans=[]
for a,b in combinations(GIDS,2):
    common=scan_sets[a]&scan_sets[b]
    shared_scans.append({
        "group_a":a,
        "group_b":b,
        "shared_scan_count":len(common),
        "shared_scans":";".join(sorted(common))
    })
shared_scans=pd.DataFrame(shared_scans)
print(shared_scans.to_string(index=False))
print()

print("SHARED FRAME NUMBERS")
frame_sets={gid:set(fr.loc[fr.group_id==gid,"frame_num"]) for gid in GIDS}
shared_frames=[]
for a,b in combinations(GIDS,2):
    common=frame_sets[a]&frame_sets[b]
    shared_frames.append({
        "group_a":a,
        "group_b":b,
        "shared_frame_count":len(common)
    })
shared_frames=pd.DataFrame(shared_frames)
print(shared_frames.to_string(index=False))
print()

print("SHARED EPOCHS")
epoch_sets={gid:set(ep.loc[ep.group_id==gid,"epoch"]) for gid in GIDS}
shared_epochs=[]
for a,b in combinations(GIDS,2):
    common=epoch_sets[a]&epoch_sets[b]
    shared_epochs.append({
        "group_a":a,
        "group_b":b,
        "shared_epoch_count":len(common),
        "shared_epochs":";".join(map(str,sorted(common)))
    })
shared_epochs=pd.DataFrame(shared_epochs)
print(shared_epochs.to_string(index=False))
print()

print("CROSS-BAND COMPARISON")
s["w1_w2_std_ratio"]=s.w2_std/s.w1_std
s["w1_w2_range_ratio"]=s.w2_range/s.w1_range
print(s[[
    "group_id","w1_std","w2_std","w1_range","w2_range",
    "w1_rchi2","w2_rchi2","w2_coverage",
    "w1_w2_std_ratio","w1_w2_range_ratio"
]].to_string(index=False))
print()

print("WITHIN-EPOCH SYSTEMATICS")
ep_summary=ep.groupby("group_id").agg(
    epochs=("epoch","count"),
    median_epoch_std=("w1_std","median"),
    median_max_abs_z=("max_abs_z","median"),
    median_frac_abs_z2=("frac_abs_z2","median"),
    max_epoch_abs_z=("max_abs_z","max")
).reset_index()
print(ep_summary.to_string(index=False))
print()

print("NEIGHBORHOOD DENSITY")
neighborhood=[]
for gid in GIDS:
    x=d[d.group_id==gid][["ra","dec"]].dropna()
    ra=float(x.ra.median())
    dec=float(x.dec.median())
    dra=(d.ra-ra)*3600*np.cos(np.deg2rad(dec))
    ddec=(d.dec-dec)*3600
    sep=np.sqrt(dra**2+ddec**2)
    neighborhood.append({
        "group_id":gid,
        "detections_within_5arcsec":int((sep<=5).sum()),
        "detections_within_10arcsec":int((sep<=10).sum()),
        "detections_within_15arcsec":int((sep<=15).sum()),
        "other_groups_within_5arcsec":int(d[(sep<=5)&(d.group_id!=gid)].group_id.nunique()),
        "other_groups_within_10arcsec":int(d[(sep<=10)&(d.group_id!=gid)].group_id.nunique()),
        "other_groups_within_15arcsec":int(d[(sep<=15)&(d.group_id!=gid)].group_id.nunique())
    })
neighborhood=pd.DataFrame(neighborhood)
print(neighborhood.to_string(index=False))
print()

print("POSITION")
print(s[[
    "group_id","ra","dec",
    "median_position_offset_arcsec","max_position_offset_arcsec"
]].to_string(index=False))
print()

print("PAIRWISE SKY SEPARATIONS")
positions=s.set_index("group_id")[["ra","dec"]]
pairs=[]
for a,b in combinations(GIDS,2):
    ra1,dec1=positions.loc[a]
    ra2,dec2=positions.loc[b]
    dra=(ra2-ra1)*np.cos(np.deg2rad((dec1+dec2)/2))
    ddec=dec2-dec1
    sep=np.sqrt(dra**2+ddec**2)*3600
    pairs.append({"group_a":a,"group_b":b,"separation_arcsec":float(sep)})
pairs=pd.DataFrame(pairs)
print(pairs.to_string(index=False))
print()

print("SYSTEMATICS FLAGS")
flags=[]

for _,r in s.iterrows():
    gid=int(r.group_id)
    e=ep_summary[ep_summary.group_id==gid].iloc[0]
    q=float(r.good_quality_fraction)
    w2=float(r.w2_coverage)
    flags.append({
        "group_id":gid,
        "low_quality":q<0.90,
        "poor_w2_coverage":w2<0.75,
        "strong_within_epoch_scatter":float(e.median_max_abs_z)>=3,
        "multiple_within_epoch_2sigma":float(e.median_frac_abs_z2)>=0.10,
        "w2_rchi2_also_high":float(r.w2_rchi2)>=2 if pd.notna(r.w2_rchi2) else False,
        "large_position_offset":float(r.max_position_offset_arcsec)>=1.0,
        "many_nearby_detections":int(neighborhood.loc[neighborhood.group_id==gid,"detections_within_10arcsec"].iloc[0])>=10
    })

flags=pd.DataFrame(flags)
print(flags.to_string(index=False))
print()

pair_shared_scan=int((shared_scans.shared_scan_count>0).sum())
pair_shared_epoch=int((shared_epochs.shared_epoch_count>0).sum())

print("OVERALL SYSTEMATICS TEST")
print("survivors:",len(GIDS))
print("pairs:",len(shared_scans))
print("pairs sharing scan IDs:",pair_shared_scan)
print("pairs sharing epochs:",pair_shared_epoch)
print("median W1 rchi2:",float(s.w1_rchi2.median()))
print("median W2 rchi2:",float(s.w2_rchi2.median()))
print("median W2 coverage:",float(s.w2_coverage.median()))
print("median quality:",float(s.good_quality_fraction.median()))
print()

shared_scan_concern=pair_shared_scan>0
shared_epoch_concern=pair_shared_epoch==len(shared_epochs)
within_epoch_concern=bool(flags.strong_within_epoch_scatter.mean()>=0.5)
w2_support=bool((s.w2_rchi2>=2).sum()>=3)
poor_w2=bool((s.w2_coverage<0.75).sum()>=3)

if shared_scan_concern:
    conclusion="INVESTIGATE_SHARED_OBSERVING_CONDITIONS"
elif within_epoch_concern:
    conclusion="INVESTIGATE_WITHIN_EPOCH_SYSTEMATICS"
elif w2_support:
    conclusion="MULTI_BAND_SUPPORT_PRESENT_BUT_NOT_CONFIRMATION"
elif poor_w2:
    conclusion="CROSS_BAND_EVIDENCE_LIMITED"
else:
    conclusion="NO_SINGLE_DOMINANT_SYSTEMATIC_IDENTIFIED"

print("AUTOMATED CONCLUSION:",conclusion)
print()

print("SCIENTIFIC INTERPRETATION")
print("This experiment does not classify any survivor as astrophysical.")
print("It tests whether the six survivors show shared observational/systematic signatures.")
print("Shared scan/frame conditions would be evidence for investigation, not proof of an artifact.")
print("W2 agreement would strengthen astrophysical plausibility but is not required for real variability.")
print("Absence of a detected common systematic does not prove astrophysical variability.")

out1=BASE/"exp020_survivor_systematics_summary.csv"
out2=BASE/"exp020_survivor_epoch_behavior.csv"
out3=BASE/"exp020_shared_scan_pairs.csv"
out4=BASE/"exp020_shared_epoch_pairs.csv"
out5=BASE/"exp020_systematics_flags.csv"
out6=BASE/"exp020_survivor_sky_pairs.csv"

s.to_csv(out1,index=False)
ep.to_csv(out2,index=False)
shared_scans.to_csv(out3,index=False)
shared_epochs.to_csv(out4,index=False)
flags.to_csv(out5,index=False)
pairs.to_csv(out6,index=False)

result={
    "experiment":"EXP-020",
    "survivor_groups":GIDS,
    "n_survivors":len(GIDS),
    "pair_count":len(shared_scans),
    "pairs_sharing_scan_ids":pair_shared_scan,
    "pairs_sharing_epochs":pair_shared_epoch,
    "median_w1_rchi2":float(s.w1_rchi2.median()),
    "median_w2_rchi2":float(s.w2_rchi2.median()),
    "median_w2_coverage":float(s.w2_coverage.median()),
    "median_quality_fraction":float(s.good_quality_fraction.median()),
    "shared_scan_concern":shared_scan_concern,
    "shared_epoch_concern":shared_epoch_concern,
    "within_epoch_concern":within_epoch_concern,
    "w2_support":w2_support,
    "poor_w2_coverage":poor_w2,
    "automated_conclusion":conclusion,
    "scientific_status":"INVESTIGATION"
}

with open(BASE/"exp020_survivor_systematics_summary.json","w") as f:
    json.dump(result,f,indent=2)

print()
print("SAVED")
for x in [out1,out2,out3,out4,out5,out6,BASE/"exp020_survivor_systematics_summary.json"]:
    print(x)

EXP-020 — SIX-SURVIVOR SYSTEMATICS AUDIT
survivors: [233, 234, 1974, 1976, 1978, 1979]
detections: 7275

SURVIVOR SUMMARY
 group_id         ra       dec  n_obs  n_epochs  time_span_days  median_w1_snr  median_w2_snr   w1_std   w2_std  w1_range  w2_range  w1_rchi2  w2_rchi2  w2_coverage  good_quality_fraction  flagged_count  unique_scan_ids  unique_frame_nums  median_position_offset_arcsec  max_position_offset_arcsec
      233 180.009612 -0.037690     22         3      160.290207      35.022581      15.400775 0.046449 0.087658     0.157     0.372  2.118475  1.447637     1.000000               1.000000              0               20                  8                       0.104495                    0.212621
      234 180.040117 -0.045336     24         3      160.290207      33.414062      14.192763 0.059269 0.087714     0.310     0.400  2.493238  1.161600     1.000000               1.000000              0               22                  8                       0.138209             

In [28]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from itertools import combinations

BASE=Path("astronomy_exp005/results")
GIDS=[233,234,1974,1976,1978,1979]

g=pd.read_csv(BASE/"exp005_all_source_groups.csv")
d=pd.read_csv(BASE/"exp005_region_detections.csv")

for x in [g,d]:
    x["group_id"]=pd.to_numeric(x["group_id"],errors="coerce").astype("Int64")

for col in ["mjd","w1mpro","w1sigmpro","w2mpro","w2sigmpro","ra","dec"]:
    if col in d.columns:
        d[col]=pd.to_numeric(d[col],errors="coerce")

d["scan_id"]=d["scan_id"].astype(str)
d["frame_num"]=d["frame_num"].astype(str)
d["good_quality"]=d["good_quality"].astype(bool)

print("EXP-021 — SCAN/FRAME COMMON-MODE SYSTEMATICS TEST")
print("="*60)
print("survivors:",GIDS)
print("detections:",len(d))
print("source groups:",len(g))
print()

survivors=set(GIDS)

def weighted_mean(x):
    x=x.dropna(subset=["w1mpro","w1sigmpro"])
    x=x[x.w1sigmpro>0]
    if len(x)==0:
        return np.nan
    return float(np.average(x.w1mpro,weights=1/(x.w1sigmpro**2)))

def prepare_group(gid):
    x=d[d.group_id==gid].copy()
    x=x[x.good_quality]
    x=x.dropna(subset=["w1mpro","w1sigmpro","scan_id","frame_num"])
    x=x[x.w1sigmpro>0]
    if len(x)<3:
        return None
    mu=weighted_mean(x)
    x["w1_resid"]=x.w1mpro-mu
    x["w1_z"]=x.w1_resid/x.w1sigmpro
    x["group_mean"]=mu
    return x

all_groups=[]
for gid in g.group_id.dropna().astype(int).unique():
    if gid in survivors:
        continue
    x=prepare_group(gid)
    if x is None:
        continue
    q=float(g.loc[g.group_id==gid,"good_quality_fraction"].iloc[0])
    nobs=int(g.loc[g.group_id==gid,"n_observations"].iloc[0])
    if q>=0.90 and nobs>=10:
        all_groups.append(gid)

print("CONTROL GROUPS ELIGIBLE:",len(all_groups))
print()

control_detections=[]
for gid in all_groups:
    x=prepare_group(gid)
    if x is not None:
        control_detections.append(x)

if len(control_detections)==0:
    raise RuntimeError("No eligible control detections")

ctrl=pd.concat(control_detections,ignore_index=True)

print("CONTROL DETECTIONS:",len(ctrl))
print("CONTROL GROUPS:",ctrl.group_id.nunique())
print()

scan_stats=ctrl.groupby("scan_id").agg(
    n_control_groups=("group_id","nunique"),
    n_control_detections=("group_id","size"),
    common_residual=("w1_resid","median"),
    common_z=("w1_z","median")
).reset_index()

scan_stats=scan_stats[scan_stats.n_control_groups>=3].copy()

print("SCAN COMMON-MODE REFERENCE")
print("usable scans:",len(scan_stats))
print("median control groups per scan:",float(scan_stats.n_control_groups.median()))
print("minimum control groups:",int(scan_stats.n_control_groups.min()))
print()

scan_common=dict(zip(scan_stats.scan_id,scan_stats.common_residual))
scan_common_z=dict(zip(scan_stats.scan_id,scan_stats.common_z))

survivor_rows=[]
survivor_scan_rows=[]
survivor_frame_rows=[]

for gid in GIDS:
    x=prepare_group(gid)
    if x is None:
        continue

    x=x.copy()
    x["scan_common_resid"]=x.scan_id.map(scan_common)
    x["scan_common_z"]=x.scan_id.map(scan_common_z)

    x["frame_common_resid"]=ctrl.groupby("frame_num")["w1_resid"].transform("median").reindex(x.index)
    frame_common=ctrl.groupby("frame_num")["w1_resid"].median().to_dict()
    x["frame_common_resid"]=x.frame_num.map(frame_common)

    sx=x.dropna(subset=["scan_common_resid"]).copy()
    fx=x.dropna(subset=["frame_common_resid"]).copy()

    scan_corr=np.nan
    scan_spearman=np.nan
    if len(sx)>=3 and sx.w1_resid.nunique()>1 and sx.scan_common_resid.nunique()>1:
        scan_corr=float(sx.w1_resid.corr(sx.scan_common_resid))
        scan_spearman=float(sx.w1_resid.rank().corr(sx.scan_common_resid.rank()))

    frame_corr=np.nan
    if len(fx)>=3 and fx.w1_resid.nunique()>1 and fx.frame_common_resid.nunique()>1:
        frame_corr=float(fx.w1_resid.corr(fx.frame_common_resid))

    same_scan_direction=np.nan
    if len(sx):
        same_scan_direction=float(np.mean(np.sign(sx.w1_resid)==np.sign(sx.scan_common_resid)))

    same_frame_direction=np.nan
    if len(fx):
        same_frame_direction=float(np.mean(np.sign(fx.w1_resid)==np.sign(fx.frame_common_resid)))

    median_common_abs=np.nan
    if len(sx):
        median_common_abs=float(np.median(np.abs(sx.scan_common_resid)))

    median_survivor_abs=np.nan
    if len(sx):
        median_survivor_abs=float(np.median(np.abs(sx.w1_resid)))

    survivor_rows.append({
        "group_id":gid,
        "n_valid_detections":len(x),
        "n_scan_matched":len(sx),
        "scan_coverage":float(len(sx)/len(x)),
        "scan_residual_corr":scan_corr,
        "scan_residual_spearman":scan_spearman,
        "same_scan_direction":same_scan_direction,
        "median_survivor_abs_resid":median_survivor_abs,
        "median_scan_common_abs_resid":median_common_abs,
        "n_frame_matched":len(fx),
        "frame_coverage":float(len(fx)/len(x)),
        "frame_residual_corr":frame_corr,
        "same_frame_direction":same_frame_direction
    })

    for _,r in sx.iterrows():
        survivor_scan_rows.append({
            "group_id":gid,
            "mjd":float(r.mjd),
            "scan_id":r.scan_id,
            "frame_num":r.frame_num,
            "w1_resid":float(r.w1_resid),
            "w1_z":float(r.w1_z),
            "scan_common_resid":float(r.scan_common_resid),
            "scan_common_z":float(r.scan_common_z),
            "same_direction":bool(np.sign(r.w1_resid)==np.sign(r.scan_common_resid))
        })

    for _,r in fx.iterrows():
        survivor_frame_rows.append({
            "group_id":gid,
            "mjd":float(r.mjd),
            "scan_id":r.scan_id,
            "frame_num":r.frame_num,
            "w1_resid":float(r.w1_resid),
            "frame_common_resid":float(r.frame_common_resid),
            "same_direction":bool(np.sign(r.w1_resid)==np.sign(r.frame_common_resid))
        })

surv=pd.DataFrame(survivor_rows)
s_scan=pd.DataFrame(survivor_scan_rows)
s_frame=pd.DataFrame(survivor_frame_rows)

print("SURVIVOR SCAN CORRELATIONS")
print(surv.to_string(index=False))
print()

print("SURVIVOR SCAN RESIDUALS")
print(s_scan.to_string(index=False))
print()

print("SURVIVOR FRAME CORRELATIONS")
print(surv[[
    "group_id",
    "n_frame_matched",
    "frame_coverage",
    "frame_residual_corr",
    "same_frame_direction"
]].to_string(index=False))
print()

print("CONTROL SCAN CORRELATION DISTRIBUTION")

control_corr_rows=[]

for gid in all_groups:
    x=prepare_group(gid)
    if x is None:
        continue

    x=x.copy()
    x["scan_common_resid"]=x.scan_id.map(scan_common)

    sx=x.dropna(subset=["scan_common_resid"])

    if len(sx)<3 or sx.w1_resid.nunique()<=1 or sx.scan_common_resid.nunique()<=1:
        continue

    corr=float(sx.w1_resid.corr(sx.scan_common_resid))
    same=float(np.mean(np.sign(sx.w1_resid)==np.sign(sx.scan_common_resid)))

    control_corr_rows.append({
        "group_id":gid,
        "n_scan_matched":len(sx),
        "scan_coverage":float(len(sx)/len(x)),
        "scan_residual_corr":corr,
        "same_scan_direction":same
    })

control_corr=pd.DataFrame(control_corr_rows)

print("controls with usable correlations:",len(control_corr))

if len(control_corr):
    print("median control correlation:",float(control_corr.scan_residual_corr.median()))
    print("95th percentile:",float(control_corr.scan_residual_corr.quantile(.95)))
    print("99th percentile:",float(control_corr.scan_residual_corr.quantile(.99)))
    print("maximum:",float(control_corr.scan_residual_corr.max()))
print()

comparison=[]

for _,r in surv.iterrows():
    gid=int(r.group_id)
    corr=r.scan_residual_corr

    if len(control_corr) and pd.notna(corr):
        percentile=float(np.mean(control_corr.scan_residual_corr<=corr))
        extreme_fraction=float(np.mean(control_corr.scan_residual_corr>=corr))
    else:
        percentile=np.nan
        extreme_fraction=np.nan

    comparison.append({
        "group_id":gid,
        "survivor_scan_corr":corr,
        "control_percentile":percentile,
        "control_fraction_at_or_above":extreme_fraction,
        "same_scan_direction":r.same_scan_direction,
        "scan_coverage":r.scan_coverage
    })

comparison=pd.DataFrame(comparison)

print("SURVIVOR VS CONTROL CORRELATION")
print(comparison.to_string(index=False))
print()

print("NEARBY SOURCE TEST")

nearby_rows=[]

for gid in GIDS:
    row=g[g.group_id==gid].iloc[0]
    ra=float(row.ra)
    dec=float(row.dec)

    dd=d.copy()
    dra=(dd.ra-ra)*3600*np.cos(np.deg2rad(dec))
    ddec=(dd.dec-dec)*3600
    sep=np.sqrt(dra**2+ddec**2)

    nearby_groups=dd[(sep<=10)&(~dd.group_id.isin(list(survivors)))].group_id.dropna().unique()

    nearby_groups=[int(x) for x in nearby_groups if int(x) in all_groups]

    x=prepare_group(gid)

    for radius in [5,10]:
        ng=dd[(sep<=radius)&(~dd.group_id.isin(list(survivors)))].group_id.dropna().unique()
        ng=[int(x) for x in ng if int(x) in all_groups]

        if len(ng):
            vals=[]
            for cg in ng:
                cx=prepare_group(cg)
                if cx is not None:
                    vals.extend(cx.w1_resid.tolist())

            nearby_median=float(np.median(vals)) if vals else np.nan
            nearby_n=len(vals)
        else:
            nearby_median=np.nan
            nearby_n=0

        nearby_rows.append({
            "group_id":gid,
            "radius_arcsec":radius,
            "nearby_control_groups":len(ng),
            "nearby_control_detections":nearby_n,
            "nearby_control_median_residual":nearby_median
        })

nearby=pd.DataFrame(nearby_rows)

print(nearby.to_string(index=False))
print()

print("SCAN-LEVEL SUMMARY")

scan_summary=[]

for gid in GIDS:
    x=s_scan[s_scan.group_id==gid]

    if len(x)==0:
        continue

    scan_summary.append({
        "group_id":gid,
        "n_shared_scans":x.scan_id.nunique(),
        "median_survivor_resid":float(x.w1_resid.median()),
        "median_common_scan_resid":float(x.scan_common_resid.median()),
        "median_abs_difference":float(np.median(np.abs(x.w1_resid-x.scan_common_resid))),
        "same_direction_fraction":float(x.same_direction.mean()),
        "positive_survivor_and_common":int(((x.w1_resid>0)&(x.scan_common_resid>0)).sum()),
        "negative_survivor_and_common":int(((x.w1_resid<0)&(x.scan_common_resid<0)).sum())
    })

scan_summary=pd.DataFrame(scan_summary)
print(scan_summary.to_string(index=False))
print()

print("DECISION FLAGS")

flags=[]

for _,r in surv.iterrows():
    gid=int(r.group_id)

    corr=bool(pd.notna(r.scan_residual_corr) and r.scan_residual_corr>=0.5)
    strong_corr=bool(pd.notna(r.scan_residual_corr) and r.scan_residual_corr>=0.7)
    direction=bool(pd.notna(r.same_scan_direction) and r.same_scan_direction>=0.70)

    if len(control_corr) and pd.notna(r.scan_residual_corr):
        control_extreme=bool(r.scan_residual_corr>=control_corr.scan_residual_corr.quantile(.95))
    else:
        control_extreme=False

    frame_corr=bool(pd.notna(r.frame_residual_corr) and r.frame_residual_corr>=0.5)

    if strong_corr and control_extreme and direction:
        verdict="STRONG_SHARED_SCAN_SIGNATURE"
    elif corr and direction:
        verdict="POSSIBLE_SHARED_SCAN_SIGNATURE"
    elif frame_corr:
        verdict="POSSIBLE_FRAME_SIGNATURE"
    else:
        verdict="NO_STRONG_COMMON_MODE_SIGNATURE"

    flags.append({
        "group_id":gid,
        "scan_corr_ge_0.5":corr,
        "scan_corr_ge_0.7":strong_corr,
        "same_scan_direction_ge_0.7":direction,
        "control_95th_percentile_exceeded":control_extreme,
        "frame_corr_ge_0.5":frame_corr,
        "systematics_verdict":verdict
    })

flags=pd.DataFrame(flags)

print(flags.to_string(index=False))
print()

print("OVERALL TEST")

n_strong=int((flags.systematics_verdict=="STRONG_SHARED_SCAN_SIGNATURE").sum())
n_possible=int((flags.systematics_verdict=="POSSIBLE_SHARED_SCAN_SIGNATURE").sum())
n_frame=int((flags.systematics_verdict=="POSSIBLE_FRAME_SIGNATURE").sum())

print("strong shared-scan signatures:",n_strong)
print("possible shared-scan signatures:",n_possible)
print("possible frame signatures:",n_frame)

if n_strong>=3:
    overall="STRONG_EVIDENCE_FOR_COMMON_SCAN_SYSTEMATIC"
elif n_strong>=1 or n_possible>=3:
    overall="INVESTIGATE_COMMON_SCAN_SYSTEMATIC"
elif n_frame>=2:
    overall="INVESTIGATE_FRAME_LEVEL_SYSTEMATIC"
else:
    overall="NO_DOMINANT_COMMON_MODE_DETECTED"

print("overall conclusion:",overall)
print()

print("SCIENTIFIC INTERPRETATION")
print("A positive scan correlation means survivor residuals move with ordinary-source residuals observed in the same scans.")
print("It does not by itself prove an instrumental artifact.")
print("A survivor must also be compared against the control correlation distribution.")
print("No common-mode correlation does not prove astrophysical variability.")
print("Nearby-source behavior is supplementary because source blending and local effects can differ between objects.")
print("The six survivors are excluded from the control common-mode calculation.")

out1=BASE/"exp021_scan_common_mode_summary.csv"
out2=BASE/"exp021_survivor_scan_residuals.csv"
out3=BASE/"exp021_survivor_frame_residuals.csv"
out4=BASE/"exp021_control_scan_correlations.csv"
out5=BASE/"exp021_survivor_control_comparison.csv"
out6=BASE/"exp021_nearby_source_test.csv"
out7=BASE/"exp021_systematics_flags.csv"
out8=BASE/"exp021_scan_common_mode_summary.json"

scan_stats.to_csv(out1,index=False)
s_scan.to_csv(out2,index=False)
s_frame.to_csv(out3,index=False)
control_corr.to_csv(out4,index=False)
comparison.to_csv(out5,index=False)
nearby.to_csv(out6,index=False)
flags.to_csv(out7,index=False)

result={
    "experiment":"EXP-021",
    "survivors":GIDS,
    "n_survivors":len(GIDS),
    "eligible_control_groups":len(all_groups),
    "control_detections":len(ctrl),
    "usable_common_mode_scans":len(scan_stats),
    "control_correlations":len(control_corr),
    "strong_shared_scan_signatures":n_strong,
    "possible_shared_scan_signatures":n_possible,
    "possible_frame_signatures":n_frame,
    "overall_conclusion":overall,
    "scientific_status":"SYSTEMATICS_INVESTIGATION"
}

with open(out8,"w") as f:
    json.dump(result,f,indent=2)

print()
print("SAVED")
for x in [out1,out2,out3,out4,out5,out6,out7,out8]:
    print(x)

EXP-021 — SCAN/FRAME COMMON-MODE SYSTEMATICS TEST
survivors: [233, 234, 1974, 1976, 1978, 1979]
detections: 7275
source groups: 175

CONTROL GROUPS ELIGIBLE: 25

CONTROL DETECTIONS: 596
CONTROL GROUPS: 25

SCAN COMMON-MODE REFERENCE
usable scans: 25
median control groups per scan: 24.0
minimum control groups: 7

SURVIVOR SCAN CORRELATIONS
 group_id  n_valid_detections  n_scan_matched  scan_coverage  scan_residual_corr  scan_residual_spearman  same_scan_direction  median_survivor_abs_resid  median_scan_common_abs_resid  n_frame_matched  frame_coverage  frame_residual_corr  same_frame_direction
      233                  22              22            1.0            0.538808                0.490961             0.727273                   0.035603                      0.011917               22             1.0             0.564674              0.590909
      234                  24              24            1.0            0.575584                0.396867             0.541667                

In [29]:
# ============================================================
# EXP-022 — LOCAL SPATIAL + SCAN/FRAME SYSTEMATICS
# ============================================================
import os,json,warnings,numpy as np,pandas as pd
from scipy.stats import pearsonr,spearmanr
warnings.filterwarnings("ignore")

R="astronomy_exp005/results"
D=pd.read_csv(f"{R}/exp005_region_detections.csv")
G=pd.read_csv(f"{R}/exp005_all_source_groups.csv")
SURV=[233,234,1974,1976,1978,1979]
RADII=[5,10,15,30]

# numeric cleanup
for c in ["ra","dec","mjd","w1mpro","w1sigmpro","w1snr","scan_id",
          "frame_num","good_quality","group_id"]:
    D[c]=pd.to_numeric(D[c],errors="coerce")
for c in ["group_id","ra","dec","n_observations","w1_weighted_mean",
          "w1_std","w1_median","w1_range","w1_reduced_chi2",
          "median_w1_snr","good_quality_fraction"]:
    G[c]=pd.to_numeric(G[c],errors="coerce")

D=D.dropna(subset=["ra","dec","mjd","w1mpro","w1sigmpro",
                   "scan_id","frame_num","group_id"]).copy()
G=G.drop_duplicates("group_id").copy()
D["group_id"]=D.group_id.astype(int)
G["group_id"]=G.group_id.astype(int)

# eligible controls: exclude six survivors
C=G[
    (~G.group_id.isin(SURV))&
    (G.n_observations>=10)&
    (G.median_w1_snr>=5)&
    (G.good_quality_fraction>=.90)
].copy()

# residuals
gm=G.set_index("group_id").w1_weighted_mean
D["w1_resid"]=D.w1mpro-D.group_id.map(gm)
D["w1_z"]=D.w1_resid/D.w1sigmpro.replace(0,np.nan)
D=D.replace([np.inf,-np.inf],np.nan).dropna(
    subset=["w1_resid","w1_z"]
)

gd=G.set_index("group_id")
dd={int(k):v for k,v in D.groupby("group_id")}

def sep(ra1,dec1,ra2,dec2):
    ra1,ra2=np.radians(ra1),np.radians(ra2)
    d1,d2=np.radians(dec1),np.radians(dec2)
    x=np.sin((d2-d1)/2)**2+np.cos(d1)*np.cos(d2)*np.sin((ra2-ra1)/2)**2
    return np.degrees(2*np.arcsin(np.sqrt(np.clip(x,0,1))))*3600

def cr(a,b):
    x=pd.DataFrame({"a":a,"b":b}).dropna()
    if len(x)<6 or x.a.std()==0 or x.b.std()==0:
        return np.nan,np.nan
    return pearsonr(x.a,x.b)[0],spearmanr(x.a,x.b)[0]

pairs=[]
local=[]
frames=[]

for sid in SURV:
    s=gd.loc[sid]
    sd=dd.get(sid,pd.DataFrame())

    if sd.empty:
        continue

    C2=C.copy()
    C2["sep_arcsec"]=sep(s.ra,s.dec,C2.ra,C2.dec)

    for rad in RADII:
        near=C2[C2.sep_arcsec<=rad].copy()

        local.append({
            "survivor":sid,
            "radius_arcsec":rad,
            "n_local_controls":len(near)
        })

        for _,c in near.iterrows():
            cid=int(c.group_id)
            cd=dd.get(cid,pd.DataFrame())
            if cd.empty:
                continue

            # same-scan residual comparison
            a=sd[["scan_id","frame_num","w1_resid","w1_z"]]
            b=cd[["scan_id","frame_num","w1_resid","w1_z"]]

            m=a.merge(b,on="scan_id",suffixes=("_s","_c"))
            r,rs=cr(m.w1_resid_s,m.w1_resid_c)

            same=(np.sign(m.w1_resid_s)==np.sign(
                m.w1_resid_c)).mean() if len(m) else np.nan

            pairs.append({
                "survivor":sid,
                "control":cid,
                "radius_arcsec":c.sep_arcsec,
                "common_scan_n":len(m),
                "pearson_r":r,
                "spearman_r":rs,
                "same_direction":same,
                "survivor_chi2":s.w1_reduced_chi2,
                "control_chi2":c.w1_reduced_chi2,
                "survivor_snr":s.median_w1_snr,
                "control_snr":c.median_w1_snr,
                "survivor_mag":s.w1_median,
                "control_mag":c.w1_median
            })

            # same-frame comparison
            mf=a.merge(b,on="frame_num",suffixes=("_s","_c"))
            rf,rsf=cr(mf.w1_resid_s,mf.w1_resid_c)

            frames.append({
                "survivor":sid,
                "control":cid,
                "radius_arcsec":c.sep_arcsec,
                "common_frame_n":len(mf),
                "pearson_r":rf,
                "spearman_r":rsf
            })

P=pd.DataFrame(pairs,columns=[
    "survivor","control","radius_arcsec","common_scan_n",
    "pearson_r","spearman_r","same_direction",
    "survivor_chi2","control_chi2","survivor_snr",
    "control_snr","survivor_mag","control_mag"
])

L=pd.DataFrame(local,columns=[
    "survivor","radius_arcsec","n_local_controls"
])

F=pd.DataFrame(frames,columns=[
    "survivor","control","radius_arcsec",
    "common_frame_n","pearson_r","spearman_r"
])

# ------------------------------------------------------------
# LOCAL VERDICTS — use 30" local population
# ------------------------------------------------------------
flags=[]

for sid in SURV:
    lc=L[(L["survivor"]==sid)&(L["radius_arcsec"]==30)]
    pp=P[(P["survivor"]==sid)&(P["radius_arcsec"]==30)]
    pp=pp[pp["common_scan_n"]>=6]

    nctrl=int(lc["n_local_controls"].iloc[0]) if len(lc) else 0
    vals=pp["pearson_r"].dropna()

    if nctrl<3:
        verdict="INSUFFICIENT_LOCAL_CONTROLS"
        mx=np.nan
        p95=np.nan
    elif len(vals)<3:
        verdict="INSUFFICIENT_LOCAL_CORRELATIONS"
        mx=vals.max() if len(vals) else np.nan
        p95=np.nan
    else:
        mx=vals.max()
        p95=np.percentile(vals,95)

        # Strong local signature if multiple controls show
        # substantial correlation OR the strongest correlation
        # is exceptionally high relative to the local population.
        nstrong=int((vals>=.50).sum())

        if nstrong>=2 or mx>=.75:
            verdict="POSSIBLE_LOCAL_SYSTEMATIC_SIGNATURE"
        else:
            verdict="NO_STRONG_LOCAL_SYSTEMATIC_SIGNATURE"

    flags.append({
        "survivor":sid,
        "verdict":verdict,
        "n_local_controls":nctrl,
        "n_common_scan_correlations":len(vals),
        "max_local_pearson":mx,
        "local_corr_p95":p95,
        "n_corr_ge_0.50":int((vals>=.50).sum()) if len(vals) else 0
    })

FLAGS=pd.DataFrame(flags)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
L.to_csv(f"{R}/exp022_local_control_summary.csv",index=False)
P.to_csv(f"{R}/exp022_local_control_pairs.csv",index=False)
F.to_csv(f"{R}/exp022_local_frame_test.csv",index=False)
FLAGS.to_csv(f"{R}/exp022_local_systematics_flags.csv",index=False)

summary={
    "experiment":"EXP-022",
    "purpose":"Local spatial + same-scan + same-frame systematics test",
    "survivors":SURV,
    "eligible_controls":int(len(C)),
    "radii_arcsec":RADII,
    "results":FLAGS.to_dict(orient="records"),
    "interpretation_rule":
        "Local correlation is evidence for possible shared systematics, "
        "not proof of instrumental origin. Absence of correlation does "
        "not prove astrophysical variability."
}

with open(f"{R}/exp022_summary.json","w") as f:
    json.dump(summary,f,indent=2)

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------
print("="*72)
print("EXP-022 COMPLETE")
print("="*72)
print(f"Eligible controls : {len(C)}")
print(f"Pair tests        : {len(P)}")
print(f"Frame tests       : {len(F)}")
print("\nLOCAL CONTROL AVAILABILITY:")
print(L.to_string(index=False))

print("\nFINAL FLAGS:")
print(FLAGS.to_string(index=False))

print("\nSaved:")
for x in [
    "exp022_local_control_summary.csv",
    "exp022_local_control_pairs.csv",
    "exp022_local_frame_test.csv",
    "exp022_local_systematics_flags.csv",
    "exp022_summary.json"
]:
    print("✓",x)

EXP-022 COMPLETE
Eligible controls : 25
Pair tests        : 0
Frame tests       : 0

LOCAL CONTROL AVAILABILITY:
Empty DataFrame
Columns: [survivor, radius_arcsec, n_local_controls]
Index: []

FINAL FLAGS:
 survivor                     verdict  n_local_controls  n_common_scan_correlations  max_local_pearson  local_corr_p95  n_corr_ge_0.50
      233 INSUFFICIENT_LOCAL_CONTROLS                 0                           0                NaN             NaN               0
      234 INSUFFICIENT_LOCAL_CONTROLS                 0                           0                NaN             NaN               0
     1974 INSUFFICIENT_LOCAL_CONTROLS                 0                           0                NaN             NaN               0
     1976 INSUFFICIENT_LOCAL_CONTROLS                 0                           0                NaN             NaN               0
     1978 INSUFFICIENT_LOCAL_CONTROLS                 0                           0                NaN             NaN 

In [30]:
# ============================================================
# EXP-022A — LOCAL POPULATION DENSITY / CONTROL AUDIT
# ============================================================
import os,json,numpy as np,pandas as pd

R="astronomy_exp005/results"
D=pd.read_csv(f"{R}/exp005_region_detections.csv")
G=pd.read_csv(f"{R}/exp005_all_source_groups.csv")
SURV=[233,234,1974,1976,1978,1979]
RADII=[5,10,15,30,60,120]

for c in ["ra","dec","group_id"]:
    G[c]=pd.to_numeric(G[c],errors="coerce")
for c in ["n_observations","median_w1_snr","good_quality_fraction",
          "w1_median","w1_reduced_chi2","w1_std","w1_range"]:
    G[c]=pd.to_numeric(G[c],errors="coerce")
G=G.dropna(subset=["group_id","ra","dec"]).drop_duplicates("group_id").copy()
G["group_id"]=G.group_id.astype(int)

def sep(ra1,dec1,ra2,dec2):
    r1,r2=np.radians(ra1),np.radians(ra2)
    d1,d2=np.radians(dec1),np.radians(dec2)
    x=np.sin((d2-d1)/2)**2+np.cos(d1)*np.cos(d2)*np.sin((r2-r1)/2)**2
    return np.degrees(2*np.arcsin(np.sqrt(np.clip(x,0,1))))*3600

rows=[]
nearest=[]

for sid in SURV:
    s=G[G.group_id==sid].iloc[0]
    X=G[G.group_id!=sid].copy()
    X["sep_arcsec"]=sep(s.ra,s.dec,X.ra,X.dec)

    for rad in RADII:
        x=X[X.sep_arcsec<=rad]
        rows.append({
            "survivor":sid,
            "radius_arcsec":rad,
            "all_nearby":len(x),
            "nobs_ge10":int((x.n_observations>=10).sum()),
            "snr_ge5":int((x.median_w1_snr>=5).sum()),
            "quality_ge90":int((x.good_quality_fraction>=.90).sum()),
            "nobs10_snr5":int(((x.n_observations>=10)&
                               (x.median_w1_snr>=5)).sum()),
            "nobs10_quality90":int(((x.n_observations>=10)&
                                    (x.good_quality_fraction>=.90)).sum()),
            "snr5_quality90":int(((x.median_w1_snr>=5)&
                                  (x.good_quality_fraction>=.90)).sum()),
            "fully_eligible":int(((x.n_observations>=10)&
                                  (x.median_w1_snr>=5)&
                                  (x.good_quality_fraction>=.90)).sum())
        })

    # nearest 10 groups regardless of quality
    nearest.append(
        X.sort_values("sep_arcsec").head(10)[[
            "group_id","sep_arcsec","n_observations",
            "median_w1_snr","good_quality_fraction",
            "w1_median","w1_reduced_chi2","w1_std","w1_range"
        ]].assign(survivor=sid)
    )

A=pd.DataFrame(rows)
N=pd.concat(nearest,ignore_index=True)

A.to_csv(f"{R}/exp022A_local_density_audit.csv",index=False)
N.to_csv(f"{R}/exp022A_nearest_sources.csv",index=False)

summary={
    "experiment":"EXP-022A",
    "purpose":"Audit local source density and availability of valid controls",
    "survivors":SURV,
    "radii_arcsec":RADII,
    "total_groups":int(len(G)),
    "results":A.to_dict(orient="records"),
    "nearest_sources":N.to_dict(orient="records")
}

with open(f"{R}/exp022A_summary.json","w") as f:
    json.dump(summary,f,indent=2)

print("="*72)
print("EXP-022A COMPLETE")
print("="*72)
print(f"Total source groups: {len(G)}")

print("\nLOCAL POPULATION COUNTS:")
print(A.to_string(index=False))

print("\nNEAREST SOURCES:")
print(N.to_string(index=False))

print("\nSaved:")
for f in [
    "exp022A_local_density_audit.csv",
    "exp022A_nearest_sources.csv",
    "exp022A_summary.json"
]:
    print("✓",f)

EXP-022A COMPLETE
Total source groups: 175

LOCAL POPULATION COUNTS:
 survivor  radius_arcsec  all_nearby  nobs_ge10  snr_ge5  quality_ge90  nobs10_snr5  nobs10_quality90  snr5_quality90  fully_eligible
      233              5           0          0        0             0            0                 0               0               0
      233             10           0          0        0             0            0                 0               0               0
      233             15           0          0        0             0            0                 0               0               0
      233             30           1          1        1             0            1                 0               0               0
      233             60           4          4        3             0            3                 0               0               0
      233            120          15         15        9             3            9                 3               3          

In [31]:
# ============================================================
# EXP-022B — MATCHED LOCAL CONTROL RESIDUAL TEST
# ============================================================
import os,json,numpy as np,pandas as pd

R="astronomy_exp005/results"
D=pd.read_csv(f"{R}/exp005_region_detections.csv")
G=pd.read_csv(f"{R}/exp005_all_source_groups.csv")

SURV=[233,234,1974,1976,1978,1979]
RADIUS=120
MIN_MATCH=3

# ---------- helpers ----------
def sep(ra1,dec1,ra2,dec2):
    r1,r2=np.radians(ra1),np.radians(ra2)
    d1,d2=np.radians(dec1),np.radians(dec2)
    x=np.sin((d2-d1)/2)**2+np.cos(d1)*np.cos(d2)*np.sin((r2-r1)/2)**2
    return np.degrees(2*np.arcsin(np.sqrt(np.clip(x,0,1))))*3600

for c in ["ra","dec","group_id"]:
    G[c]=pd.to_numeric(G[c],errors="coerce")
for c in ["n_observations","median_w1_snr","good_quality_fraction",
          "w1_median","w1_reduced_chi2"]:
    G[c]=pd.to_numeric(G[c],errors="coerce")
G=G.dropna(subset=["group_id","ra","dec"]).drop_duplicates("group_id").copy()
G["group_id"]=G.group_id.astype(int)

# ---------- residual builder ----------
def residuals(gid):
    x=D[D.group_id==gid].copy()
    if len(x)<10 or "w1sigmpro" not in x:
        return pd.DataFrame()
    x["err"]=pd.to_numeric(x.w1sigmpro,errors="coerce")
    x["mag"]=pd.to_numeric(x.w1mpro,errors="coerce")
    x=x[(x.err>0)&np.isfinite(x.mag)].copy()
    if len(x)<10:return pd.DataFrame()
    med=np.median(x.mag)
    x["resid"]=x.mag-med
    return x

# ---------- matched local controls ----------
pairs=[]
tests=[]
availability=[]

for sid in SURV:
    s=G[G.group_id==sid].iloc[0]
    X=G[(G.group_id!=sid)&(~G.group_id.isin(SURV))].copy()
    X["sep_arcsec"]=sep(s.ra,s.dec,X.ra,X.dec)
    X=X[X.sep_arcsec<=RADIUS].copy()

    # Match on observational quality without redefining eligibility.
    # Prefer high-quality controls, then comparable SNR and Nobs.
    X["score"]=(abs(X.median_w1_snr-s.median_w1_snr)/
                max(s.median_w1_snr,1))
    X["score"]+=0.5*(abs(X.n_observations-s.n_observations)/
                     max(s.n_observations,1))
    X["score"]+=0.5*abs(X.good_quality_fraction-
                        s.good_quality_fraction)
    X["hq"]=(X.good_quality_fraction>=.90)&(X.n_observations>=10)&(X.median_w1_snr>=5)

    H=X[X.hq].sort_values(["score","sep_arcsec"]).head(10)

    availability.append({
        "survivor":sid,
        "nearby_groups_120":len(X),
        "matched_hq_controls":len(H)
    })

    for _,c in H.iterrows():
        pairs.append({
            "survivor":sid,
            "control":int(c.group_id),
            "separation_arcsec":c.sep_arcsec,
            "match_score":c.score,
            "control_nobs":c.n_observations,
            "control_snr":c.median_w1_snr,
            "control_quality":c.good_quality_fraction
        })

    sr=residuals(sid)
    if sr.empty or len(H)<MIN_MATCH:
        continue

    # Same-scan residual comparison
    for _,c in H.iterrows():
        cr=residuals(int(c.group_id))
        if cr.empty: continue

        z=sr.merge(
            cr[["scan_id","frame_num","resid"]],
            on=["scan_id","frame_num"],
            suffixes=("_surv","_ctrl")
        )

        if len(z)>=5:
            corr=z.resid_surv.corr(z.resid_ctrl)
            same=np.mean(np.sign(z.resid_surv)==np.sign(z.resid_ctrl))
            tests.append({
                "survivor":sid,
                "control":int(c.group_id),
                "separation_arcsec":c.sep_arcsec,
                "n_common":len(z),
                "pearson":corr,
                "same_direction":same,
                "survivor_abs_resid_median":np.median(abs(z.resid_surv)),
                "control_abs_resid_median":np.median(abs(z.resid_ctrl))
            })

A=pd.DataFrame(availability)
P=pd.DataFrame(pairs,columns=[
    "survivor","control","separation_arcsec","match_score",
    "control_nobs","control_snr","control_quality"
])
T=pd.DataFrame(tests,columns=[
    "survivor","control","separation_arcsec","n_common",
    "pearson","same_direction",
    "survivor_abs_resid_median","control_abs_resid_median"
])

# ---------- aggregate survivor-level result ----------
out=[]
for sid in SURV:
    t=T[T.survivor==sid]
    if len(t)==0:
        verdict="INSUFFICIENT_MATCHED_TESTS"
        out.append([sid,verdict,0,np.nan,np.nan,np.nan])
        continue

    vals=t.pearson.dropna()
    p95=np.nanpercentile(vals,95) if len(vals) else np.nan
    mx=np.nanmax(vals) if len(vals) else np.nan
    strong=int((vals>=.50).sum())

    if strong>=2 and mx>=.50:
        verdict="POSSIBLE_LOCAL_SHARED_SIGNATURE"
    else:
        verdict="NO_STRONG_LOCAL_SHARED_SIGNATURE"

    out.append([sid,verdict,len(t),mx,p95,strong])

F=pd.DataFrame(out,columns=[
    "survivor","verdict","n_control_tests",
    "max_local_pearson","local_corr_p95","n_corr_ge_0.50"
])

# ---------- save ----------
A.to_csv(f"{R}/exp022B_control_availability.csv",index=False)
P.to_csv(f"{R}/exp022B_matched_controls.csv",index=False)
T.to_csv(f"{R}/exp022B_local_residual_tests.csv",index=False)
F.to_csv(f"{R}/exp022B_systematics_flags.csv",index=False)

summary={
    "experiment":"EXP-022B",
    "purpose":"Matched local same-scan/frame residual systematics test",
    "radius_arcsec":RADIUS,
    "minimum_matched_controls":MIN_MATCH,
    "survivors":SURV,
    "availability":A.to_dict(orient="records"),
    "systematics_flags":F.to_dict(orient="records"),
    "tests_performed":int(len(T))
}
with open(f"{R}/exp022B_summary.json","w") as f:
    json.dump(summary,f,indent=2)

print("="*72)
print("EXP-022B COMPLETE")
print("="*72)
print("CONTROL AVAILABILITY:")
print(A.to_string(index=False))

print("\nMATCHED CONTROLS:")
print(P.to_string(index=False))

print("\nLOCAL RESIDUAL TESTS:")
print(T.to_string(index=False))

print("\nFINAL FLAGS:")
print(F.to_string(index=False))

print("\nSaved:")
for f in [
    "exp022B_control_availability.csv",
    "exp022B_matched_controls.csv",
    "exp022B_local_residual_tests.csv",
    "exp022B_systematics_flags.csv",
    "exp022B_summary.json"
]:
    print("✓",f)

EXP-022B COMPLETE
CONTROL AVAILABILITY:
 survivor  nearby_groups_120  matched_hq_controls
      233                 14                    2
      234                 12                    1
     1974                 16                    4
     1976                 20                    5
     1978                 15                    3
     1979                 17                    3

MATCHED CONTROLS:
 survivor  control  separation_arcsec  match_score  control_nobs  control_snr  control_quality
      233      231          88.827446     0.466056            23        50.30         1.000000
      233       28          97.929452     0.557878            23        16.20         1.000000
      234     1277         106.795545     0.004498            24        33.50         1.000000
     1974     1975          28.037491     0.137778            26        19.85         1.000000
     1974     1977          37.825361     0.273333            28        17.70         1.000000
     1974       26   

In [32]:
# ============================================================
# EXP-023 — SCAN/FRAME OBSERVING-CONDITION SYSTEMATICS
# ============================================================

import os
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATHS / CONSTANTS
# ------------------------------------------------------------

R = "astronomy_exp005/results"

SURVIVORS = [233, 234, 1974, 1976, 1978, 1979]

MIN_NOBS = 10
MIN_SNR = 5
MIN_QUALITY = 0.90

MIN_CONTROL_SOURCES_PER_SCAN = 5
MIN_CONTROL_SOURCES_PER_FRAME = 5

CORR_THRESHOLD = 0.50
GT_CONTROL_THRESHOLD = 0.50


# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

D = pd.read_csv(
    f"{R}/exp005_region_detections.csv"
)

G = pd.read_csv(
    f"{R}/exp005_all_source_groups.csv"
)


# ------------------------------------------------------------
# CLEAN DETECTION DATA
# ------------------------------------------------------------

for c in [
    "group_id",
    "scan_id",
    "frame_num",
    "w1mpro",
    "w1sigmpro"
]:
    D[c] = pd.to_numeric(D[c], errors="coerce")

D = D.dropna(
    subset=[
        "group_id",
        "w1mpro",
        "w1sigmpro"
    ]
).copy()

D = D[D["w1sigmpro"] > 0].copy()

D["group_id"] = D["group_id"].astype(int)


# ------------------------------------------------------------
# W1 RESIDUALS
# Residual = observation magnitude - source median magnitude
# ------------------------------------------------------------

D["resid"] = (
    D["w1mpro"]
    -
    D.groupby("group_id")["w1mpro"].transform("median")
)


# ------------------------------------------------------------
# CLEAN SOURCE-GROUP DATA
# ------------------------------------------------------------

for c in [
    "group_id",
    "n_observations",
    "median_w1_snr",
    "good_quality_fraction"
]:
    G[c] = pd.to_numeric(
        G[c],
        errors="coerce"
    )

G = (
    G
    .dropna(
        subset=[
            "group_id",
            "n_observations",
            "median_w1_snr",
            "good_quality_fraction"
        ]
    )
    .drop_duplicates("group_id")
    .copy()
)

G["group_id"] = G["group_id"].astype(int)


# ------------------------------------------------------------
# DEFINE STRICT CONTROL POPULATION
#
# Controls must:
#   n_observations >= 10
#   median W1 SNR >= 5
#   good-quality fraction >= 0.90
#
# All six survivors are excluded.
# ------------------------------------------------------------

C = G[
    (~G["group_id"].isin(SURVIVORS))
    &
    (G["n_observations"] >= MIN_NOBS)
    &
    (G["median_w1_snr"] >= MIN_SNR)
    &
    (G["good_quality_fraction"] >= MIN_QUALITY)
].copy()


# ------------------------------------------------------------
# CONTROL DETECTIONS
# ------------------------------------------------------------

CD = D[
    D["group_id"].isin(
        C["group_id"]
    )
].copy()


# ------------------------------------------------------------
# SAME-SCAN CONTROL POPULATION
#
# For every scan, calculate the median absolute W1 residual
# across eligible control sources.
#
# Require at least 5 distinct control sources.
# ------------------------------------------------------------

SS = (
    CD
    .groupby("scan_id")
    .agg(
        n_control_sources=(
            "group_id",
            "nunique"
        ),

        control_median_abs_resid=(
            "resid",
            lambda x: np.median(np.abs(x))
        )
    )
    .reset_index()
)

SS = SS[
    SS["n_control_sources"]
    >= MIN_CONTROL_SOURCES_PER_SCAN
].copy()


# ------------------------------------------------------------
# SAME-FRAME CONTROL POPULATION
#
# Same logic, but grouped by frame number.
# ------------------------------------------------------------

FF = (
    CD
    .groupby("frame_num")
    .agg(
        n_control_sources=(
            "group_id",
            "nunique"
        ),

        control_median_abs_resid=(
            "resid",
            lambda x: np.median(np.abs(x))
        )
    )
    .reset_index()
)

FF = FF[
    FF["n_control_sources"]
    >= MIN_CONTROL_SOURCES_PER_FRAME
].copy()


# ------------------------------------------------------------
# SURVIVOR TESTS
# ------------------------------------------------------------

rows = []

for sid in SURVIVORS:

    X = D[
        D["group_id"] == sid
    ].copy()


    # ========================================================
    # SCAN-LEVEL TEST
    # ========================================================

    A = X.merge(
        SS,
        on="scan_id",
        how="inner"
    )

    if len(A) >= 5:

        scan_corr = (
            A["resid"]
            .abs()
            .corr(
                A["control_median_abs_resid"]
            )
        )

        scan_gt_control_fraction = (
            A["resid"].abs()
            >
            A["control_median_abs_resid"]
        ).mean()

    else:

        scan_corr = np.nan
        scan_gt_control_fraction = np.nan


    # ========================================================
    # FRAME-LEVEL TEST
    # ========================================================

    B = X.merge(
        FF,
        on="frame_num",
        how="inner"
    )

    if len(B) >= 5:

        frame_corr = (
            B["resid"]
            .abs()
            .corr(
                B["control_median_abs_resid"]
            )
        )

        frame_gt_control_fraction = (
            B["resid"].abs()
            >
            B["control_median_abs_resid"]
        ).mean()

    else:

        frame_corr = np.nan
        frame_gt_control_fraction = np.nan


    # ========================================================
    # VERDICT
    # ========================================================

    if len(A) < 5:

        verdict = (
            "INSUFFICIENT_SCAN_CONTROLS"
        )

    elif (
        np.isfinite(scan_corr)
        and
        scan_corr >= CORR_THRESHOLD
        and
        scan_gt_control_fraction >= GT_CONTROL_THRESHOLD
    ):

        verdict = (
            "POSSIBLE_SHARED_OBSERVING_SYSTEMATIC"
        )

    else:

        verdict = (
            "NO_DOMINANT_SHARED_OBSERVING_SYSTEMATIC"
        )


    # ========================================================
    # SAVE RESULT
    # ========================================================

    rows.append({

        "survivor": sid,

        "scan_tests": len(A),

        "frame_tests": len(B),

        "scan_corr": scan_corr,

        "frame_corr": frame_corr,

        "scan_gt_control_fraction":
            scan_gt_control_fraction,

        "frame_gt_control_fraction":
            frame_gt_control_fraction,

        "verdict": verdict
    })


# ------------------------------------------------------------
# CREATE OUTPUT DATAFRAME
# ------------------------------------------------------------

OUT = pd.DataFrame(
    rows,
    columns=[
        "survivor",
        "scan_tests",
        "frame_tests",
        "scan_corr",
        "frame_corr",
        "scan_gt_control_fraction",
        "frame_gt_control_fraction",
        "verdict"
    ]
)


# ------------------------------------------------------------
# SAVE CSV RESULTS
# ------------------------------------------------------------

OUT.to_csv(
    f"{R}/exp023_systematics_summary.csv",
    index=False
)

SS.to_csv(
    f"{R}/exp023_control_scan_stats.csv",
    index=False
)

FF.to_csv(
    f"{R}/exp023_control_frame_stats.csv",
    index=False
)


# ------------------------------------------------------------
# SUMMARY JSON
# ------------------------------------------------------------

summary = {

    "experiment":
        "EXP-023",

    "purpose":
        "Test whether survivor W1 residual magnitude tracks "
        "control-source residual levels under the same "
        "scan/frame observing conditions.",

    "survivors":
        SURVIVORS,

    "eligible_controls":
        int(len(C)),

    "usable_scans":
        int(len(SS)),

    "usable_frames":
        int(len(FF)),

    "minimum_control_sources_per_scan":
        MIN_CONTROL_SOURCES_PER_SCAN,

    "minimum_control_sources_per_frame":
        MIN_CONTROL_SOURCES_PER_FRAME,

    "correlation_threshold":
        CORR_THRESHOLD,

    "greater_than_control_threshold":
        GT_CONTROL_THRESHOLD,

    "results":
        OUT.to_dict(orient="records")
}


with open(
    f"{R}/exp023_summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ------------------------------------------------------------
# PRINT RESULTS
# ------------------------------------------------------------

print("=" * 72)
print("EXP-023 COMPLETE")
print("=" * 72)

print(
    f"Eligible controls: {len(C)}"
)

print(
    f"Usable scans: {len(SS)}"
)

print(
    f"Usable frames: {len(FF)}"
)

print("\nRESULTS:")
print(
    OUT.to_string(index=False)
)

print("\nSaved:")

for filename in [
    "exp023_systematics_summary.csv",
    "exp023_control_scan_stats.csv",
    "exp023_control_frame_stats.csv",
    "exp023_summary.json"
]:

    print(
        f"✓ {filename}"
    )

print("=" * 72)

EXP-023 COMPLETE
Eligible controls: 25
Usable scans: 0
Usable frames: 9

RESULTS:
 survivor  scan_tests  frame_tests  scan_corr  frame_corr  scan_gt_control_fraction  frame_gt_control_fraction                    verdict
      233           0           22        NaN    0.551445                       NaN                   0.272727 INSUFFICIENT_SCAN_CONTROLS
      234           0           24        NaN    0.295266                       NaN                   0.291667 INSUFFICIENT_SCAN_CONTROLS
     1974           0           25        NaN   -0.187008                       NaN                   0.640000 INSUFFICIENT_SCAN_CONTROLS
     1976           0           25        NaN    0.192328                       NaN                   0.720000 INSUFFICIENT_SCAN_CONTROLS
     1978           0           27        NaN   -0.085378                       NaN                   0.666667 INSUFFICIENT_SCAN_CONTROLS
     1979           0           24        NaN    0.433414                       NaN       

In [33]:
# ============================================================
# EXP-024 — ASTROMETRIC / EXTRACTION-GEOMETRY SYSTEMATICS
# ============================================================

import os
import json
import numpy as np
import pandas as pd

R = "astronomy_exp005/results"
SURV = [233, 234, 1974, 1976, 1978, 1979]

D = pd.read_csv(f"{R}/exp005_region_detections.csv")
G = pd.read_csv(f"{R}/exp005_all_source_groups.csv")

for c in [
    "group_id","mjd","w1mpro","w1sigmpro",
    "candidate1_sep_arcsec"
]:
    if c in D.columns:
        D[c] = pd.to_numeric(D[c], errors="coerce")

D = D.dropna(
    subset=["group_id","w1mpro","w1sigmpro"]
).copy()

D = D[D.w1sigmpro > 0].copy()
D["group_id"] = D.group_id.astype(int)

# Residual relative to each source median
D["w1_resid"] = (
    D.w1mpro
    - D.groupby("group_id").w1mpro.transform("median")
)

# Available astrometric displacement proxy
if "candidate1_sep_arcsec" in D.columns:
    D["position_offset_arcsec"] = D["candidate1_sep_arcsec"]

elif "median_position_offset_arcsec" in G.columns:
    D = D.merge(
        G[["group_id","median_position_offset_arcsec"]],
        on="group_id",
        how="left"
    )
    D["position_offset_arcsec"] = D[
        "median_position_offset_arcsec"
    ]

else:
    raise ValueError(
        "No positional-offset field available."
    )

D = D.dropna(
    subset=["position_offset_arcsec"]
).copy()

rows = []

for sid in SURV:

    X = D[D.group_id == sid].copy()

    if len(X) < 5:
        rows.append({
            "survivor": sid,
            "n": len(X),
            "corr_abs_resid_position": np.nan,
            "spearman_abs_resid_position": np.nan,
            "median_position_offset_arcsec": np.nan,
            "median_abs_resid": np.nan,
            "high_offset_fraction": np.nan,
            "high_offset_abs_resid_median": np.nan,
            "low_offset_abs_resid_median": np.nan,
            "verdict": "INSUFFICIENT_DATA"
        })
        continue

    X["abs_resid"] = X.w1_resid.abs()

    corr = X["abs_resid"].corr(
        X["position_offset_arcsec"]
    )

    spearman = X[
        ["abs_resid","position_offset_arcsec"]
    ].corr(method="spearman").iloc[0,1]

    q75 = X.position_offset_arcsec.quantile(.75)

    low = X[
        X.position_offset_arcsec < q75
    ]

    high = X[
        X.position_offset_arcsec >= q75
    ]

    high_fraction = len(high) / len(X)

    high_med = (
        high.abs_resid.median()
        if len(high) else np.nan
    )

    low_med = (
        low.abs_resid.median()
        if len(low) else np.nan
    )

    if (
        np.isfinite(corr)
        and corr >= .50
        and high_med > low_med
    ):
        verdict = "POSSIBLE_GEOMETRY_SIGNATURE"
    else:
        verdict = "NO_DOMINANT_GEOMETRY_SIGNATURE"

    rows.append({
        "survivor": sid,
        "n": len(X),
        "corr_abs_resid_position": corr,
        "spearman_abs_resid_position": spearman,
        "median_position_offset_arcsec":
            X.position_offset_arcsec.median(),
        "median_abs_resid":
            X.abs_resid.median(),
        "high_offset_fraction":
            high_fraction,
        "high_offset_abs_resid_median":
            high_med,
        "low_offset_abs_resid_median":
            low_med,
        "verdict": verdict
    })

OUT = pd.DataFrame(rows)

OUT.to_csv(
    f"{R}/exp024_geometry_summary.csv",
    index=False
)

D[
    D.group_id.isin(SURV)
][[
    "group_id","mjd","w1mpro",
    "w1_resid","position_offset_arcsec"
]].to_csv(
    f"{R}/exp024_survivor_geometry.csv",
    index=False
)

with open(
    f"{R}/exp024_summary.json","w"
) as f:
    json.dump({
        "experiment": "EXP-024",
        "purpose":
            "Test whether W1 residual amplitude is associated "
            "with astrometric displacement within detections.",
        "survivors": SURV,
        "results": OUT.to_dict(orient="records")
    }, f, indent=2)

print("EXP-024 COMPLETE")
print(OUT.to_string(index=False))
print("\nSaved:")
print("✓ exp024_geometry_summary.csv")
print("✓ exp024_survivor_geometry.csv")
print("✓ exp024_summary.json")

EXP-024 COMPLETE
 survivor  n  corr_abs_resid_position  spearman_abs_resid_position  median_position_offset_arcsec  median_abs_resid  high_offset_fraction  high_offset_abs_resid_median  low_offset_abs_resid_median                        verdict
      233 22                 0.058911                    -0.053642                     323.194295            0.0285              0.272727                        0.0230                       0.0315 NO_DOMINANT_GEOMETRY_SIGNATURE
      234 24                 0.319044                     0.376414                     405.949106            0.0205              0.250000                        0.0310                       0.0100 NO_DOMINANT_GEOMETRY_SIGNATURE
     1974 25                 0.170262                     0.090035                     153.763440            0.0500              0.280000                        0.0540                       0.0445 NO_DOMINANT_GEOMETRY_SIGNATURE
     1976 25                -0.088485                    -0.294287     

In [34]:
# ============================================================
# EXP-025 — MATCHED W1/W2 VARIABILITY CONTRAST
# ============================================================

import os,json,numpy as np,pandas as pd

R="astronomy_exp005/results"
SURV=[233,234,1974,1976,1978,1979]

D=pd.read_csv(f"{R}/exp005_region_detections.csv")
G=pd.read_csv(f"{R}/exp005_all_source_groups.csv")

for c in [
    "group_id","w1mpro","w1sigmpro",
    "w2mpro","w2sigmpro"
]:
    D[c]=pd.to_numeric(D[c],errors="coerce")

for c in [
    "group_id","n_observations",
    "median_w1_snr","good_quality_fraction",
    "w1_reduced_chi2"
]:
    G[c]=pd.to_numeric(G[c],errors="coerce")

G=G.drop_duplicates("group_id").copy()
G["group_id"]=G.group_id.astype(int)

# W2 reduced chi-square from detections
D=D.dropna(
    subset=[
        "group_id","w1mpro","w1sigmpro",
        "w2mpro","w2sigmpro"
    ]
).copy()

D=D[
    (D.w1sigmpro>0)&
    (D.w2sigmpro>0)
].copy()

D["group_id"]=D.group_id.astype(int)

def chi2(x,m,s):
    if len(x)<2:
        return np.nan
    m=np.asarray(m)
    s=np.asarray(s)
    return np.sum(((x-np.mean(x))/s)**2)/(len(x)-1)

W2=D.groupby("group_id").apply(
    lambda x:chi2(
        x.w2mpro,
        x.w2mpro,
        x.w2sigmpro
    ),
    include_groups=False
).rename("w2_reduced_chi2")

# Correct calculation using group mean
W2=D.groupby("group_id").apply(
    lambda x:
        np.sum(
            ((x.w2mpro-x.w2mpro.mean())/
             x.w2sigmpro)**2
        )/(len(x)-1)
        if len(x)>=2 else np.nan,
    include_groups=False
).rename("w2_reduced_chi2")

G=G.merge(
    W2,
    left_on="group_id",
    right_index=True,
    how="left"
)

G["w1_w2_contrast"] = (
    G.w1_reduced_chi2 -
    G.w2_reduced_chi2
)

# Strict ordinary controls
C=G[
    (~G.group_id.isin(SURV))&
    (G.n_observations>=10)&
    (G.median_w1_snr>=5)&
    (G.good_quality_fraction>=.90)&
    G.w2_reduced_chi2.notna()
].copy()

# Match controls using observational properties only
matches=[]
tests=[]

for sid in SURV:

    s=G[G.group_id==sid].iloc[0]

    X=C.copy()

    X["match_distance"] = (
        abs(X.n_observations-s.n_observations)/
        max(s.n_observations,1)
        +
        abs(X.median_w1_snr-s.median_w1_snr)/
        max(s.median_w1_snr,1)
        +
        abs(
            X.good_quality_fraction-
            s.good_quality_fraction
        )
    )

    X=X.sort_values("match_distance").head(5)

    for _,c in X.iterrows():

        tests.append({
            "survivor":sid,
            "control":int(c.group_id),
            "match_distance":c.match_distance,
            "survivor_w1_chi2":s.w1_reduced_chi2,
            "control_w1_chi2":c.w1_reduced_chi2,
            "survivor_w2_chi2":s.w2_reduced_chi2,
            "control_w2_chi2":c.w2_reduced_chi2,
            "survivor_contrast":s.w1_w2_contrast,
            "control_contrast":c.w1_w2_contrast
        })

    if len(X):

        control_contrasts=X.w1_w2_contrast.values

        pct=(
            np.mean(
                control_contrasts <= s.w1_w2_contrast
            )*100
        )

        tests.append({
            "survivor":sid,
            "control":-1,
            "match_distance":np.nan,
            "survivor_w1_chi2":s.w1_reduced_chi2,
            "control_w1_chi2":np.nan,
            "survivor_w2_chi2":s.w2_reduced_chi2,
            "control_w2_chi2":np.nan,
            "survivor_contrast":s.w1_w2_contrast,
            "control_contrast":np.nan,
            "percentile_vs_5_matched":pct
        })

# Survivor summary
S=G[G.group_id.isin(SURV)][[
    "group_id",
    "n_observations",
    "median_w1_snr",
    "good_quality_fraction",
    "w1_reduced_chi2",
    "w2_reduced_chi2",
    "w1_w2_contrast"
]].copy()

rows=[]

for sid in SURV:

    s=S[S.group_id==sid].iloc[0]

    X=C.copy()

    X["d"] = (
        abs(X.n_observations-s.n_observations)/
        max(s.n_observations,1)
        +
        abs(X.median_w1_snr-s.median_w1_snr)/
        max(s.median_w1_snr,1)
        +
        abs(
            X.good_quality_fraction-
            s.good_quality_fraction
        )
    )

    X=X.sort_values("d").head(5)

    vals=X.w1_w2_contrast.dropna()

    pct=(
        np.mean(vals<=s.w1_w2_contrast)*100
        if len(vals) else np.nan
    )

    med=X.w1_w2_contrast.median()

    rows.append({
        "survivor":sid,
        "n_matched_controls":len(vals),
        "survivor_w1_chi2":s.w1_reduced_chi2,
        "survivor_w2_chi2":s.w2_reduced_chi2,
        "survivor_w1_w2_contrast":s.w1_w2_contrast,
        "matched_control_contrast_median":med,
        "percentile_vs_matched_controls":pct,
        "verdict":
            "W1_DOMINANT_CONTRAST" if
            np.isfinite(pct) and pct>=80
            else
            "NO_DOMINANT_W1_W2_CONTRAST"
    })

OUT=pd.DataFrame(rows)
T=pd.DataFrame(tests)

OUT.to_csv(
    f"{R}/exp025_variability_contrast_summary.csv",
    index=False
)

T.to_csv(
    f"{R}/exp025_matched_w1_w2_tests.csv",
    index=False
)

with open(f"{R}/exp025_summary.json","w") as f:
    json.dump({
        "experiment":"EXP-025",
        "purpose":
            "Compare W1-vs-W2 variability contrast of survivors "
            "against matched ordinary controls.",
        "eligible_controls":int(len(C)),
        "results":OUT.to_dict(orient="records")
    },f,indent=2)

print("EXP-025 COMPLETE")
print(OUT.to_string(index=False))
print("\nSaved:")
print("✓ exp025_variability_contrast_summary.csv")
print("✓ exp025_matched_w1_w2_tests.csv")
print("✓ exp025_summary.json")

EXP-025 COMPLETE
 survivor  n_matched_controls  survivor_w1_chi2  survivor_w2_chi2  survivor_w1_w2_contrast  matched_control_contrast_median  percentile_vs_matched_controls              verdict
      233                   5          2.118475          1.469643                 0.648832                         0.532042                            80.0 W1_DOMINANT_CONTRAST
      234                   5          2.493238          1.166835                 1.326403                        -0.018096                           100.0 W1_DOMINANT_CONTRAST
     1974                   5          3.026153          1.215094                 1.811058                        -0.181656                           100.0 W1_DOMINANT_CONTRAST
     1976                   5          2.115925          1.206987                 0.908938                        -0.164345                            80.0 W1_DOMINANT_CONTRAST
     1978                   5          3.939207          1.452327                 2.486880        

In [35]:
# EXP-026 — EPOCH-LEVEL W1/W2 COHERENCE

import os,json,numpy as np,pandas as pd

R="astronomy_exp005/results"
SURV=[233,234,1974,1976,1978,1979]

D=pd.read_csv(f"{R}/exp005_region_detections.csv")

for c in ["group_id","mjd","w1mpro","w1sigmpro","w2mpro","w2sigmpro"]:
    D[c]=pd.to_numeric(D[c],errors="coerce")

D=D.dropna(
    subset=["group_id","mjd","w1mpro","w1sigmpro",
            "w2mpro","w2sigmpro"]
).copy()

D=D[(D.w1sigmpro>0)&(D.w2sigmpro>0)].copy()
D["group_id"]=D.group_id.astype(int)

# Approximate observing epochs from the NEOWISE cadence
D["epoch"]=np.floor(D.mjd/150).astype(int)

rows=[]
epoch_rows=[]

for sid in SURV:

    X=D[D.group_id==sid].copy()

    X["w1_resid"]=X.w1mpro-X.w1mpro.median()
    X["w2_resid"]=X.w2mpro-X.w2mpro.median()

    E=X.groupby("epoch").agg(
        n=("group_id","size"),
        w1_resid=("w1_resid","median"),
        w2_resid=("w2_resid","median"),
        w1_abs=("w1_resid",lambda x:np.median(abs(x))),
        w2_abs=("w2_resid",lambda x:np.median(abs(x)))
    ).reset_index()

    E["survivor"]=sid

    # Require at least 3 epochs for a coherence test
    if len(E)>=3:

        corr=E.w1_resid.corr(E.w2_resid)

        same=(
            np.sign(E.w1_resid)==
            np.sign(E.w2_resid)
        ).mean()

        joint=((E.w1_abs>0)&(E.w2_abs>0)).mean()

        if np.isfinite(corr) and corr>=.50 and same>=.60:
            verdict="POSSIBLE_CROSS_BAND_COHERENCE"
        else:
            verdict="NO_STRONG_CROSS_BAND_COHERENCE"

    else:

        corr=np.nan
        same=np.nan
        joint=np.nan
        verdict="INSUFFICIENT_EPOCHS"

    rows.append({
        "survivor":sid,
        "n_epochs":len(E),
        "w1_w2_epoch_corr":corr,
        "same_direction_fraction":same,
        "nonzero_joint_fraction":joint,
        "verdict":verdict
    })

    epoch_rows.append(E)

OUT=pd.DataFrame(rows)
EP=pd.concat(epoch_rows,ignore_index=True)

OUT.to_csv(
    f"{R}/exp026_epoch_coherence_summary.csv",
    index=False
)

EP.to_csv(
    f"{R}/exp026_epoch_residuals.csv",
    index=False
)

with open(f"{R}/exp026_summary.json","w") as f:
    json.dump({
        "experiment":"EXP-026",
        "purpose":
            "Test epoch-level W1/W2 residual coherence.",
        "epoch_definition":
            "floor(MJD/150), approximate cadence grouping; "
            "not a formal NEOWISE epoch definition.",
        "results":OUT.to_dict(orient="records")
    },f,indent=2)

print("EXP-026 COMPLETE")
print(OUT.to_string(index=False))
print("\nSaved:")
print("✓ exp026_epoch_coherence_summary.csv")
print("✓ exp026_epoch_residuals.csv")
print("✓ exp026_summary.json")

EXP-026 COMPLETE
 survivor  n_epochs  w1_w2_epoch_corr  same_direction_fraction  nonzero_joint_fraction                        verdict
      233         3          0.934566                 1.000000                     1.0  POSSIBLE_CROSS_BAND_COHERENCE
      234         3          0.669111                 1.000000                     1.0  POSSIBLE_CROSS_BAND_COHERENCE
     1974         3         -0.818035                 0.333333                     1.0 NO_STRONG_CROSS_BAND_COHERENCE
     1976         3          0.141314                 0.333333                     1.0 NO_STRONG_CROSS_BAND_COHERENCE
     1978         3          0.825045                 1.000000                     1.0  POSSIBLE_CROSS_BAND_COHERENCE
     1979         3         -0.971613                 0.000000                     1.0 NO_STRONG_CROSS_BAND_COHERENCE

Saved:
✓ exp026_epoch_coherence_summary.csv
✓ exp026_epoch_residuals.csv
✓ exp026_summary.json


In [36]:
# EXP-027 — MATCHED-CONTROL EPOCH COHERENCE BENCHMARK

import os,json,numpy as np,pandas as pd

R="astronomy_exp005/results"
SURV=[233,234,1974,1976,1978,1979]

D=pd.read_csv(f"{R}/exp005_region_detections.csv")
G=pd.read_csv(f"{R}/exp005_all_source_groups.csv")

for c in ["group_id","mjd","w1mpro","w1sigmpro","w2mpro","w2sigmpro"]:
    D[c]=pd.to_numeric(D[c],errors="coerce")

D=D.dropna(subset=["group_id","mjd","w1mpro","w1sigmpro","w2mpro","w2sigmpro"]).copy()
D=D[(D.w1sigmpro>0)&(D.w2sigmpro>0)].copy()
D["group_id"]=D.group_id.astype(int)
D["epoch"]=np.floor(D.mjd/150).astype(int)

def epoch_metrics(sid):
    X=D[D.group_id==sid].copy()
    if len(X)==0:return None

    X["w1_resid"]=X.w1mpro-X.w1mpro.median()
    X["w2_resid"]=X.w2mpro-X.w2mpro.median()

    E=X.groupby("epoch").agg(
        w1=("w1_resid","median"),
        w2=("w2_resid","median")
    ).reset_index()

    if len(E)<3:return None

    corr=E.w1.corr(E.w2)
    same=(np.sign(E.w1)==np.sign(E.w2)).mean()

    return len(E),corr,same

# Strict ordinary controls; variability is NOT used for matching
eligible=G[
    (~G.group_id.isin(SURV))&
    (G.n_observations>=10)&
    (G.median_w1_snr>=5)&
    (G.good_quality_fraction>=.90)
].copy()

results=[]
control_rows=[]

for sid in SURV:

    S=G[G.group_id==sid].iloc[0]

    C=eligible.copy()
    C["match_score"]=(
        abs(C.n_observations-S.n_observations)/S.n_observations+
        abs(C.median_w1_snr-S.median_w1_snr)/S.median_w1_snr+
        abs(C.good_quality_fraction-S.good_quality_fraction)
    )

    C=C.nsmallest(10,"match_score")

    sm=epoch_metrics(sid)

    if sm is None:
        results.append({
            "survivor":sid,
            "n_epochs":np.nan,
            "w1_w2_epoch_corr":np.nan,
            "same_direction_fraction":np.nan,
            "n_matched_controls":0,
            "corr_percentile":np.nan,
            "same_direction_percentile":np.nan,
            "verdict":"INSUFFICIENT_EPOCHS"
        })
        continue

    ne,sc,ss=sm

    for _,r in C.iterrows():
        cm=epoch_metrics(int(r.group_id))
        if cm is not None:
            control_rows.append({
                "survivor":sid,
                "control_group":int(r.group_id),
                "match_score":r.match_score,
                "control_epochs":cm[0],
                "control_corr":cm[1],
                "control_same_direction":cm[2]
            })

    T=pd.DataFrame([
        x for x in control_rows if x["survivor"]==sid
    ])

    if len(T):
        cp=(T.control_corr<=sc).mean()*100
        sp=(T.control_same_direction<=ss).mean()*100
    else:
        cp=sp=np.nan

    if len(T)<5:
        verdict="INSUFFICIENT_CONTROL_TESTS"
    elif cp>=90 and sp>=90:
        verdict="UNUSUAL_CROSS_BAND_COHERENCE"
    else:
        verdict="CONSISTENT_WITH_CONTROL_COHERENCE"

    results.append({
        "survivor":sid,
        "n_epochs":ne,
        "w1_w2_epoch_corr":sc,
        "same_direction_fraction":ss,
        "n_matched_controls":len(T),
        "corr_percentile":cp,
        "same_direction_percentile":sp,
        "verdict":verdict
    })

OUT=pd.DataFrame(results)
CTL=pd.DataFrame(control_rows)

OUT.to_csv(f"{R}/exp027_epoch_coherence_benchmark.csv",index=False)
CTL.to_csv(f"{R}/exp027_matched_control_epoch_metrics.csv",index=False)

with open(f"{R}/exp027_summary.json","w") as f:
    json.dump({
        "experiment":"EXP-027",
        "purpose":"Benchmark survivor epoch-level W1/W2 coherence against observationally matched ordinary controls.",
        "epoch_definition":"floor(MJD/150), same approximate epoch definition as EXP-026.",
        "control_selection":"nobs, median W1 SNR and quality; variability excluded from matching.",
        "results":OUT.to_dict(orient="records")
    },f,indent=2)

print("EXP-027 COMPLETE")
print(OUT.to_string(index=False))
print("\nSaved:")
print("✓ exp027_epoch_coherence_benchmark.csv")
print("✓ exp027_matched_control_epoch_metrics.csv")
print("✓ exp027_summary.json")

EXP-027 COMPLETE
 survivor  n_epochs  w1_w2_epoch_corr  same_direction_fraction  n_matched_controls  corr_percentile  same_direction_percentile                           verdict
      233         3          0.934566                 1.000000                  10            100.0                      100.0      UNUSUAL_CROSS_BAND_COHERENCE
      234         3          0.669111                 1.000000                  10             70.0                      100.0 CONSISTENT_WITH_CONTROL_COHERENCE
     1974         3         -0.818035                 0.333333                  10             30.0                       70.0 CONSISTENT_WITH_CONTROL_COHERENCE
     1976         3          0.141314                 0.333333                  10             70.0                       70.0 CONSISTENT_WITH_CONTROL_COHERENCE
     1978         3          0.825045                 1.000000                  10             90.0                      100.0      UNUSUAL_CROSS_BAND_COHERENCE
     1979        

In [37]:
# EXP-028 — ROBUST PHOTOMETRIC ROBUSTNESS TEST

import os,json,numpy as np,pandas as pd

R="astronomy_exp005/results"
SURV=[233,1978]

D=pd.read_csv(f"{R}/exp005_region_detections.csv")
G=pd.read_csv(f"{R}/exp005_all_source_groups.csv")

for c in ["group_id","w1mpro","w1sigmpro","w2mpro","w2sigmpro"]:
    D[c]=pd.to_numeric(D[c],errors="coerce")

D=D.dropna(subset=["group_id","w1mpro","w1sigmpro","w2mpro","w2sigmpro"]).copy()
D=D[(D.w1sigmpro>0)&(D.w2sigmpro>0)].copy()
D["group_id"]=D.group_id.astype(int)

rows=[]

for sid in SURV:

    X=D[D.group_id==sid].copy()

    w1=X.w1mpro.to_numpy()
    e1=X.w1sigmpro.to_numpy()
    w2=X.w2mpro.to_numpy()
    e2=X.w2sigmpro.to_numpy()

    # Original error-weighted reduced chi-square
    c1=np.sum(((w1-np.average(w1,weights=1/e1**2))/e1)**2)/(len(w1)-1)
    c2=np.sum(((w2-np.average(w2,weights=1/e2**2))/e2)**2)/(len(w2)-1)

    # Robust center: median
    w1_med=np.median(w1)
    w2_med=np.median(w2)

    # Robust scatter: MAD scaled to Gaussian sigma
    mad1=1.4826*np.median(np.abs(w1-w1_med))
    mad2=1.4826*np.median(np.abs(w2-w2_med))

    # Median absolute deviation relative to reported errors
    norm1=np.median(np.abs(w1-w1_med)/e1)
    norm2=np.median(np.abs(w2-w2_med)/e2)

    # Fraction of measurements exceeding 2 reported-error units
    z1=np.abs(w1-w1_med)/e1
    z2=np.abs(w2-w2_med)/e2
    f1=(z1>=2).mean()
    f2=(z2>=2).mean()

    # Trimmed scatter removes the most extreme 10%
    lo1,hi1=np.percentile(w1,[10,90])
    lo2,hi2=np.percentile(w2,[10,90])
    t1=w1[(w1>=lo1)&(w1<=hi1)]
    t2=w2[(w2>=lo2)&(w2<=hi2)]

    trim_std1=np.std(t1,ddof=1)
    trim_std2=np.std(t2,ddof=1)

    rows.append({
        "survivor":sid,
        "n":len(X),
        "original_w1_chi2":c1,
        "original_w2_chi2":c2,
        "original_contrast":c1-c2,
        "robust_w1_mad_sigma":mad1,
        "robust_w2_mad_sigma":mad2,
        "median_abs_w1_error_units":norm1,
        "median_abs_w2_error_units":norm2,
        "w1_fraction_abs_z_ge_2":f1,
        "w2_fraction_abs_z_ge_2":f2,
        "trimmed_w1_std":trim_std1,
        "trimmed_w2_std":trim_std2,
        "robust_w1_dominance":mad1/mad2 if mad2>0 else np.nan,
        "trimmed_w1_dominance":trim_std1/trim_std2 if trim_std2>0 else np.nan
    })

OUT=pd.DataFrame(rows)

# Stability criteria:
# anomaly remains if W1 has excess robust scatter, >=2-sigma fraction,
# and trimmed scatter relative to W2 remains >1.
OUT["robustness_pass"]=(
    (OUT.robust_w1_dominance>1) &
    (OUT.trimmed_w1_dominance>1) &
    (OUT.w1_fraction_abs_z_ge_2>OUT.w2_fraction_abs_z_ge_2)
)

OUT["verdict"]=np.where(
    OUT.robustness_pass,
    "ROBUST_W1_VARIABILITY_PERSISTS",
    "ROBUSTNESS_NOT_CONFIRMED"
)

OUT.to_csv(f"{R}/exp028_photometric_robustness.csv",index=False)

with open(f"{R}/exp028_summary.json","w") as f:
    json.dump({
        "experiment":"EXP-028",
        "purpose":"Test whether leading survivor W1 variability persists under robust photometric statistics.",
        "methods":[
            "error-weighted reduced chi-square",
            "median/MAD robust scatter",
            "median absolute residual in reported-error units",
            "fraction above 2 reported-error units",
            "10-90 percent trimmed scatter"
        ],
        "results":OUT.to_dict(orient="records")
    },f,indent=2)

print("EXP-028 COMPLETE")
print(OUT.to_string(index=False))
print("\nSaved:")
print("✓ exp028_photometric_robustness.csv")
print("✓ exp028_summary.json")

EXP-028 COMPLETE
 survivor  n  original_w1_chi2  original_w2_chi2  original_contrast  robust_w1_mad_sigma  robust_w2_mad_sigma  median_abs_w1_error_units  median_abs_w2_error_units  w1_fraction_abs_z_ge_2  w2_fraction_abs_z_ge_2  trimmed_w1_std  trimmed_w2_std  robust_w1_dominance  trimmed_w1_dominance  robustness_pass                  verdict
      233 22          2.118475          1.447637           0.670838             0.042254             0.083767                   1.000000                   0.650178                0.181818                0.181818        0.030347        0.049383             0.504425              0.614515            False ROBUSTNESS_NOT_CONFIRMED
     1978 28          3.939207          1.291133           2.648073             0.084508             0.211270                   0.775191                   0.614798                0.142857                0.035714        0.058683        0.154711             0.400000              0.379305            False ROBUSTNESS_NOT_CONFIR

In [38]:
# EXP-029 — ROBUST PHOTOMETRIC TEST: REMAINING SURVIVORS

import os,json,numpy as np,pandas as pd

R="astronomy_exp005/results"
SURV=[234,1974,1976,1979]

D=pd.read_csv(f"{R}/exp005_region_detections.csv")

for c in ["group_id","w1mpro","w1sigmpro","w2mpro","w2sigmpro"]:
    D[c]=pd.to_numeric(D[c],errors="coerce")

D=D.dropna(subset=["group_id","w1mpro","w1sigmpro","w2mpro","w2sigmpro"]).copy()
D=D[(D.w1sigmpro>0)&(D.w2sigmpro>0)].copy()
D["group_id"]=D.group_id.astype(int)

rows=[]

for sid in SURV:

    X=D[D.group_id==sid].copy()

    w1=X.w1mpro.to_numpy()
    e1=X.w1sigmpro.to_numpy()
    w2=X.w2mpro.to_numpy()
    e2=X.w2sigmpro.to_numpy()

    c1=np.sum(((w1-np.average(w1,weights=1/e1**2))/e1)**2)/(len(w1)-1)
    c2=np.sum(((w2-np.average(w2,weights=1/e2**2))/e2)**2)/(len(w2)-1)

    m1=np.median(w1)
    m2=np.median(w2)

    mad1=1.4826*np.median(np.abs(w1-m1))
    mad2=1.4826*np.median(np.abs(w2-m2))

    z1=np.abs(w1-m1)/e1
    z2=np.abs(w2-m2)/e2

    norm1=np.median(z1)
    norm2=np.median(z2)

    f1=(z1>=2).mean()
    f2=(z2>=2).mean()

    lo1,hi1=np.percentile(w1,[10,90])
    lo2,hi2=np.percentile(w2,[10,90])

    t1=w1[(w1>=lo1)&(w1<=hi1)]
    t2=w2[(w2>=lo2)&(w2<=hi2)]

    ts1=np.std(t1,ddof=1)
    ts2=np.std(t2,ddof=1)

    rows.append({
        "survivor":sid,
        "n":len(X),
        "original_w1_chi2":c1,
        "original_w2_chi2":c2,
        "original_contrast":c1-c2,
        "robust_w1_mad_sigma":mad1,
        "robust_w2_mad_sigma":mad2,
        "median_abs_w1_error_units":norm1,
        "median_abs_w2_error_units":norm2,
        "w1_fraction_abs_z_ge_2":f1,
        "w2_fraction_abs_z_ge_2":f2,
        "trimmed_w1_std":ts1,
        "trimmed_w2_std":ts2,
        "robust_w1_dominance":mad1/mad2 if mad2>0 else np.nan,
        "trimmed_w1_dominance":ts1/ts2 if ts2>0 else np.nan
    })

OUT=pd.DataFrame(rows)

OUT["robustness_pass"]=(
    (OUT.robust_w1_dominance>1)&
    (OUT.trimmed_w1_dominance>1)&
    (OUT.w1_fraction_abs_z_ge_2>OUT.w2_fraction_abs_z_ge_2)
)

OUT["verdict"]=np.where(
    OUT.robustness_pass,
    "ROBUST_W1_VARIABILITY_PERSISTS",
    "ROBUSTNESS_NOT_CONFIRMED"
)

OUT.to_csv(f"{R}/exp029_photometric_robustness.csv",index=False)

with open(f"{R}/exp029_summary.json","w") as f:
    json.dump({
        "experiment":"EXP-029",
        "purpose":"Apply the EXP-028 robust photometric test to the four remaining survivors.",
        "methods":[
            "error-weighted reduced chi-square",
            "median/MAD robust scatter",
            "median absolute residual in reported-error units",
            "fraction above 2 reported-error units",
            "10-90 percent trimmed scatter"
        ],
        "results":OUT.to_dict(orient="records")
    },f,indent=2)

print("EXP-029 COMPLETE")
print(OUT.to_string(index=False))
print("\nSaved:")
print("✓ exp029_photometric_robustness.csv")
print("✓ exp029_summary.json")

EXP-029 COMPLETE
 survivor  n  original_w1_chi2  original_w2_chi2  original_contrast  robust_w1_mad_sigma  robust_w2_mad_sigma  median_abs_w1_error_units  median_abs_w2_error_units  w1_fraction_abs_z_ge_2  w2_fraction_abs_z_ge_2  trimmed_w1_std  trimmed_w2_std  robust_w1_dominance  trimmed_w1_dominance  robustness_pass                  verdict
      234 24          2.493238          1.161600           1.331638             0.030393             0.070423                   0.545697                   0.629005                0.250000                0.083333        0.020408        0.053360             0.431579              0.382463            False ROBUSTNESS_NOT_CONFIRMED
     1974 25          3.026153          1.166683           1.859470             0.074130             0.085991                   0.870968                   0.518519                0.200000                0.080000        0.047388        0.072531             0.862069              0.653339            False ROBUSTNESS_NOT_CONFIR

In [39]:
# EXP-030 — ERROR-AWARE ROBUST PHOTOMETRIC AUDIT

import os,json,numpy as np,pandas as pd

R="astronomy_exp005/results"
SURV=[233,234,1974,1976,1978,1979]

D=pd.read_csv(f"{R}/exp005_region_detections.csv")

for c in ["group_id","w1mpro","w1sigmpro","w2mpro","w2sigmpro"]:
    D[c]=pd.to_numeric(D[c],errors="coerce")

D=D.dropna(subset=["group_id","w1mpro","w1sigmpro","w2mpro","w2sigmpro"]).copy()
D=D[(D.w1sigmpro>0)&(D.w2sigmpro>0)].copy()
D["group_id"]=D.group_id.astype(int)

rows=[]

for sid in SURV:
    X=D[D.group_id==sid].copy()

    results={}

    for band,m,e in [("w1",X.w1mpro,X.w1sigmpro),
                     ("w2",X.w2mpro,X.w2sigmpro)]:

        m=m.to_numpy()
        e=e.to_numpy()

        mu=np.average(m,weights=1/e**2)
        r=(m-mu)/e
        ar=np.abs(r)

        mad_r=1.4826*np.median(np.abs(r-np.median(r)))
        med_abs_z=np.median(ar)
        frac_z2=np.mean(ar>=2)

        q10,q90=np.percentile(r,[10,90])
        rt=r[(r>=q10)&(r<=q90)]
        trimmed_mean_z2=np.mean(rt**2)

        robust_reduced_z2=np.median(r**2)/0.4549364

        results[f"{band}_median_abs_z"]=med_abs_z
        results[f"{band}_mad_z"]=mad_r
        results[f"{band}_frac_abs_z_ge2"]=frac_z2
        results[f"{band}_trimmed_mean_z2"]=trimmed_mean_z2
        results[f"{band}_robust_reduced_z2"]=robust_reduced_z2

    rows.append({
        "survivor":sid,
        "n":len(X),
        **results
    })

OUT=pd.DataFrame(rows)

OUT["median_abs_z_ratio"]=OUT.w1_median_abs_z/OUT.w2_median_abs_z
OUT["mad_z_ratio"]=OUT.w1_mad_z/OUT.w2_mad_z
OUT["trimmed_z2_ratio"]=OUT.w1_trimmed_mean_z2/OUT.w2_trimmed_mean_z2
OUT["robust_reduced_z2_ratio"]=OUT.w1_robust_reduced_z2/OUT.w2_robust_reduced_z2

OUT["w1_dominant_metrics"]=(
    (OUT.median_abs_z_ratio>1)&
    (OUT.mad_z_ratio>1)&
    (OUT.trimmed_z2_ratio>1)&
    (OUT.robust_reduced_z2_ratio>1)
)

OUT["verdict"]=np.where(
    OUT.w1_dominant_metrics,
    "ERROR_AWARE_W1_DOMINANCE_PERSISTS",
    "ERROR_AWARE_W1_DOMINANCE_NOT_CONFIRMED"
)

OUT.to_csv(f"{R}/exp030_error_aware_robustness.csv",index=False)

summary={
    "experiment":"EXP-030",
    "purpose":"Test whether apparent W1 dominance persists after normalizing residuals by each band's reported measurement uncertainty.",
    "interpretation_rule":"W1 dominance requires W1 to exceed W2 across all four error-normalized robust metrics.",
    "results":OUT.to_dict(orient="records")
}

with open(f"{R}/exp030_summary.json","w") as f:
    json.dump(summary,f,indent=2)

print("EXP-030 COMPLETE")
print(OUT.to_string(index=False))
print("\nSaved:")
print("✓ exp030_error_aware_robustness.csv")
print("✓ exp030_summary.json")

EXP-030 COMPLETE
 survivor  n  w1_median_abs_z  w1_mad_z  w1_frac_abs_z_ge2  w1_trimmed_mean_z2  w1_robust_reduced_z2  w2_median_abs_z  w2_mad_z  w2_frac_abs_z_ge2  w2_trimmed_mean_z2  w2_robust_reduced_z2  median_abs_z_ratio  mad_z_ratio  trimmed_z2_ratio  robust_reduced_z2_ratio  w1_dominant_metrics                                verdict
      233 22         1.067381  1.482600           0.090909            0.838315              2.504741         0.816731  0.881644           0.090909            0.445219              1.466273            1.306895     1.681631          1.882928                 1.708237                 True      ERROR_AWARE_W1_DOMINANCE_PERSISTS
      234 24         0.552406  0.811656           0.208333            0.349728              0.677204         0.766645  0.960008           0.041667            0.443268              1.311324            0.720550     0.845468          0.788977                 0.516428                False ERROR_AWARE_W1_DOMINANCE_NOT_CONFIRMED
     197

In [40]:
# EXP-031 — ERROR-MODEL CALIBRATION AGAINST MATCHED CONTROLS

import os,json,numpy as np,pandas as pd

R="astronomy_exp005/results"
SURV=[233,234,1974,1976,1978,1979]

D=pd.read_csv(f"{R}/exp005_region_detections.csv")
G=pd.read_csv(f"{R}/exp005_all_source_groups.csv")

for c in ["group_id","w1mpro","w1sigmpro","w2mpro","w2sigmpro"]:
    if c in D.columns:
        D[c]=pd.to_numeric(D[c],errors="coerce")

D=D.dropna(subset=["group_id","w1mpro","w1sigmpro"]).copy()
D=D[D.w1sigmpro>0].copy()
D["group_id"]=D.group_id.astype(int)

for c in ["group_id","n_observations","median_w1_snr","good_quality_fraction"]:
    G[c]=pd.to_numeric(G[c],errors="coerce")

eligible=G[
    (~G.group_id.isin(SURV))&
    (G.n_observations>=10)&
    (G.median_w1_snr>=5)&
    (G.good_quality_fraction>=.90)
].copy()

def metrics(x):
    m=x.w1mpro.to_numpy()
    e=x.w1sigmpro.to_numpy()

    mu=np.average(m,weights=1/e**2)
    z=(m-mu)/e

    chi2=np.sum(z**2)/(len(z)-1)

    medabs=np.median(np.abs(z))
    mad=1.4826*np.median(np.abs(z-np.median(z)))
    f2=np.mean(np.abs(z)>=2)
    f3=np.mean(np.abs(z)>=3)

    q10,q90=np.percentile(z,[10,90])
    t=z[(z>=q10)&(z<=q90)]
    trimmed=np.mean(t**2)

    robust=np.median(z**2)/0.4549364

    return {
        "n":len(z),
        "w1_chi2":chi2,
        "median_abs_z":medabs,
        "mad_z":mad,
        "frac_abs_z_ge2":f2,
        "frac_abs_z_ge3":f3,
        "trimmed_mean_z2":trimmed,
        "robust_reduced_z2":robust
    }

all_metrics=[]

for gid in G.group_id.dropna().astype(int).unique():
    x=D[D.group_id==gid]
    if len(x)>=10:
        mm=metrics(x)
        all_metrics.append({"group_id":gid,**mm})

M=pd.DataFrame(all_metrics)

pairs=[]
results=[]

rng=np.random.default_rng(42)

for sid in SURV:

    sg=G[G.group_id==sid].iloc[0]

    cand=eligible.copy()

    cand["match_score"]=(
        abs(cand.n_observations-sg.n_observations)/sg.n_observations+
        abs(cand.median_w1_snr-sg.median_w1_snr)/sg.median_w1_snr+
        abs(cand.good_quality_fraction-sg.good_quality_fraction)
    )

    controls=cand.sort_values("match_score").head(10)

    sm=M[M.group_id==sid].iloc[0]

    for _,c in controls.iterrows():
        cm=M[M.group_id==int(c.group_id)]

        if len(cm)==0:
            continue

        cm=cm.iloc[0]

        pairs.append({
            "survivor":sid,
            "control":int(c.group_id),
            "separation_match_score":c.match_score,
            "survivor_chi2":sm.w1_chi2,
            "control_chi2":cm.w1_chi2,
            "survivor_median_abs_z":sm.median_abs_z,
            "control_median_abs_z":cm.median_abs_z,
            "survivor_mad_z":sm.mad_z,
            "control_mad_z":cm.mad_z,
            "survivor_frac_z2":sm.frac_abs_z_ge2,
            "control_frac_z2":cm.frac_abs_z_ge2,
            "survivor_frac_z3":sm.frac_abs_z_ge3,
            "control_frac_z3":cm.frac_abs_z_ge3,
            "survivor_trimmed_z2":sm.trimmed_mean_z2,
            "control_trimmed_z2":cm.trimmed_mean_z2,
            "survivor_robust_z2":sm.robust_reduced_z2,
            "control_robust_z2":cm.robust_reduced_z2
        })

    P=pd.DataFrame([p for p in pairs if p["survivor"]==sid])

    if len(P)==0:
        continue

    def pct(s,c):
        return 100*np.mean(P[c]<=s)

    results.append({
        "survivor":sid,
        "n_controls":len(P),
        "w1_chi2":sm.w1_chi2,
        "chi2_percentile":pct(sm.w1_chi2,"control_chi2"),
        "median_abs_z_percentile":pct(sm.median_abs_z,"control_median_abs_z"),
        "mad_z_percentile":pct(sm.mad_z,"control_mad_z"),
        "frac_z2_percentile":pct(sm.frac_abs_z_ge2,"control_frac_z2"),
        "frac_z3_percentile":pct(sm.frac_abs_z_ge3,"control_frac_z3"),
        "trimmed_z2_percentile":pct(sm.trimmed_mean_z2,"control_trimmed_z2"),
        "robust_z2_percentile":pct(sm.robust_reduced_z2,"control_robust_z2")
    })

P=pd.DataFrame(pairs)
S=pd.DataFrame(results)

cols=[
    "chi2_percentile","median_abs_z_percentile","mad_z_percentile",
    "frac_z2_percentile","frac_z3_percentile",
    "trimmed_z2_percentile","robust_z2_percentile"
]

S["median_metric_percentile"]=S[cols].median(axis=1)
S["metrics_ge_90pct"]=(S[cols]>=90).sum(axis=1)
S["metrics_ge_95pct"]=(S[cols]>=95).sum(axis=1)

S["verdict"]=np.select(
    [
        S.metrics_ge_95pct>=4,
        S.metrics_ge_90pct>=4
    ],
    [
        "STRONGLY_UNUSUAL_VS_CONTROLS",
        "POTENTIALLY_UNUSUAL_VS_CONTROLS"
    ],
    default="CONSISTENT_WITH_CONTROL_RANGE"
)

S.to_csv(f"{R}/exp031_error_model_calibration_summary.csv",index=False)
P.to_csv(f"{R}/exp031_matched_control_metrics.csv",index=False)

with open(f"{R}/exp031_summary.json","w") as f:
    json.dump({
        "experiment":"EXP-031",
        "purpose":"Determine whether survivor error-normalized W1 residual behavior is unusual relative to observationally matched ordinary controls.",
        "control_selection":"10 nearest controls matched on observation count, median W1 SNR, and quality; variability metrics were not used for matching.",
        "interpretation":"Empirical percentiles are descriptive and are not p-values or formal statistical significance.",
        "results":S.to_dict(orient="records")
    },f,indent=2)

print("EXP-031 COMPLETE")
print(S.to_string(index=False))
print("\nSaved:")
print("✓ exp031_error_model_calibration_summary.csv")
print("✓ exp031_matched_control_metrics.csv")
print("✓ exp031_summary.json")

EXP-031 COMPLETE
 survivor  n_controls  w1_chi2  chi2_percentile  median_abs_z_percentile  mad_z_percentile  frac_z2_percentile  frac_z3_percentile  trimmed_z2_percentile  robust_z2_percentile  median_metric_percentile  metrics_ge_90pct  metrics_ge_95pct                       verdict
      233          10 2.118475            100.0                    100.0             100.0                50.0                90.0                   90.0                 100.0                     100.0                 6                 4  STRONGLY_UNUSUAL_VS_CONTROLS
      234          10 2.493238            100.0                     20.0              20.0                90.0               100.0                   20.0                  20.0                      20.0                 3                 2 CONSISTENT_WITH_CONTROL_RANGE
     1974          10 3.026153            100.0                    100.0              90.0                90.0                90.0                  100.0                 100.0    

In [41]:
# EXP-032 — LEAVE-ONE-OBSERVATION-OUT STABILITY TEST

import os,json,numpy as np,pandas as pd

R="astronomy_exp005/results"
SURV=[233,1974,1976,1978]

D=pd.read_csv(f"{R}/exp005_region_detections.csv")

for c in ["group_id","w1mpro","w1sigmpro"]:
    D[c]=pd.to_numeric(D[c],errors="coerce")

D=D.dropna(subset=["group_id","w1mpro","w1sigmpro"]).copy()
D=D[D.w1sigmpro>0].copy()
D["group_id"]=D.group_id.astype(int)

def chi2(x):
    m=x.w1mpro.to_numpy()
    e=x.w1sigmpro.to_numpy()
    mu=np.average(m,weights=1/e**2)
    return np.sum(((m-mu)/e)**2)/(len(m)-1)

rows=[]
summary=[]

for sid in SURV:
    X=D[D.group_id==sid].copy().reset_index(drop=True)
    base=chi2(X)
    vals=[]

    for i in range(len(X)):
        Y=X.drop(index=i)
        c=chi2(Y)
        vals.append(c)

        rows.append({
            "survivor":sid,
            "removed_index":i,
            "removed_mjd":X.loc[i,"mjd"] if "mjd" in X.columns else np.nan,
            "removed_w1mpro":X.loc[i,"w1mpro"],
            "removed_w1sigmpro":X.loc[i,"w1sigmpro"],
            "full_chi2":base,
            "leave_one_out_chi2":c,
            "delta_chi2":c-base,
            "still_above_2":c>=2,
            "still_above_3":c>=3
        })

    vals=np.array(vals)

    summary.append({
        "survivor":sid,
        "n_obs":len(X),
        "full_chi2":base,
        "loo_min_chi2":vals.min(),
        "loo_median_chi2":np.median(vals),
        "loo_max_chi2":vals.max(),
        "loo_fraction_still_ge2":np.mean(vals>=2),
        "loo_fraction_still_ge3":np.mean(vals>=3),
        "largest_drop":base-vals.min(),
        "largest_increase":vals.max()-base,
        "verdict":(
            "STABLE_DISTRIBUTED_SIGNAL" if np.all(vals>=2)
            else "MOSTLY_STABLE_BUT_POINT_SENSITIVE" if np.mean(vals>=2)>=.8
            else "POINT_SENSITIVE_SIGNAL"
        )
    })

OUT=pd.DataFrame(rows)
S=pd.DataFrame(summary)

OUT.to_csv(f"{R}/exp032_leave_one_out.csv",index=False)
S.to_csv(f"{R}/exp032_summary.csv",index=False)

with open(f"{R}/exp032_summary.json","w") as f:
    json.dump({
        "experiment":"EXP-032",
        "purpose":"Test whether W1 variability remains above the authoritative chi-square threshold when each observation is removed individually.",
        "thresholds":{
            "validator_chi2":2,
            "secondary_chi2":3
        },
        "results":S.to_dict(orient="records")
    },f,indent=2)

print("EXP-032 COMPLETE")
print(S.to_string(index=False))
print("\nSaved:")
print("✓ exp032_leave_one_out.csv")
print("✓ exp032_summary.csv")
print("✓ exp032_summary.json")

EXP-032 COMPLETE
 survivor  n_obs  full_chi2  loo_min_chi2  loo_median_chi2  loo_max_chi2  loo_fraction_still_ge2  loo_fraction_still_ge3  largest_drop  largest_increase                           verdict
      233     22   2.118475      1.653138         2.164732      2.223905                0.909091                0.000000      0.465337          0.105430 MOSTLY_STABLE_BUT_POINT_SENSITIVE
     1974     25   3.026153      2.142018         3.116816      3.157322                1.000000                0.720000      0.884135          0.131170         STABLE_DISTRIBUTED_SIGNAL
     1976     25   2.115925      1.823840         2.162583      2.206882                0.880000                0.000000      0.292084          0.090958 MOSTLY_STABLE_BUT_POINT_SENSITIVE
     1978     28   3.939207      2.798462         4.051062      4.090674                1.000000                0.964286      1.140745          0.151467         STABLE_DISTRIBUTED_SIGNAL

Saved:
✓ exp032_leave_one_out.csv
✓ exp032_summ

In [42]:
# EXP-033 — INFLUENTIAL-OBSERVATION AUDIT

import os,json,numpy as np,pandas as pd

R="astronomy_exp005/results"
SURV=[233,1974,1976,1978]

D=pd.read_csv(f"{R}/exp005_region_detections.csv")
D["group_id"]=pd.to_numeric(D["group_id"],errors="coerce").astype("Int64")
D["w1mpro"]=pd.to_numeric(D["w1mpro"],errors="coerce")
D["w1sigmpro"]=pd.to_numeric(D["w1sigmpro"],errors="coerce")
D=D.dropna(subset=["group_id","w1mpro","w1sigmpro"]).copy()
D=D[D.w1sigmpro>0].copy()
D["group_id"]=D.group_id.astype(int)

def weighted_stats(x):
    m=x.w1mpro.to_numpy()
    e=x.w1sigmpro.to_numpy()
    w=1/e**2
    mu=np.average(m,weights=w)
    z=(m-mu)/e
    chi=np.sum(z**2)/(len(x)-1)
    return mu,z,chi

rows=[]
summary=[]

for sid in SURV:
    X=D[D.group_id==sid].copy().reset_index(drop=True)
    mu,z,base=weighted_stats(X)

    X["weighted_mean"]=mu
    X["z"] = z
    X["abs_z"]=np.abs(z)

    loo=[]
    for i in range(len(X)):
        Y=X.drop(index=i)
        _,_,c=weighted_stats(Y)
        loo.append(c)

    X["loo_chi2"]=loo
    X["chi2_drop"]=base-X["loo_chi2"]

    # Observations whose removal has the largest effect
    X=X.sort_values("chi2_drop",ascending=False)

    for rank,(_,r) in enumerate(X.iterrows(),1):
        rows.append({
            "survivor":sid,
            "influence_rank":rank,
            "mjd":r.get("mjd",np.nan),
            "scan_id":r.get("scan_id",np.nan),
            "frame_num":r.get("frame_num",np.nan),
            "w1mpro":r.w1mpro,
            "w1sigmpro":r.w1sigmpro,
            "z_full":r.z,
            "abs_z_full":r.abs_z,
            "loo_chi2":r.loo_chi2,
            "chi2_drop":r.chi2_drop,
            "good_quality":r.get("good_quality",np.nan),
            "quality_flagged":r.get("quality_flagged",np.nan),
            "qual_frame":r.get("qual_frame",np.nan),
            "ph_qual":r.get("ph_qual",np.nan),
            "cc_flags":r.get("cc_flags",np.nan),
            "w1snr":r.get("w1snr",np.nan),
            "w2snr":r.get("w2snr",np.nan),
            "w2mpro":r.get("w2mpro",np.nan),
            "w2sigmpro":r.get("w2sigmpro",np.nan)
        })

    top=X.head(3)

    summary.append({
        "survivor":sid,
        "n_obs":len(X),
        "full_chi2":base,
        "most_influential_abs_z":float(X.iloc[0].abs_z),
        "largest_chi2_drop":float(X.iloc[0].chi2_drop),
        "chi2_after_largest_drop":float(X.iloc[0].loo_chi2),
        "top3_mean_abs_z":float(top.abs_z.mean()),
        "top3_mean_chi2_drop":float(top.chi2_drop.mean()),
        "top3_all_good_quality":bool(np.all(top.good_quality==True)) if "good_quality" in top else False,
        "verdict":(
            "INFLUENTIAL_POINT_NEEDS_IMAGE_AUDIT"
            if X.iloc[0].loo_chi2 < 2
            else "INFLUENTIAL_POINTS_BUT_SIGNAL_SURVIVES"
        )
    })

OUT=pd.DataFrame(rows)
S=pd.DataFrame(summary)

OUT.to_csv(f"{R}/exp033_influential_observations.csv",index=False)
S.to_csv(f"{R}/exp033_summary.csv",index=False)

with open(f"{R}/exp033_summary.json","w") as f:
    json.dump({
        "experiment":"EXP-033",
        "purpose":"Audit observations with the greatest influence on W1 reduced chi-square.",
        "interpretation":"Influence on chi-square does not itself establish an artifact or astrophysical origin.",
        "results":S.to_dict(orient="records")
    },f,indent=2)

print("EXP-033 COMPLETE")
print("\nSUMMARY")
print(S.to_string(index=False))

print("\nTOP 5 INFLUENTIAL OBSERVATIONS PER SURVIVOR")
print(
    OUT.groupby("survivor",group_keys=False)
       .head(5)
       [["survivor","influence_rank","mjd","scan_id","frame_num",
         "w1mpro","w1sigmpro","abs_z_full","loo_chi2","chi2_drop",
         "good_quality","quality_flagged","qual_frame","ph_qual",
         "cc_flags","w1snr","w2snr"]]
       .to_string(index=False)
)

print("\nSaved:")
print("✓ exp033_influential_observations.csv")
print("✓ exp033_summary.csv")
print("✓ exp033_summary.json")

EXP-033 COMPLETE

SUMMARY
 survivor  n_obs  full_chi2  most_influential_abs_z  largest_chi2_drop  chi2_after_largest_drop  top3_mean_abs_z  top3_mean_chi2_drop  top3_all_good_quality                                verdict
      233     22   2.118475                3.303136           0.465337                 1.653138         2.705209             0.291561                   True    INFLUENTIAL_POINT_NEEDS_IMAGE_AUDIT
     1974     25   3.026153                4.770346           0.884135                 2.142018         3.759872             0.542167                   True INFLUENTIAL_POINTS_BUT_SIGNAL_SURVIVES
     1976     25   2.115925                2.904466           0.292084                 1.823840         2.516078             0.197813                   True    INFLUENTIAL_POINT_NEEDS_IMAGE_AUDIT
     1978     28   3.939207                5.692101           1.140745                 2.798462         4.784734             0.786418                   True INFLUENTIAL_POINTS_BUT_SIGNAL_SUR

In [43]:
# EXP-034A — NEOWISE IMAGE METADATA RETRIEVAL

import os, requests, pandas as pd, numpy as np

R="astronomy_exp005/results"

I=pd.read_csv(f"{R}/exp033_influential_observations.csv")
I=I[I.survivor.isin([1974,1978])].copy()

# Top 4 influential observations for each source
I=I.sort_values(["survivor","influence_rank"])
I=I.groupby("survivor",group_keys=False).head(4).copy()

# IRSA WISE image metadata service
URL="https://irsa.ipac.caltech.edu/ibe/search/wise/merge_int/merge_i1bm_frm"

rows=[]

for _,r in I.iterrows():
    for band in [1,2]:
        params={
            "scan_id":str(r.scan_id),
            "frame_num":int(r.frame_num),
            "band":band,
            "format":"csv"
        }

        try:
            res=requests.get(URL,params=params,timeout=30)
            res.raise_for_status()

            text=res.text.strip()

            if not text:
                rows.append({
                    "survivor":r.survivor,
                    "influence_rank":r.influence_rank,
                    "scan_id":r.scan_id,
                    "frame_num":r.frame_num,
                    "band":band,
                    "status":"EMPTY_RESPONSE"
                })
                continue

            from io import StringIO
            q=pd.read_csv(StringIO(text))

            if len(q)==0:
                rows.append({
                    "survivor":r.survivor,
                    "influence_rank":r.influence_rank,
                    "scan_id":r.scan_id,
                    "frame_num":r.frame_num,
                    "band":band,
                    "status":"NO_MATCH"
                })
            else:
                for _,x in q.iterrows():
                    d=x.to_dict()
                    d.update({
                        "survivor":r.survivor,
                        "influence_rank":r.influence_rank,
                        "target_mjd":r.mjd,
                        "target_w1mpro":r.w1mpro,
                        "target_w1sigmpro":r.w1sigmpro,
                        "status":"FOUND"
                    })
                    rows.append(d)

        except Exception as e:
            rows.append({
                "survivor":r.survivor,
                "influence_rank":r.influence_rank,
                "scan_id":r.scan_id,
                "frame_num":r.frame_num,
                "band":band,
                "status":"ERROR",
                "error":str(e)
            })

M=pd.DataFrame(rows)

M.to_csv(f"{R}/exp034A_image_metadata.csv",index=False)

print("EXP-034A COMPLETE")
print("\nStatus counts:")
print(M.status.value_counts(dropna=False))

print("\nResults:")
cols=[c for c in [
    "survivor","influence_rank","scan_id","frame_num",
    "band","status","crval1","crval2","naxis1","naxis2",
    "qual_frame","wrelease","date_obs"
] if c in M.columns]

print(M[cols].to_string(index=False))

print("\nSaved:")
print("✓ exp034A_image_metadata.csv")

EXP-034A COMPLETE

Status counts:
status
FOUND    96
Name: count, dtype: int64

Results:
 survivor  influence_rank status
     1974               1  FOUND
     1974               1  FOUND
     1974               1  FOUND
     1974               1  FOUND
     1974               1  FOUND
     1974               1  FOUND
     1974               1  FOUND
     1974               1  FOUND
     1974               1  FOUND
     1974               1  FOUND
     1974               1  FOUND
     1974               1  FOUND
     1974               2  FOUND
     1974               2  FOUND
     1974               2  FOUND
     1974               2  FOUND
     1974               2  FOUND
     1974               2  FOUND
     1974               2  FOUND
     1974               2  FOUND
     1974               2  FOUND
     1974               2  FOUND
     1974               2  FOUND
     1974               2  FOUND
     1974               3  FOUND
     1974               3  FOUND
     1974           

In [44]:
# EXP-034B — INSPECT AND DEDUPLICATE NEOWISE IMAGE PRODUCTS

import pandas as pd, os

R="astronomy_exp005/results"
M=pd.read_csv(f"{R}/exp034A_image_metadata.csv")

print("Rows:",len(M))
print("Columns:")
print(M.columns.tolist())

# Show the most useful identifying fields that actually exist
preferred=[
    "survivor","influence_rank","scan_id","frame_num","band",
    "coadd_id","fname","filename","path","url",
    "instrument","wrelease","date_obs","mjd",
    "crval1","crval2","naxis1","naxis2"
]
cols=[c for c in preferred if c in M.columns]

print("\nPRODUCT RECORDS:")
print(M[cols].drop_duplicates().to_string(index=False))

print("\nUnique combinations:")
for keys in [
    ["survivor","influence_rank","scan_id","frame_num","band"],
    ["scan_id","frame_num","band"],
]:
    if all(c in M.columns for c in keys):
        print(f"{keys}: {len(M[keys].drop_duplicates())}")

# Save deduplicated metadata
U=M.drop_duplicates().copy()
U.to_csv(f"{R}/exp034B_image_products.csv",index=False)

print("\nSaved:")
print("✓ exp034B_image_products.csv")

Rows: 96
Columns:
['<?xml version="1.0" encoding="utf-8"?>', 'survivor', 'influence_rank', 'target_mjd', 'target_w1mpro', 'target_w1sigmpro', 'status']

PRODUCT RECORDS:
 survivor  influence_rank
     1974               1
     1974               2
     1974               3
     1974               4
     1978               1
     1978               2
     1978               3
     1978               4

Unique combinations:

Saved:
✓ exp034B_image_products.csv


In [45]:
import requests
import pandas as pd
import os

R="astronomy_exp005/results"

I=pd.read_csv(f"{R}/exp033_influential_observations.csv")
I=I[I.survivor.isin([1974,1978])].sort_values(
    ["survivor","influence_rank"]
).groupby("survivor",group_keys=False).head(1)

r=I.iloc[0]

URL="https://irsa.ipac.caltech.edu/ibe/search/wise/merge_int/merge_i1bm_frm"

params={
    "scan_id":str(r.scan_id),
    "frame_num":int(r.frame_num),
    "band":1,
    "format":"csv"
}

res=requests.get(URL,params=params,timeout=30)

print("HTTP status:",res.status_code)
print("Content-Type:",res.headers.get("Content-Type"))
print("Final URL:")
print(res.url)

print("\nResponse length:",len(res.text))

print("\nFirst 2000 characters:")
print(res.text[:2000])

HTTP status: 200
Content-Type: text/xml
Final URL:
https://irsa.ipac.caltech.edu/ibe/search/wise/merge_int/merge_i1bm_frm?scan_id=62285r&frame_num=112&band=1&format=csv

Response length: 479

First 2000 characters:
<?xml version="1.0" encoding="utf-8"?>
<VOTABLE version="1.3" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://www.ivoa.net/xml/VOTable/v1.3" xsi:schemaLocation="http://www.ivoa.net/xml/VOTable/v1.3 http://www.ivoa.net/xml/VOTable/v1.3">
  <DESCRIPTION>Caltech/IPAC-IRSA Image Backend Service</DESCRIPTION>
  <RESOURCE type="results">
    <INFO name="QUERY_STATUS" value="ERROR">UsageFault: BAD_REQUEST: Unknown parameter: band</INFO>
  </RESOURCE>
</VOTABLE>



In [46]:
import requests
import pandas as pd
import os

R="astronomy_exp005/results"
OUT=f"{R}/exp034_images"
os.makedirs(OUT,exist_ok=True)

# First influential observation: group 1974
scan_id="62285r"
frame_num=112
band=1

# Group 1974 coordinates
G=pd.read_csv(f"{R}/exp005_all_source_groups.csv")
g=G[G.group_id==1974].iloc[0]

scangrp=scan_id[-2:]

filename=f"{scan_id}{frame_num:03d}-w{band}-int-1b.fits"

url=(
    "https://irsa.ipac.caltech.edu/data/wise/merge_int/"
    f"merge_i1bm_frm/{scangrp}/{scan_id}/{frame_num:03d}/{filename}"
)

params={
    "center":f"{g.ra},{g.dec}",
    "size":"100pix",
    "gzip":"false"
}

print("URL:")
print(url)

print("\nTarget:")
print("RA =",g.ra)
print("DEC =",g.dec)

res=requests.get(url,params=params,timeout=120)

print("\nHTTP status:",res.status_code)
print("Content-Type:",res.headers.get("Content-Type"))
print("Bytes:",len(res.content))
print("First 20 bytes:",res.content[:20])

if res.status_code==200 and res.content[:6]==b"SIMPLE":
    path=f"{OUT}/test_1974_rank1_w1.fits"

    with open(path,"wb") as f:
        f.write(res.content)

    print("\nSUCCESS")
    print("Saved:",path)
else:
    print("\nFAILED")
    print(res.text[:1000])

URL:
https://irsa.ipac.caltech.edu/data/wise/merge_int/merge_i1bm_frm/5r/62285r/112/62285r112-w1-int-1b.fits

Target:
RA = 179.9281935
DEC = -0.0478388

HTTP status: 404
Content-Type: text/html; charset=iso-8859-1
Bytes: 196
First 20 bytes: b'<!DOCTYPE HTML PUBLI'

FAILED
<!DOCTYPE HTML PUBLIC "-//IETF//DTD HTML 2.0//EN">
<html><head>
<title>404 Not Found</title>
</head><body>
<h1>Not Found</h1>
<p>The requested URL was not found on this server.</p>
</body></html>



In [47]:
import requests
import pandas as pd
from io import StringIO

scan_id="62285r"
frame_num=112
band=1

url="https://irsa.ipac.caltech.edu/ibe/search/wise/neowiser/p1bm_frm"

params={
    "where":f"scan_id='{scan_id}' and frame_num={frame_num} and band={band}",
    "columns":"scan_id,frame_num,band,ra,dec,date_obs,mjd_obs,qual_frame",
    "ct":"CSV"
}

r=requests.get(url,params=params,timeout=120)

print("HTTP:",r.status_code)
print("Content-Type:",r.headers.get("Content-Type"))
print("URL:",r.url)
print("\nResponse:")
print(r.text[:3000])

if r.status_code==200 and "ERROR" not in r.text[:200]:
    try:
        df=pd.read_csv(StringIO(r.text))
        print("\nParsed rows:",len(df))
        print(df.to_string(index=False))
    except Exception as e:
        print("\nCSV parse failed:",e)

HTTP: 200
Content-Type: text/xml
URL: https://irsa.ipac.caltech.edu/ibe/search/wise/neowiser/p1bm_frm?where=scan_id%3D%2762285r%27+and+frame_num%3D112+and+band%3D1&columns=scan_id%2Cframe_num%2Cband%2Cra%2Cdec%2Cdate_obs%2Cmjd_obs%2Cqual_frame&ct=CSV

Response:
<?xml version="1.0" encoding="utf-8"?>
<VOTABLE version="1.3" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://www.ivoa.net/xml/VOTable/v1.3" xsi:schemaLocation="http://www.ivoa.net/xml/VOTable/v1.3 http://www.ivoa.net/xml/VOTable/v1.3">
  <DESCRIPTION>Caltech/IPAC-IRSA Image Backend Service</DESCRIPTION>
  <RESOURCE type="results">
    <INFO name="QUERY_STATUS" value="ERROR">UsageFault: BAD_REQUEST:  column &quot;ra&quot; does not exist
</INFO>
  </RESOURCE>
</VOTABLE>


Parsed rows: 7
                                                                                                                                                                                   <?xml version="1.0" encoding="utf-8"?>
<VOTABLE 

In [48]:
import requests

url="https://irsa.ipac.caltech.edu/ibe/search/wise/neowiser/p1bm_frm"

params={
    "where":"scan_id='62285r' and frame_num=112 and band=1",
    "ct":"CSV"
}

r=requests.get(url,params=params,timeout=120)

print("HTTP:",r.status_code)
print("Content-Type:",r.headers.get("Content-Type"))
print("URL:",r.url)
print("\nResponse:")
print(r.text[:10000])

HTTP: 200
Content-Type: text/csv
URL: https://irsa.ipac.caltech.edu/ibe/search/wise/neowiser/p1bm_frm?where=scan_id%3D%2762285r%27+and+frame_num%3D112+and+band%3D1&ct=CSV

Response:
band,crval1,crval2,ra1,dec1,ra2,dec2,ra3,dec3,ra4,dec4,magzp,magzpunc,modeint,scan_id,scangrp,frame_num,date_obs,mjd_obs,dtanneal,utanneal,exptime,debgain,febgain,moon_sep,saa_sep,qa_status,qual_frame,qual_scan
1,180.152479761343,0.244772981522,179.946209139184,0.755154003873,180.661616403405,0.446867445625,180.356899847518,-0.263096163726,179.639760745049,0.043345819599,20.76090,-9.990000,22.563,62285r,5r,112,2024-05-29 19:48:27.141,60459.82531413,999999.00000000000,2016-11-20 03:23:20.494,7.7,,,147.353,16.685,Prelim,10,5



In [49]:
import requests
import os

scan_id="62285r"
scangrp="5r"
frame_num=112
band=1

url=(
    "https://irsa.ipac.caltech.edu/ibe/data/"
    f"wise/neowiser/p1bm_frm/"
    f"{scangrp}/{scan_id}/{frame_num:03d}/"
    f"{scan_id}{frame_num:03d}-w{band}-int-1b.fits"
)

params={
    "center":"179.9281935,-0.0478388",
    "size":"100pix",
    "gzip":"false"
}

print("URL:")
print(url)

r=requests.get(url,params=params,timeout=120)

print("\nHTTP:",r.status_code)
print("Content-Type:",r.headers.get("Content-Type"))
print("Bytes:",len(r.content))
print("First 20 bytes:",r.content[:20])

if r.status_code==200 and r.content[:6]==b"SIMPLE":
    path="astronomy_exp005/results/exp034_test_w1.fits"

    with open(path,"wb") as f:
        f.write(r.content)

    print("\nSUCCESS")
    print("Saved:",path)
else:
    print("\nFAILED")
    print(r.text[:1000])

URL:
https://irsa.ipac.caltech.edu/ibe/data/wise/neowiser/p1bm_frm/5r/62285r/112/62285r112-w1-int-1b.fits

HTTP: 200
Content-Type: application/fits
Bytes: 74880
First 20 bytes: b'SIMPLE  =           '

SUCCESS
Saved: astronomy_exp005/results/exp034_test_w1.fits


In [50]:
import os
import requests
import pandas as pd
from urllib.parse import urlencode

R="astronomy_exp005/results"
OUT=f"{R}/exp034_images"
os.makedirs(OUT,exist_ok=True)

I=pd.read_csv(f"{R}/exp033_influential_observations.csv")
I=I[I.survivor.isin([1974,1978])].copy()
I=I.sort_values(["survivor","influence_rank"])
I=I.groupby("survivor",group_keys=False).head(4)

G=pd.read_csv(f"{R}/exp005_all_source_groups.csv")[[
    "group_id","ra","dec"
]]

I=I.merge(
    G,
    left_on="survivor",
    right_on="group_id",
    how="left"
)

META={
    "62285r":"5r",
    "62273r":"3r",
    "62279r":"9r",
    "57318r":"8r",
    "62263r":"3r",
    "57320r":"0r",
    "62277r":"7r",
    "57322r":"2r"
}

BASE="https://irsa.ipac.caltech.edu/ibe/data/wise/neowiser/p1bm_frm"

results=[]

for _,r in I.iterrows():

    scan=str(r.scan_id)
    frame=int(r.frame_num)
    scangrp=META.get(scan)

    if scangrp is None:
        results.append({
            "survivor":int(r.survivor),
            "rank":int(r.influence_rank),
            "scan_id":scan,
            "frame_num":frame,
            "status":"NO_SCANGRP"
        })
        continue

    for band in [1,2]:

        filename=f"{scan}{frame:03d}-w{band}-int-1b.fits"

        url=(
            f"{BASE}/{scangrp}/{scan}/{frame:03d}/"
            f"{filename}"
        )

        params={
            "center":f"{r.ra},{r.dec}",
            "size":"100pix",
            "gzip":"false"
        }

        dest=(
            f"{OUT}/"
            f"g{int(r.survivor)}_"
            f"rank{int(r.influence_rank)}_"
            f"w{band}.fits"
        )

        try:
            res=requests.get(
                url+"?"+urlencode(params),
                timeout=120
            )

            ok=(
                res.status_code==200 and
                res.content[:6]==b"SIMPLE"
            )

            if ok:
                with open(dest,"wb") as f:
                    f.write(res.content)

                results.append({
                    "survivor":int(r.survivor),
                    "rank":int(r.influence_rank),
                    "scan_id":scan,
                    "frame_num":frame,
                    "band":band,
                    "scangrp":scangrp,
                    "ra":r.ra,
                    "dec":r.dec,
                    "url":url,
                    "file":dest,
                    "bytes":len(res.content),
                    "status":"DOWNLOADED"
                })
            else:
                results.append({
                    "survivor":int(r.survivor),
                    "rank":int(r.influence_rank),
                    "scan_id":scan,
                    "frame_num":frame,
                    "band":band,
                    "url":url,
                    "http_status":res.status_code,
                    "response_start":res.content[:200].decode(
                        "utf-8","replace"
                    ),
                    "status":"FAILED"
                })

        except Exception as e:
            results.append({
                "survivor":int(r.survivor),
                "rank":int(r.influence_rank),
                "scan_id":scan,
                "frame_num":frame,
                "band":band,
                "url":url,
                "status":"ERROR",
                "error":str(e)
            })

D=pd.DataFrame(results)

D.to_csv(
    f"{R}/exp034B_downloads.csv",
    index=False
)

print("EXP-034B-R5 COMPLETE")
print("\nStatus:")
print(D.status.value_counts(dropna=False))

print("\nDownloaded:")
cols=[
    "survivor","rank","scan_id",
    "frame_num","band","scangrp",
    "bytes","status"
]
print(D[[c for c in cols if c in D.columns]].to_string(index=False))

print("\nFiles downloaded:",
      int((D.status=="DOWNLOADED").sum()))

print("\nSaved:")
print("✓ exp034B_downloads.csv")
print(f"✓ FITS cutouts → {OUT}")

EXP-034B-R5 COMPLETE

Status:
status
DOWNLOADED    16
Name: count, dtype: int64

Downloaded:
 survivor  rank scan_id  frame_num  band scangrp  bytes     status
     1974     1  62285r        112     1      5r  74880 DOWNLOADED
     1974     1  62285r        112     2      5r  74880 DOWNLOADED
     1974     2  62273r        111     1      3r  80640 DOWNLOADED
     1974     2  62273r        111     2      3r  80640 DOWNLOADED
     1974     3  62279r        136     1      9r  80640 DOWNLOADED
     1974     3  62279r        136     2      9r  80640 DOWNLOADED
     1974     4  57318r        137     1      8r  80640 DOWNLOADED
     1974     4  57318r        137     2      8r  80640 DOWNLOADED
     1978     1  62263r         37     1      3r  80640 DOWNLOADED
     1978     1  62263r         37     2      3r  80640 DOWNLOADED
     1978     2  57320r        113     1      0r  66240 DOWNLOADED
     1978     2  57320r        113     2      0r  66240 DOWNLOADED
     1978     3  62277r        112  

In [51]:
import os
import glob
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u

R="astronomy_exp005/results"
IMG=f"{R}/exp034_images"

G=pd.read_csv(f"{R}/exp005_all_source_groups.csv")[[
    "group_id","ra","dec"
]]

files=glob.glob(f"{IMG}/g*_rank*_w*.fits")

rows=[]

for path in sorted(files):

    name=os.path.basename(path)

    parts=name.replace(".fits","").split("_")
    gid=int(parts[0][1:])
    rank=int(parts[1][4:])
    band=int(parts[2][1:])

    g=G[G.group_id==gid].iloc[0]

    with fits.open(path) as hdul:

        data=hdul[0].data.astype(float)
        header=hdul[0].header
        wcs=WCS(header)

        if data.ndim>2:
            data=np.squeeze(data)

        target=SkyCoord(
            g.ra,g.dec,
            unit="deg",
            frame="icrs"
        )

        x,y=wcs.world_to_pixel(target)

        ny,nx=data.shape

        inside=(
            np.isfinite(x) and
            np.isfinite(y) and
            0<=x<nx and
            0<=y<ny
        )

        if not inside:
            rows.append({
                "survivor":gid,
                "rank":rank,
                "band":band,
                "file":path,
                "x":x,
                "y":y,
                "inside":False
            })
            continue

        # 3-pixel target aperture
        yy,xx=np.indices(data.shape)
        r=np.sqrt((xx-x)**2+(yy-y)**2)

        ap=r<=3.0

        # background annulus
        ann=(r>=6.0)&(r<=12.0)

        ap_values=data[ap]
        ann_values=data[ann]

        ap_values=ap_values[np.isfinite(ap_values)]
        ann_values=ann_values[np.isfinite(ann_values)]

        if len(ann_values):
            background=np.median(ann_values)
            bg_std=1.4826*np.median(
                np.abs(ann_values-background)
            )
        else:
            background=np.nan
            bg_std=np.nan

        raw_flux=np.sum(ap_values)

        n_ap=np.sum(ap)

        bg_sub_flux=(
            raw_flux-background*n_ap
            if np.isfinite(background)
            else np.nan
        )

        peak=np.nanmax(ap_values) if len(ap_values) else np.nan

        # local maximum distance from WCS target
        local_r=r[(r<=5)&np.isfinite(data)]

        rows.append({
            "survivor":gid,
            "rank":rank,
            "band":band,
            "file":path,
            "x":x,
            "y":y,
            "nx":nx,
            "ny":ny,
            "inside":True,
            "aperture_pixels":int(n_ap),
            "background":background,
            "background_robust_sigma":bg_std,
            "raw_aperture_flux":raw_flux,
            "background_subtracted_flux":bg_sub_flux,
            "peak_pixel":peak,
            "aperture_snr_proxy":(
                bg_sub_flux/(bg_std*np.sqrt(n_ap))
                if np.isfinite(bg_std) and bg_std>0
                else np.nan
            ),
            "finite_background_pixels":len(ann_values)
        })

D=pd.DataFrame(rows)

D.to_csv(
    f"{R}/exp034C_image_photometry.csv",
    index=False
)

print("EXP-034C COMPLETE")
print("\nRows:",len(D))
print("\nInside cutout:")
print(D.inside.value_counts(dropna=False))

print("\nImage photometry:")
cols=[
    "survivor","rank","band",
    "x","y","background",
    "background_robust_sigma",
    "raw_aperture_flux",
    "background_subtracted_flux",
    "peak_pixel",
    "aperture_snr_proxy"
]

print(D[cols].to_string(index=False))

print("\nSaved:")
print("✓ exp034C_image_photometry.csv")

EXP-034C COMPLETE

Rows: 16

Inside cutout:
inside
True    16
Name: count, dtype: int64

Image photometry:
 survivor  rank  band                  x                  y  background  background_robust_sigma  raw_aperture_flux  background_subtracted_flux  peak_pixel  aperture_snr_proxy
     1974     1     1 50.402526304045615 49.779455205425684   24.990196                 4.366206        1289.535107                  564.819416  131.021973           24.021849
     1974     1     2   50.0576454574733  49.68211919118533   74.168976                 5.229052        2193.201843                  190.639496  108.069237            7.016299
     1974     2     1  49.60571670653834   49.7696424134971   23.211053                 3.798492        1275.706795                  602.586261   83.161407           29.458388
     1974     2     2   50.0082197238363  50.32688363295746   73.787804                 6.789270        2067.115128                  148.632233   99.602722            4.293419
     1974    

In [55]:
import os,requests,pandas as pd,numpy as np
from urllib.parse import urlencode
from astropy.table import Table
from astropy.coordinates import SkyCoord

R="astronomy_exp005/results"; OUT=f"{R}/exp034D"; os.makedirs(OUT,exist_ok=True)
D=pd.read_csv(f"{R}/exp034B_downloads.csv"); D=D[D.status=="DOWNLOADED"].copy()
G=pd.read_csv(f"{R}/exp005_all_source_groups.csv")[["group_id","ra","dec"]].set_index("group_id")
BASE="https://irsa.ipac.caltech.edu/ibe/data/wise/neowiser/p1bm_frm"
types={"D":"diffraction_spike","O":"optical_ghost","P":"latent","H":"halo"}

rows=[]
for _,r in D.iterrows():
    gid=int(r["survivor"]); scan=str(r["scan_id"]); fr=int(r["frame_num"]); band=int(r["band"]); sg=str(r["scangrp"])
    target=SkyCoord(float(G.loc[gid,"ra"]),float(G.loc[gid,"dec"]),unit="deg")
    stem=f"{scan}{fr:03d}"
    for code,atype in types.items():
        fn=f"{stem}-art-w{band}-{code}.tbl"; url=f"{BASE}/{sg}/{scan}/{fr:03d}/{fn}"
        try:
            z=requests.get(url,timeout=60)
            if z.status_code==200:
                try:
                    t=Table.read(z.text,format="ascii.ipac"); names={x.lower():x for x in t.colnames}
                    rc,dc=names.get("ra"),names.get("dec")
                    if rc and dc and len(t):
                        c=SkyCoord(np.asarray(t[rc],float),np.asarray(t[dc],float),unit="deg")
                        s=target.separation(c).arcsec; near=float(s.min()); n5=int((s<=5).sum()); n10=int((s<=10).sum()); n30=int((s<=30).sum())
                    else: near=np.nan;n5=n10=n30=0
                    nr=len(t)
                except: near=np.nan;n5=n10=n30=0;nr=np.nan
                st="TABLE_FOUND"
            elif z.status_code==404:
                near=np.nan;n5=n10=n30=0;nr=0;st="NO_TABLE"
            else:
                near=np.nan;n5=n10=n30=0;nr=np.nan;st=f"HTTP_{z.status_code}"
        except:
            near=np.nan;n5=n10=n30=0;nr=np.nan;st="ERROR"
        rows.append([gid,int(r["rank"]),scan,fr,band,atype,code,nr,near,n5,n10,n30,st,url])

A=pd.DataFrame(rows,columns=["survivor","rank","scan_id","frame_num","band","artifact_type","artifact_code","n_artifact_rows","nearest_artifact_arcsec","n_within_5arcsec","n_within_10arcsec","n_within_30arcsec","status","url"])
A.to_csv(f"{R}/exp034D_artifact_tables.csv",index=False)

mr=[]
for _,r in D.iterrows():
    scan=str(r["scan_id"]);fr=int(r["frame_num"]);band=int(r["band"]);sg=str(r["scangrp"]);gid=int(r["survivor"]);rank=int(r["rank"])
    fn=f"{scan}{fr:03d}-w{band}-msk-1b.fits";url=f"{BASE}/{sg}/{scan}/{fr:03d}/{fn}"
    q=url+"?"+urlencode({"center":f'{G.loc[gid,"ra"]},{G.loc[gid,"dec"]}',"size":"100pix","gzip":"false"})
    dest=f"{OUT}/g{gid}_rank{rank}_w{band}_mask.fits"
    try:
        z=requests.get(q,timeout=120);ok=z.status_code==200
        if ok: open(dest,"wb").write(z.content)
        mr.append([gid,rank,scan,fr,band,"DOWNLOADED" if ok else "FAILED",z.status_code,len(z.content)])
    except: mr.append([gid,rank,scan,fr,band,"ERROR",np.nan,0])

M=pd.DataFrame(mr,columns=["survivor","rank","scan_id","frame_num","band","status","http_status","bytes"])
M.to_csv(f"{R}/exp034D_masks.csv",index=False)

print("EXP-034D COMPLETE")
print("\nArtifact status:")
print(A.groupby(["artifact_type","status"]).size().to_string())
print("\nClosest artifacts:")
print(A[A.status=="TABLE_FOUND"][["survivor","rank","band","artifact_type","n_artifact_rows","nearest_artifact_arcsec","n_within_5arcsec","n_within_10arcsec","n_within_30arcsec"]].sort_values("nearest_artifact_arcsec").to_string(index=False))
print("\nMask status:")
print(M.status.value_counts(dropna=False))
print("\nSaved:",f"{R}/exp034D_artifact_tables.csv",f"{R}/exp034D_masks.csv")

EXP-034D COMPLETE

Artifact status:
artifact_type      status     
diffraction_spike  NO_TABLE        3
                   TABLE_FOUND    13
halo               TABLE_FOUND    16
latent             NO_TABLE        4
                   TABLE_FOUND    12
optical_ghost      NO_TABLE       13
                   TABLE_FOUND     3

Closest artifacts:
 survivor  rank  band     artifact_type  n_artifact_rows  nearest_artifact_arcsec  n_within_5arcsec  n_within_10arcsec  n_within_30arcsec
     1978     3     1 diffraction_spike              151               382.949703                 0                  0                  0
     1978     3     1              halo              526               391.208047                 0                  0                  0
     1978     1     1 diffraction_spike               32               392.491584                 0                  0                  0
     1978     1     1              halo               38               393.685502                 0   